# 04 · JPEG-aware baseline and selective-detail development
**Established FBCNN preprocessing + frozen classical/DPIR inverses · development data only**

This experiment tests whether blind JPEG preprocessing improves the fixed reconstruction baseline that degraded at the JPEG stage in Notebook 03. It then reassesses operator-sensitivity patch selection with the improved or worsened reconstruction. FBCNN is an existing published method; this composition is not a new-method claim or an exact JPEG likelihood.

**Pre-delivery validation:** source 0801 completed both conditions on CPU; independent array readback passed. The exact executed notebook and results are preserved separately. This clean notebook has no inherited result outputs. CUDA bootstrap and corrected descriptive metadata were checked separately; GPU/Drive and full four-source inference remain pending.

**Default run:** four sources (0801–0804) × two conditions = eight observations. **Local validation plan:** source 0801 at both conditions, identical full resolution and settings, on CPU. Any saved local results concern that subset only; Section 2 prints the actual execution scope. A four-source decision is never inferred from a one-source run.

**Run in Colab:** select a GPU runtime, choose **Run all**, and authorize Drive mounting. Use the same four original PNGs in `MyDrive/reliable-reconstruction-under-mismatch/samples`. Verified DRUNet weights are reused; official FBCNN colour weights (~288 MB) are downloaded once and cached. No repository clone is needed. Dependencies: Python, NumPy, pandas, Pillow, Matplotlib, Torch, IPython and nbformat (normally provided by Colab).

**Retrieve results:** Section 11 prints the exact dated `SUMMARY.zip` and an `upload_parts` folder. Upload the summary ZIP **and every numbered RAW part ZIP** from that folder. Parts are automatically verified and kept below 100 MiB; no additional packaging notebook is needed. The unsplit RAW ZIP is also preserved. Allow several GB of Drive space for arrays, weights, archives and transfer copies.

### Fixed comparisons and limits
- Input, classical inverse, DPIR, FBCNN alone, FBCNN + classical, FBCNN + DPIR. Every method runs on 8-bit uncompressed and JPEG Q75 observations from the same noise draw.
- All methods receive decoded pixels; operational reconstruction never sees clean truth, true blur or JPEG quality metadata. FBCNN estimates quality automatically. References are used only for evaluation, texture stratification and oracle ranking.
- Preserve the 576 × 576 centre crop, 512 × 512 evaluation region, true blur 1.6, nominal blur 1.0, noise 2/255, eight DPIR iterations and classical lambda 0.05. No hyperparameter search.
- The two pretrained families report DIV2K training; checkpoint-specific overlap is unresolved. These four sources are already exposed development data, not an independent test. Patches and stages do not create independent source replicates.
- A deblocking-then-deblurring cascade can alter the effective noise and remove useful details. Frozen downstream settings isolate this intervention, not the best possible cascade.
- Whole-pipeline rotations, measurement residual, image gradient and exact expected random selection are controls. Rotation spread is a limited heuristic; a stronger trained uncertainty comparator remains outstanding. Residuals use the original supplied observation and are not JPEG likelihoods.
- The FBCNN paper reports MATLAB JPEG training; the repository's dataset/demo use OpenCV. Our inherited Pillow encoder is preserved. This is not a reproduction of the FBCNN paper's benchmark.

### 1. Verify and load the embedded code
The inherited helpers remain unchanged. Both upstream licences, pinned vendor bytes, checkpoint identities, earlier verified anchors and the frozen design are included. Source hashes are checked before inference.

In [ ]:
import os, json, hashlib, types
# Set before Torch/CUDA initialization for deterministic matrix operations on supported CUDA runtimes.
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display, Image as DisplayImage

SOURCES = {'baseline': '"""Pilot 01: classical baseline development; no independent-test claims.\n\nThe notebook embeds these functions verbatim so Colab needs no repository install.\n"""\nimport hashlib\nimport io\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport PIL\nfrom PIL import Image\n\n\nCONFIG = {\n    \'source_ids\': [\'0801\', \'0802\', \'0803\', \'0804\'],\n    \'seed\': 20260917,\n    \'crop_size\': 512,\n    \'context_border\': 32,\n    \'true_sigmas\': [1.2, 1.6, 2.0],\n    \'noise_stds\': [2 / 255, 5 / 255],\n    \'nominal_sigma\': 1.0,\n    \'lambdas\': [0.00001, 0.0001, 0.0005, 0.002, 0.01, 0.05, 0.2],\n    \'families\': [\'ridge\', \'gradient\'],\n    \'scenarios\': [\'blur_noise\', \'blur_noise_jpeg\'],\n    \'jpeg_quality\': 75,\n    \'jpeg_subsampling\': 0,\n    \'smooth_sigma\': 0.5,\n    \'patch_size\': 16,\n    \'coverages\': [0.5, 0.75, 0.9, 1.0],\n    \'operator_factors\': [0.8, 1.0, 1.2],\n    \'noise_probe_std\': 2 / 255,\n    \'role\': \'development_only\',\n    \'selection\': \'leave-one-development-source-out; one lambda per family and operator-information mode across all conditions\',\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef transfer(shape, sigma):\n    fy = np.fft.fftfreq(shape[0])[:, None]\n    fx = np.fft.fftfreq(shape[1])[None, :]\n    return np.exp(-2 * np.pi**2 * sigma**2 * (fx**2 + fy**2))\n\n\ndef penalty(shape, family):\n    if family == \'ridge\':\n        return np.ones(shape)\n    if family == \'gradient\':\n        fy = np.fft.fftfreq(shape[0])[:, None]\n        fx = np.fft.fftfreq(shape[1])[None, :]\n        return 4 * (np.sin(np.pi * fx)**2 + np.sin(np.pi * fy)**2)\n    raise ValueError(f\'Unknown regulariser: {family}\')\n\n\ndef apply(image, sigma):\n    return np.fft.ifft2(np.fft.fft2(image, axes=(0, 1)) *\n                       transfer(image.shape[:2], sigma)[..., None], axes=(0, 1)).real\n\n\ndef inverse_spectrum(spectrum, sigma, family, strength, clip=True):\n    h = transfer(spectrum.shape[:2], sigma)\n    filt = h / (h**2 + strength * penalty(spectrum.shape[:2], family))\n    x = np.fft.ifft2(spectrum * filt[..., None], axes=(0, 1)).real\n    return np.clip(x, 0, 1) if clip else x\n\n\ndef interior(x, config):\n    b = config[\'context_border\']\n    return x[b:-b, b:-b] if b else x\n\n\ndef psnr(mse):\n    return float(-10 * np.log10(mse)) if mse > 0 else float(\'inf\')\n\n\ndef load_sources(data_dir, expected_hashes, config):\n    records, sources, seen = [], {}, set()\n    extent = config[\'crop_size\'] + 2 * config[\'context_border\']\n    for sid in config[\'source_ids\']:\n        path = Path(data_dir) / (sid + \'.png\')\n        with Image.open(path) as image:\n            image.verify()\n        with Image.open(path) as image:\n            image.load()\n            assert image.format == \'PNG\' and image.mode in (\'RGB\', \'RGBA\'), path.name\n            if image.mode == \'RGBA\':\n                assert image.getchannel(\'A\').getextrema() == (255, 255), \'Nonopaque alpha\'\n            width, height = image.size\n            rgb = np.asarray(image.convert(\'RGB\'))\n        pixel_hash = hashlib.sha256(rgb.tobytes()).hexdigest()\n        assert pixel_hash == expected_hashes[sid], f\'Official decoded RGB mismatch: {sid}\'\n        assert pixel_hash not in seen, \'Duplicate source\'\n        seen.add(pixel_hash)\n        assert min(width, height) >= extent\n        left, top = (width - extent) // 2, (height - extent) // 2\n        sources[sid] = rgb[top:top + extent, left:left + extent].astype(np.float64) / 255\n        records.append(dict(source_id=sid, filename=path.name, sha256=sha256(path),\n                            rgb_sha256=pixel_hash, width=width, height=height,\n                            crop_left=left, crop_top=top, extent=extent, role=\'development_only\'))\n    return sources, pd.DataFrame(records)\n\n\ndef observations(reference, sid, config):\n    identity = int(hashlib.sha256(sid.encode()).hexdigest()[:8], 16)\n    rng = np.random.default_rng(np.random.SeedSequence([config[\'seed\'], identity]))\n    # Common random numbers couple severities; these are not independent noise repeats.\n    z = rng.standard_normal(reference.shape)\n    for true_sigma in config[\'true_sigmas\']:\n        blurred = apply(reference, true_sigma)\n        for noise_std in config[\'noise_stds\']:\n            linear = blurred + noise_std * z\n            quantised = np.rint(np.clip(linear, 0, 1) * 255).astype(\'uint8\')\n            stream = io.BytesIO()\n            Image.fromarray(quantised).save(stream, format=\'JPEG\',\n                quality=config[\'jpeg_quality\'], subsampling=config[\'jpeg_subsampling\'], optimize=False)\n            stream.seek(0)\n            with Image.open(stream) as image:\n                jpeg = np.asarray(image.convert(\'RGB\'), dtype=np.float64) / 255\n            for scenario, y in [(\'blur_noise\', linear), (\'blur_noise_jpeg\', jpeg)]:\n                if scenario in config[\'scenarios\']:\n                    yield dict(source_id=sid, true_sigma=true_sigma, noise_std=noise_std,\n                               scenario=scenario), y\n\n\ndef validate_math():\n    """Independent dense solve on a tiny non-square grid, plus analytic DC checks."""\n    shape = (5, 6)\n    n = int(np.prod(shape))\n    eye = np.eye(n)\n    a = np.column_stack([apply(eye[:, j].reshape(*shape, 1), 1.3).ravel() for j in range(n)])\n    d = []\n    for axis in (0, 1):\n        d.append(np.column_stack([(np.roll(eye[:, j].reshape(shape), -1, axis=axis) -\n                                   eye[:, j].reshape(shape)).ravel() for j in range(n)]))\n    rng = np.random.default_rng(71)\n    y = rng.normal(size=(*shape, 1))\n    errors = {}\n    for family in (\'ridge\', \'gradient\'):\n        regulariser = np.eye(n) if family == \'ridge\' else sum(x.T @ x for x in d)\n        dense = np.linalg.solve(a.T @ a + 0.03 * regulariser, a.T @ y.ravel())\n        fast = inverse_spectrum(np.fft.fft2(y, axes=(0, 1)), 1.3, family, 0.03, clip=False)\n        errors[family] = float(np.max(np.abs(dense - fast.ravel())))\n        assert errors[family] < 1e-10\n    const = np.ones((*shape, 1))\n    assert np.allclose(apply(const, 1.3), const)\n    assert np.allclose(inverse_spectrum(np.fft.fft2(const, axes=(0, 1)), 1.3,\n                                       \'gradient\', 0.03, clip=False), const)\n    assert np.allclose(inverse_spectrum(np.fft.fft2(const, axes=(0, 1)), 1.3,\n                                       \'ridge\', 0.03, clip=False), const / 1.03)\n    assert np.isclose(psnr(0.01), 20)\n    # Selection arithmetic, checked against a hand-computable example.\n    assert np.isclose(selected_risk(np.array([.1, .4, .2, .3]),\n                                    np.array([1, 4, 2, 3]), .5), .15)\n    return dict(dense_solver_max_abs_error=errors, dc_checks=True, selection_check=True)\n\n\ndef metric_row(meta, image, truth, model, info=\'none\', strength=None, elapsed=None, **extra):\n    mse = float(np.mean((image - truth)**2))\n    return dict(**meta, model=model, operator_info=info, strength=strength,\n                mse=mse, psnr_db=psnr(mse), inverse_seconds=elapsed, **extra)\n\n\ndef grid_search(sources, config):\n    rows, controls = [], []\n    for sid, reference in sources.items():\n        truth = interior(reference, config)\n        for meta, y in observations(reference, sid, config):\n            spectrum = np.fft.fft2(y, axes=(0, 1))\n            controls.append(metric_row(meta, interior(np.clip(y, 0, 1), config), truth, \'observed\'))\n            controls.append(metric_row(meta, interior(np.clip(apply(y, config[\'smooth_sigma\']), 0, 1), config),\n                                       truth, \'smooth_0.5px\'))\n            for info, sigma in [(\'nominal\', config[\'nominal_sigma\']), (\'oracle_blur\', meta[\'true_sigma\'])]:\n                for family in config[\'families\']:\n                    for strength in config[\'lambdas\']:\n                        tic = time.perf_counter()\n                        estimate = inverse_spectrum(spectrum, sigma, family, strength)\n                        elapsed = time.perf_counter() - tic\n                        rows.append(metric_row(meta, interior(estimate, config), truth,\n                                               family, info, strength, elapsed))\n        print(f\'Grid complete: {sid}\', flush=True)\n    return pd.DataFrame(rows), pd.DataFrame(controls)\n\n\ndef choose_source_excluded(grid, config):\n    selections, chosen = [], []\n    for sid in config[\'source_ids\']:\n        training = grid[grid.source_id != sid]\n        for (family, info), frame in training.groupby([\'model\', \'operator_info\']):\n            # Equal source and condition weighting. Tie-break is ascending lambda.\n            means = frame.groupby(\'strength\').mse.mean().sort_index()\n            strength = float(means.idxmin())\n            used = sorted(frame.source_id.unique().tolist())\n            assert sid not in used and len(used) == len(config[\'source_ids\']) - 1\n            selections.append(dict(excluded_source=sid, model=family, operator_info=info,\n                selected_lambda=strength, tuning_sources=\';\'.join(used),\n                tuning_mean_mse=float(means.loc[strength]),\n                grid_boundary=bool(strength in (min(config[\'lambdas\']), max(config[\'lambdas\'])))))\n            rows = grid[(grid.source_id == sid) & (grid.model == family) &\n                        (grid.operator_info == info) & (grid.strength == strength)].copy()\n            rows[\'tuning_sources\'] = \';\'.join(used)\n            chosen.append(rows)\n    return pd.DataFrame(selections), pd.concat(chosen, ignore_index=True)\n\n\ndef patch_mean(array, patch_size):\n    h, w = array.shape\n    assert h % patch_size == w % patch_size == 0\n    return array.reshape(h // patch_size, patch_size, w // patch_size, patch_size).mean(axis=(1, 3))\n\n\ndef selected_risk(errors, scores, coverage, seed=82):\n    errors, scores = np.asarray(errors).ravel(), np.asarray(scores).ravel()\n    assert len(errors) == len(scores) > 0 and 0 < coverage <= 1\n    order = np.lexsort((np.random.default_rng(seed).random(len(scores)), scores))\n    count = int(np.ceil(coverage * len(scores)))\n    return float(errors[order[:count]].mean())\n\n\ndef score_diagnostic(sources, selections, config):\n    rows, examples = [], []\n    for sid, reference in sources.items():\n        truth = interior(reference, config)\n        gy, gx = np.gradient(truth.mean(axis=-1))\n        texture = patch_mean(np.hypot(gx, gy), config[\'patch_size\']).ravel()\n        # Reference-defined stratum is evaluation-only, never an operational score.\n        textured = np.argsort(texture, kind=\'stable\')[-int(np.ceil(len(texture) / 4)):]\n        for meta, y in observations(reference, sid, config):\n            spectrum = np.fft.fft2(y, axes=(0, 1))\n            base = dict(reference=truth, observed=interior(np.clip(y, 0, 1), config))\n            for family in config[\'families\']:\n                pick = selections[(selections.excluded_source == sid) &\n                                  (selections.model == family) & (selections.operator_info == \'nominal\')].iloc[0]\n                strength = float(pick.selected_lambda)\n                sigma = config[\'nominal_sigma\']\n                est = inverse_spectrum(spectrum, sigma, family, strength)\n                operators = [inverse_spectrum(spectrum, sigma * f, family, strength)\n                             for f in config[\'operator_factors\']]\n                key = f"noise_control:{sid}:{meta[\'true_sigma\']}:{meta[\'noise_std\']}"\n                seed = int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)\n                # Fixed amplitude: the control does not receive true acquisition noise.\n                noise = np.random.default_rng(seed).normal(0, config[\'noise_probe_std\'], y.shape)\n                noises = [inverse_spectrum(np.fft.fft2(y + s * noise, axes=(0, 1)),\n                          sigma, family, strength) for s in (-1, 0, 1)]\n                gy, gx = np.gradient(est.mean(axis=-1))\n                scores = {\n                    \'operator_spread\': np.sqrt(np.mean(np.var(operators, axis=0), axis=-1)),\n                    \'measurement_noise_spread\': np.sqrt(np.mean(np.var(noises, axis=0), axis=-1)),\n                    \'measurement_residual\': np.sqrt(np.mean((apply(est, sigma) - y)**2, axis=-1)),\n                    \'image_gradient\': np.hypot(gx, gy),\n                }\n                errors = patch_mean(np.mean((interior(est, config) - truth)**2, axis=-1),\n                                    config[\'patch_size\']).ravel()\n                mapped = {name: patch_mean(interior(score, config), config[\'patch_size\']).ravel()\n                          for name, score in scores.items()}\n                for region, index in [(\'all\', np.arange(len(errors))), (\'textured_quartile\', textured)]:\n                    e = errors[index]\n                    for coverage in config[\'coverages\']:\n                        oracle = selected_risk(e, e, coverage)\n                        for name, score in mapped.items():\n                            risk = selected_risk(e, score[index], coverage)\n                            assert risk >= oracle - 1e-12\n                            if coverage == 1:\n                                assert np.isclose(risk, e.mean())\n                            rows.append(dict(**meta, model=family, strength=strength, region=region,\n                                score=name, coverage=coverage, retained_patches=int(np.ceil(coverage * len(e))),\n                                available_patches=len(e), mse=risk))\n                        for name, risk in [(\'random_expected\', float(e.mean())), (\'oracle_error\', oracle)]:\n                            rows.append(dict(**meta, model=family, strength=strength, region=region,\n                                score=name, coverage=coverage, retained_patches=int(np.ceil(coverage * len(e))),\n                                available_patches=len(e), mse=risk))\n                base[family] = interior(est, config)\n            if meta[\'true_sigma\'] == 1.6 and meta[\'noise_std\'] == 2 / 255 and meta[\'scenario\'] == \'blur_noise\':\n                examples.append(dict(source_id=sid, **base))\n        print(f\'Patch diagnostics complete: {sid}\', flush=True)\n    return pd.DataFrame(rows), examples\n\n\ndef quality_summary(selected, controls):\n    frame = pd.concat([selected, controls], ignore_index=True)\n    result = frame.groupby([\'scenario\', \'model\', \'operator_info\'], as_index=False).agg(\n        mean_mse=(\'mse\', \'mean\'), mean_image_psnr_db=(\'psnr_db\', \'mean\'),\n        sources=(\'source_id\', \'nunique\'), rows=(\'mse\', \'size\'))\n    result[\'pooled_psnr_db\'] = result.mean_mse.map(psnr)\n    return result\n\n\ndef plot_figures(grid, selected, controls, curves, examples, output_dir):\n    plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 10,\n        \'axes.spines.top\': False, \'axes.spines.right\': False, \'figure.facecolor\': \'white\'})\n    colors = {\'ridge\': \'#236A9B\', \'gradient\': \'#C46F28\'}\n    figs = []\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)\n    for ax, scenario in zip(axes, [\'blur_noise\', \'blur_noise_jpeg\']):\n        for family in colors:\n            series = grid[(grid.scenario == scenario) & (grid.operator_info == \'nominal\') &\n                          (grid.model == family)].groupby(\'strength\').mse.mean()\n            ax.plot(series.index, series * 1000, \'-o\', color=colors[family], label=family)\n        baseline = controls[(controls.scenario == scenario) & (controls.model == \'observed\')].mse.mean()\n        ax.axhline(baseline * 1000, color=\'#555555\', ls=\'--\', label=\'degraded input\')\n        ax.set_xscale(\'log\'); ax.set_yscale(\'log\')\n        ax.set_title(\'Blur + noise\' if scenario == \'blur_noise\' else \'Blur + noise + JPEG chain\')\n        ax.set_xlabel(\'Regularisation strength\'); ax.grid(axis=\'y\', alpha=.2)\n    axes[0].set_ylabel(\'Mean RGB MSE (x 0.001)\'); axes[1].legend(frameon=False)\n    fig.suptitle(\'Development sweep | Nominal blur fixed at 1.0 px\')\n    fig.tight_layout(); fig.savefig(output_dir / \'regularisation_sweep.png\', dpi=140); figs.append(fig)\n\n    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True, sharey=True)\n    for row, scenario in enumerate([\'blur_noise\', \'blur_noise_jpeg\']):\n        for col, info in enumerate([\'nominal\', \'oracle_blur\']):\n            ax = axes[row, col]\n            for j, family in enumerate(colors):\n                a = selected[(selected.scenario == scenario) & (selected.operator_info == info) &\n                             (selected.model == family)].groupby(\'source_id\').mse.mean()\n                b = controls[(controls.scenario == scenario) & (controls.model == \'observed\')].groupby(\'source_id\').mse.mean()\n                gain = 10 * np.log10(b / a)\n                ax.plot(np.arange(len(gain)) + (j - .5) * .08, gain.values,\n                        \'o\' if j == 0 else \'s\', color=colors[family], label=family)\n            ax.axhline(0, ls=\'--\', color=\'#555555\'); ax.grid(axis=\'y\', alpha=.2)\n            ax.set_xticks(range(4), [\'0801\', \'0802\', \'0803\', \'0804\'])\n            ax.set_title((\'Linear\' if row == 0 else \'JPEG chain\') + \' | \' + info)\n            if col == 0: ax.set_ylabel(\'PSNR gain over input (dB)\')\n    axes[0, 1].legend(frameon=False)\n    fig.suptitle(\'Source-excluded tuning | Each point pools six conditions for one development source\')\n    fig.tight_layout(); fig.savefig(output_dir / \'source_excluded_gains.png\', dpi=140); figs.append(fig)\n\n    score_colors = {\'operator_spread\': \'#236A9B\', \'measurement_noise_spread\': \'#8B5C94\',\n                    \'measurement_residual\': \'#C46F28\', \'image_gradient\': \'#778839\',\n                    \'random_expected\': \'#555555\'}\n    styles = [\'-o\', \'--s\', \'-.^\', \':D\', \'--\']\n    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=\'row\')\n    for row, region in enumerate([\'all\', \'textured_quartile\']):\n        for col, family in enumerate([\'ridge\', \'gradient\']):\n            ax = axes[row, col]\n            # JPEG-only plot; complete tables preserve both scenarios.\n            a = curves[(curves.region == region) & (curves.model == family) &\n                       (curves.scenario == \'blur_noise_jpeg\')]\n            for (score, color), style in zip(score_colors.items(), styles):\n                series = a[a.score == score].groupby(\'coverage\').mse.mean()\n                ax.plot(series.index * 100, series * 1000, style, color=color,\n                        label=score.replace(\'_\', \' \'), markersize=4)\n            ax.set_title(f\'{family} | {region.replace("_", " ")}\')\n            ax.set_ylim(bottom=0); ax.grid(axis=\'y\', alpha=.2)\n            if row == 1: ax.set_xlabel(\'Patches retained within stratum (%)\')\n            if col == 0: ax.set_ylabel(\'Retained patch MSE (x 0.001)\')\n    handles, labels = axes[0, 0].get_legend_handles_labels()\n    fig.legend(handles, labels, loc=\'lower center\', ncol=3, frameon=False, fontsize=9)\n    fig.suptitle(\'JPEG-chain patch selection | Development diagnostic; no calibration guarantee\')\n    fig.tight_layout(rect=(0, .09, 1, .96))\n    fig.savefig(output_dir / \'patch_risk_coverage.png\', dpi=140); figs.append(fig)\n\n    fig, axes = plt.subplots(len(examples), 4, figsize=(11, 2.7 * len(examples)), squeeze=False)\n    for row, example in enumerate(examples):\n        for col, key in enumerate([\'reference\', \'observed\', \'ridge\', \'gradient\']):\n            axes[row, col].imshow(example[key]); axes[row, col].axis(\'off\')\n            axes[row, col].set_title(f\'{example["source_id"]} | {key}\')\n    fig.suptitle(\'Anchor condition | True blur 1.6 px, noise 2/255; nominal inverse blur 1.0 px\')\n    fig.tight_layout(); fig.savefig(output_dir / \'reconstruction_examples.png\', dpi=120); figs.append(fig)\n    return figs\n\n\ndef save_results(result, output_dir):\n    output_dir = Path(output_dir)\n    for key in [\'manifest\', \'grid\', \'controls\', \'selections\', \'selected\', \'summary\', \'curves\']:\n        result[key].to_csv(output_dir / f\'{key}.csv\', index=False)\n    for key in [\'config\', \'checks\', \'environment\']:\n        (output_dir / f\'{key}.json\').write_text(json.dumps(result[key], indent=2) + \'\\n\')\n    files = sorted(p for p in output_dir.iterdir() if p.is_file() and p.name != \'export_manifest.json\')\n    payload = {\'experiment\': \'pilot_01\', \'role\': \'development_only\',\n               \'files\': [dict(name=p.name, size_bytes=p.stat().st_size, sha256=sha256(p)) for p in files]}\n    (output_dir / \'export_manifest.json\').write_text(json.dumps(payload, indent=2) + \'\\n\')\n    assert all(sha256(output_dir / r[\'name\']) == r[\'sha256\'] for r in payload[\'files\'])\n\n\ndef run(data_dir, output_dir, expected_hashes, config=None):\n    config = json.loads(json.dumps(CONFIG if config is None else config))\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=False)\n    assert Path(data_dir).resolve() != output_dir.resolve()\n    checks = validate_math()\n    tic = time.perf_counter()\n    sources, manifest = load_sources(data_dir, expected_hashes, config)\n    grid, controls = grid_search(sources, config)\n    selections, selected = choose_source_excluded(grid, config)\n    curves, examples = score_diagnostic(sources, selections, config)\n    summary = quality_summary(selected, controls)\n    n_obs = len(sources) * len(config[\'true_sigmas\']) * len(config[\'noise_stds\']) * len(config[\'scenarios\'])\n    assert len(grid) == n_obs * 2 * len(config[\'families\']) * len(config[\'lambdas\'])\n    assert len(controls) == n_obs * 2\n    assert len(selected) == n_obs * 2 * len(config[\'families\'])\n    for row in manifest.itertuples():\n        assert sha256(Path(data_dir) / row.filename) == row.sha256\n    # Anchor reproduction: original Pilot 00\'s fixed ridge setting and same source/noise identity.\n    anchor = grid[(grid.true_sigma == 1.6) & (grid.noise_std == 2 / 255) &\n                  (grid.model == \'ridge\') & (grid.strength == .002)]\n    checks.update(source_count=len(sources), observation_count=n_obs, grid_rows=len(grid),\n        selected_inverse_rows=len(selected), control_rows=len(controls), patch_curve_rows=len(curves),\n        input_hashes_unchanged=True, tuning_excludes_evaluated_source=True,\n        source_role=\'development_only\', elapsed_seconds=time.perf_counter() - tic,\n        anchor_pooled_psnr={f\'{a}_{b}\': psnr(float(f.mse.mean()))\n                           for (a, b), f in anchor.groupby([\'scenario\', \'operator_info\'])})\n    environment = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,\n                       pillow=PIL.__version__, matplotlib=matplotlib.__version__, platform=platform.platform())\n    result = dict(config=config, checks=checks, environment=environment, sources=sources,\n                  manifest=manifest, grid=grid, controls=controls, selections=selections,\n                  selected=selected, summary=summary, curves=curves, examples=examples)\n    result[\'figures\'] = plot_figures(grid, selected, controls, curves, examples, output_dir)\n    for figure in result[\'figures\']:\n        plt.close(figure)\n    save_results(result, output_dir)\n    return result\n', 'learned': '"""Frozen DRUNet/DPIR-style development adapter; no independent-test claims.\n\nCaller supplies the audited Baseline 01 module as B01, the official vendor\ndirectory, the provenance manifest and the expected source-image hashes.\n"""\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport platform\nimport resource\nimport sys\nimport time\nimport types\nimport urllib.request\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport PIL\nimport torch\n\nB01 = None\nCONFIG = {\n    \'source_ids\': [\'0801\', \'0802\', \'0803\', \'0804\'], \'seed\': 20260917,\n    \'crop_size\': 512, \'context_border\': 32, \'true_sigmas\': [1.6],\n    \'noise_stds\': [2 / 255], \'nominal_sigma\': 1.0, \'nominal_noise_std\': 2 / 255,\n    \'scenarios\': [\'blur_noise\', \'blur_noise_jpeg\'], \'jpeg_quality\': 75,\n    \'jpeg_subsampling\': 0, \'gradient_lambda\': 0.05, \'iterations\': 8,\n    \'model_sigma_start_255\': 49.0, \'model_sigma_end_255\': 2.0,\n    \'prior_tradeoff\': 0.23, \'periodic_x8\': True,\n    \'operator_sigmas\': [0.8, 1.0, 1.2], \'outer_rotations\': [0, 1, 2],\n    \'patch_size\': 16, \'coverages\': [0.5, 0.75, 0.9, 1.0],\n    \'detail_blur_sigma\': 1.0, \'detail_rmse_tolerances\': [0.025, 0.05, 0.10],\n    \'cpu_threads\': 4, \'role\': \'development_only\',\n    \'model_information\': \'fixed nominal blur/noise; true blur only in oracle diagnostic\',\n    \'uncertainty_interpretation\': \'heuristic sensitivities; not posterior samples or calibrated uncertainty\',\n}\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef download_weights(destination, provenance):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if not destination.exists():\n        partial = destination.with_suffix(\'.part\')\n        try:\n            with urllib.request.urlopen(provenance[\'weight_url\'], timeout=60) as response, partial.open(\'wb\') as out:\n                while True:\n                    chunk = response.read(4 * 1024 * 1024)\n                    if not chunk:\n                        break\n                    out.write(chunk)\n            assert partial.stat().st_size == provenance[\'weight_bytes\'], \'Incomplete checkpoint download\'\n            assert digest(partial) == provenance[\'weight_sha256\'], \'Checkpoint hash mismatch\'\n            partial.replace(destination)\n        finally:\n            if partial.exists():\n                partial.unlink()\n    assert destination.stat().st_size == provenance[\'weight_bytes\']\n    assert digest(destination) == provenance[\'weight_sha256\'], \'Cached checkpoint mismatch\'\n    return destination\n\n\ndef load_model(vendor_dir, checkpoint, provenance, config):\n    vendor_dir = Path(vendor_dir)\n    for relative, expected in provenance[\'files\'].items():\n        assert digest(vendor_dir / relative) == expected, f\'Vendor file changed: {relative}\'\n    torch.set_num_threads(config[\'cpu_threads\'])\n    torch.manual_seed(config[\'seed\'])\n    torch.use_deterministic_algorithms(True)\n    if torch.cuda.is_available():\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cuda.matmul.allow_tf32 = False\n    # Private module names prevent collisions with another notebook\'s `models`.\n    spec = importlib.util.spec_from_file_location(\'dpir02_basicblock\', vendor_dir / \'models/basicblock.py\')\n    basic = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = basic\n    spec.loader.exec_module(basic)\n    network = types.ModuleType(\'dpir02_network\')\n    source = (vendor_dir / \'models/network_unet.py\').read_text()\n    assert source.count(\'import models.basicblock as B\') == 1\n    exec(compile(source.replace(\'import models.basicblock as B\', \'import dpir02_basicblock as B\'),\n                 str(vendor_dir / \'models/network_unet.py\'), \'exec\'), network.__dict__)\n    model = network.UNetRes(in_nc=4, out_nc=3, nc=[64, 128, 256, 512], nb=4,\n        act_mode=\'R\', downsample_mode=\'strideconv\', upsample_mode=\'convtranspose\')\n    weights = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n    assert isinstance(weights, dict) and all(isinstance(v, torch.Tensor) for v in weights.values())\n    model.load_state_dict(weights, strict=True)\n    model.eval().requires_grad_(False)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = model.to(device)\n    assert sum(p.numel() for p in model.parameters()) == 32640960\n    print(f\'Loaded verified DRUNet: 32,640,960 parameters; device={device}\', flush=True)\n    return model, device\n\n\ndef augment(x, mode):\n    # Same mode ordering as official utils_image.augment_img_tensor4.\n    if mode == 0: return x\n    if mode == 1: return x.rot90(1, [2, 3]).flip([2])\n    if mode == 2: return x.flip([2])\n    if mode == 3: return x.rot90(3, [2, 3])\n    if mode == 4: return x.rot90(2, [2, 3]).flip([2])\n    if mode == 5: return x.rot90(1, [2, 3])\n    if mode == 6: return x.rot90(2, [2, 3])\n    if mode == 7: return x.rot90(3, [2, 3]).flip([2])\n    raise ValueError(mode)\n\n\ndef undo_augment(x, mode):\n    return augment(x, 8 - mode if mode in (3, 5) else mode)\n\n\ndef schedule(config):\n    sigmas = np.logspace(np.log10(config[\'model_sigma_start_255\']),\n        np.log10(config[\'model_sigma_end_255\']), config[\'iterations\']).astype(np.float32) / 255\n    rhos = np.array([config[\'prior_tradeoff\'] * config[\'nominal_noise_std\']**2 / s**2\n                    for s in sigmas], dtype=np.float32)\n    return rhos, sigmas\n\n\ndef data_step(z, fy, h, rho):\n    return torch.fft.ifft2((h.conj() * fy + rho * torch.fft.fft2(z)) /\n                           (h.abs().square() + rho)).real\n\n\ndef sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n\n\ndef to_tensor(image, device):\n    return torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1), dtype=np.float32))[None].to(device)\n\n\ndef reconstruct(observation, blur_sigma, model, device, config, denoise_only=False):\n    assert observation.ndim == 3 and observation.shape[-1] == 3\n    assert observation.shape[0] % 8 == observation.shape[1] % 8 == 0\n    sync(device)\n    if device.type == \'cuda\': torch.cuda.reset_peak_memory_stats(device)\n    tic = time.perf_counter()\n    trace = []\n    with torch.inference_mode():\n        y = to_tensor(observation, device)\n        z = y.clone()\n        if denoise_only:\n            sigma = config[\'nominal_noise_std\']\n            z = model(torch.cat([z, torch.full_like(z[:, :1], sigma)], dim=1))\n            calls = 1\n        else:\n            h = torch.as_tensor(B01.transfer(observation.shape[:2], blur_sigma),\n                                dtype=torch.float32, device=device)[None, None]\n            fy = torch.fft.fft2(y)\n            rhos, sigmas = schedule(config)\n            for i, (rho, sigma) in enumerate(zip(rhos, sigmas)):\n                previous = z\n                x = data_step(z, fy, h, float(rho))\n                mode = i % 8 if config[\'periodic_x8\'] else 0\n                x = augment(x, mode)\n                x = model(torch.cat([x, torch.full_like(x[:, :1], float(sigma))], dim=1))\n                z = undo_augment(x, mode)\n                assert bool(z.isfinite().all()), f\'Non-finite learned output at iteration {i}\'\n                residual = torch.fft.ifft2(torch.fft.fft2(z) * h).real - y\n                trace.append(dict(iteration=i + 1, rho=float(rho), denoiser_sigma=float(sigma),\n                    iterate_change_mse=float((z - previous).square().mean()),\n                    measurement_residual_mse=float(residual.square().mean())))\n            calls = config[\'iterations\']\n        assert bool(z.isfinite().all())\n        clipped_fraction = float(((z < 0) | (z > 1)).float().mean())\n        result = z.clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.float64)\n    sync(device)\n    timing = dict(elapsed_seconds=time.perf_counter() - tic, denoiser_calls=calls,\n        input_height=observation.shape[0], input_width=observation.shape[1],\n        clipped_fraction=clipped_fraction,\n        peak_cuda_allocated_mib=(torch.cuda.max_memory_allocated(device) / 2**20 if device.type == \'cuda\' else None),\n        process_peak_rss_mib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024)\n    return result, timing, trace\n\n\ndef validate_adapter(model, device, vendor_dir, config):\n    # Independent tiny dense objective, including a nonzero HQS prior centre.\n    shape = (5, 6); n = int(np.prod(shape)); eye = np.eye(n)\n    a = np.column_stack([B01.apply(eye[:, j].reshape(*shape, 1), 1.3).ravel() for j in range(n)])\n    rng = np.random.default_rng(37)\n    y, z = rng.normal(size=(2, *shape)); rho = .03\n    dense = np.linalg.solve(a.T @ a + rho * np.eye(n), a.T @ y.ravel() + rho * z.ravel())\n    ht = torch.from_numpy(B01.transfer(shape, 1.3))\n    actual = data_step(torch.from_numpy(z), torch.fft.fft2(torch.from_numpy(y)), ht, rho).numpy()\n    err = float(abs(dense - actual.ravel()).max()); assert err < 1e-10\n    for mode in range(8):\n        x = torch.arange(1 * 3 * 16 * 24).reshape(1, 3, 16, 24)\n        assert torch.equal(undo_augment(augment(x, mode), mode), x)\n    image = rng.normal(size=(32, 40, 3))\n    for rotation in config[\'outer_rotations\']:\n        assert np.allclose(B01.apply(np.rot90(image, rotation), 1.3),\n                           np.rot90(B01.apply(image, 1.3), rotation), atol=1e-12)\n    spec = importlib.util.spec_from_file_location(\'dpir02_schedule\', Path(vendor_dir) / \'utils/utils_pnp.py\')\n    official = importlib.util.module_from_spec(spec); spec.loader.exec_module(official)\n    er, es = official.get_rho_sigma(sigma=config[\'nominal_noise_std\'], iter_num=config[\'iterations\'],\n        modelSigma1=config[\'model_sigma_start_255\'], modelSigma2=config[\'model_sigma_end_255\'], w=1.0)\n    ar, ass = schedule(config)\n    assert np.allclose(ar, er, rtol=1e-6) and np.array_equal(ass, es)\n    # Smoke test actual learned weights, including an independent repeat.\n    x = torch.zeros(1, 4, 64, 64, device=device); x[:, 3] = 2 / 255\n    with torch.inference_mode():\n        one, two = model(x), model(x)\n        assert one.shape == (1, 3, 64, 64) and bool(one.isfinite().all())\n        assert torch.equal(one, two)\n    return dict(dense_hqs_max_abs_error=err, transform_inverse_checks=True,\n        isotropic_operator_rotation_equivariance=True, official_schedule_matches=True,\n        strict_weight_load=True, learned_smoke_finite=True, repeat_smoke_identical=True)\n\n\ndef detail(image, config):\n    return image - B01.apply(image, config[\'detail_blur_sigma\'])\n\n\ndef evaluate_scores(truth, nominal, observation, operators, transforms, config):\n    inside = lambda x: B01.interior(x, config)\n    pool = lambda x: B01.patch_mean(x, config[\'patch_size\']).ravel()\n    rgb_error = pool(np.mean((inside(nominal) - inside(truth))**2, axis=-1))\n    detail_error = pool(np.mean(inside(detail(nominal, config) - detail(truth, config))**2, axis=-1))\n    gy, gx = np.gradient(nominal.mean(axis=-1))\n    raw_scores = {\n        \'operator_spread_detail\': np.sqrt(np.var([detail(x, config) for x in operators], axis=0).mean(axis=-1)),\n        \'image_transform_spread_detail\': np.sqrt(np.var([detail(x, config) for x in transforms], axis=0).mean(axis=-1)),\n        \'operator_spread_rgb\': np.sqrt(np.var(operators, axis=0).mean(axis=-1)),\n        \'measurement_residual\': np.sqrt(np.mean((B01.apply(nominal, config[\'nominal_sigma\']) - observation)**2, axis=-1)),\n        \'image_gradient\': np.hypot(gx, gy),\n    }\n    scores = {name: pool(inside(value)) for name, value in raw_scores.items()}\n    gy, gx = np.gradient(inside(truth).mean(axis=-1)); texture = pool(np.hypot(gx, gy))\n    textured = np.argsort(texture, kind=\'stable\')[-int(np.ceil(len(texture) / 4)):]\n    rows = []\n    for region, ids in [(\'all\', np.arange(len(rgb_error))), (\'textured_quartile\', textured)]:\n        re, de = rgb_error[ids], detail_error[ids]\n        for name, score in list(scores.items()) + [(\'random_expected\', None), (\'oracle_detail_error\', detail_error)]:\n            order = None if score is None else np.lexsort((np.random.default_rng(82).random(len(ids)), score[ids]))\n            for coverage in config[\'coverages\']:\n                count = int(np.ceil(coverage * len(ids)))\n                chosen = np.arange(len(ids)) if order is None else order[:count]\n                # Random\'s reported risk is its exact expectation, not an actual selected mask.\n                row = dict(region=region, score=name, coverage=coverage,\n                    available_patches=len(ids), retained_patches=count,\n                    rgb_mse=float(re[chosen].mean()), detail_mse=float(de[chosen].mean()))\n                for threshold in config[\'detail_rmse_tolerances\']:\n                    row[f\'bad_detail_rate_{threshold:g}\'] = float((np.sqrt(de[chosen]) > threshold).mean())\n                if coverage == 1:\n                    assert np.isclose(row[\'rgb_mse\'], re.mean()) and np.isclose(row[\'detail_mse\'], de.mean())\n                if order is not None:\n                    oracle = np.sort(de)[:count].mean()\n                    assert row[\'detail_mse\'] >= oracle - 1e-12\n                rows.append(row)\n    return pd.DataFrame(rows), dict(rgb_patch_error=rgb_error, detail_patch_error=detail_error,\n        reference_texture=texture, **scores)\n\n\ndef plot_results(quality, curves, trajectories, examples, output_dir):\n    plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 10,\n                         \'axes.spines.top\': False, \'axes.spines.right\': False})\n    figs = []; scenarios = quality.scenario.unique().tolist()\n    fig, axes = plt.subplots(1, len(scenarios), figsize=(6.2 * len(scenarios), 4.2), squeeze=False)\n    for ax, scenario in zip(axes[0], scenarios):\n        q = quality[quality.scenario == scenario].groupby(\'model\').mse.mean()\n        order = [\'observed\', \'gradient_nominal\', \'drunet_denoise_only\', \'dpir_nominal\', \'dpir_oracle_blur\']\n        values = -10 * np.log10(q.reindex(order))\n        ax.barh(range(len(order)), values, color=[\'#777777\',\'#C46F28\',\'#8B5C94\',\'#236A9B\',\'#599795\'])\n        ax.set_yticks(range(len(order)), [x.replace(\'_\', \' \') for x in order]); ax.invert_yaxis()\n        ax.set_xlabel(\'Pooled RGB PSNR (dB)\'); ax.set_title(scenario.replace(\'_\', \' + \')); ax.set_xlim(left=0)\n    fig.suptitle(\'Learned baseline development | Executed sources only\')\n    fig.tight_layout(); fig.savefig(output_dir / \'quality.png\', dpi=140); figs.append(fig)\n    fig, axes = plt.subplots(2, len(scenarios), figsize=(6.2 * len(scenarios), 7.5), squeeze=False)\n    palette = {\'operator_spread_detail\':\'#236A9B\', \'image_transform_spread_detail\':\'#8B5C94\',\n        \'measurement_residual\':\'#C46F28\', \'image_gradient\':\'#778839\', \'random_expected\':\'#666666\'}\n    for col, scenario in enumerate(scenarios):\n        for row, region in enumerate([\'all\',\'textured_quartile\']):\n            ax = axes[row,col]\n            subset = curves[(curves.scenario == scenario) & (curves.region == region)]\n            for (name,color),style in zip(palette.items(),[\'-o\',\'--s\',\'-.^\',\':D\',\'--\']):\n                s = subset[subset.score == name].groupby(\'coverage\').detail_mse.mean()\n                ax.plot(s.index*100,s.values*1e4,style,color=color,label=name.replace(\'_\',\' \'),markersize=4)\n            ax.set_title(f\'{scenario} | {region}\'.replace(\'_\',\' \')); ax.set_ylim(bottom=0)\n            ax.set_ylabel(\'Retained detail MSE (× 0.0001)\'); ax.set_xlabel(\'Patches retained within stratum (%)\')\n    handles,labels=axes[0,0].get_legend_handles_labels()\n    fig.legend(handles,labels,loc=\'lower center\',ncol=2,fontsize=8,frameon=False)\n    fig.suptitle(\'DPIR detail selection | Heuristic scores; no calibration guarantee\')\n    fig.tight_layout(rect=(0,.12,1,.94)); fig.savefig(output_dir/\'detail_risk_coverage.png\',dpi=140);figs.append(fig)\n    fig, axes=plt.subplots(len(examples),4,figsize=(11,2.7*len(examples)),squeeze=False)\n    for i,ex in enumerate(examples):\n        for j,key in enumerate([\'reference\',\'observed\',\'gradient_nominal\',\'dpir_nominal\']):\n            axes[i,j].imshow(ex[key]);axes[i,j].axis(\'off\');axes[i,j].set_title(ex[\'source_id\']+\' | \'+key.replace(\'_\',\' \'))\n    fig.suptitle(\'Full-resolution anchor crops | Linear condition when available\')\n    fig.tight_layout();fig.savefig(output_dir/\'reconstruction_examples.png\',dpi=120);figs.append(fig)\n    fig, axes=plt.subplots(1,2,figsize=(11,4.2))\n    for (sid,scenario),frame in trajectories[trajectories.variant==\'nominal\'].groupby([\'source_id\',\'scenario\']):\n        axes[0].plot(frame.iteration,frame.measurement_residual_mse,label=sid+\' \'+scenario)\n        axes[1].plot(frame.iteration,frame.iterate_change_mse,label=sid+\' \'+scenario)\n    for ax,title in zip(axes,[\'Measurement residual MSE\',\'Iterate-change MSE\']):\n        ax.set_yscale(\'log\');ax.set_xlabel(\'Iteration\');ax.set_title(title)\n    axes[1].legend(fontsize=7,frameon=False)\n    fig.suptitle(\'Nominal DPIR trajectories | Changing schedules are not convergence proofs\')\n    fig.tight_layout();fig.savefig(output_dir/\'iteration_trajectories.png\',dpi=140);figs.append(fig)\n    return figs\n\n\ndef save_snapshot(result, output_dir, status):\n    for name in [\'quality\',\'summary\',\'compute\',\'curves\',\'trajectories\',\'source_manifest\']:\n        if name in result: result[name].to_csv(output_dir / (name+\'.csv\'),index=False)\n    for name in [\'config\',\'checks\',\'environment\',\'provenance\']:\n        (output_dir / (name+\'.json\')).write_text(json.dumps(result[name],indent=2)+\'\\n\')\n    (output_dir/\'status.json\').write_text(json.dumps({\'status\':status,\n        \'completed_observations\':result.get(\'completed_observations\',0),\n        \'planned_observations\':len(result[\'config\'][\'source_ids\'])*len(result[\'config\'][\'scenarios\']),\n        \'updated_utc\':datetime.now(timezone.utc).isoformat()},indent=2)+\'\\n\')\n    files=sorted(p for p in output_dir.rglob(\'*\') if p.is_file() and p.name!=\'export_manifest.json\')\n    manifest=[{\'name\':str(p.relative_to(output_dir)),\'bytes\':p.stat().st_size,\'sha256\':digest(p)} for p in files]\n    (output_dir/\'export_manifest.json\').write_text(json.dumps({\'experiment\':\'learned_02\',\'role\':\'development_only\',\'files\':manifest},indent=2)+\'\\n\')\n    assert all(digest(output_dir/r[\'name\'])==r[\'sha256\'] for r in manifest)\n\n\ndef run(data_dir, output_dir, vendor_dir, checkpoint, provenance, expected_rgb_hashes, config=None):\n    config=json.loads(json.dumps(CONFIG if config is None else config))\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=False)\n    (output_dir/\'predictions\').mkdir()\n    model,device=load_model(vendor_dir,checkpoint,provenance,config)\n    checks=validate_adapter(model,device,vendor_dir,config)\n    sources,manifest=B01.load_sources(data_dir,expected_rgb_hashes,config)\n    result=dict(config=config,checks=checks,provenance=provenance,source_manifest=manifest,\n        environment={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\n            \'torch\':torch.__version__,\'pillow\':PIL.__version__,\'matplotlib\':matplotlib.__version__,\n            \'device\':str(device),\'cuda_name\':torch.cuda.get_device_name(device) if device.type==\'cuda\' else None,\n            \'cpu_threads\':torch.get_num_threads(),\'deterministic_algorithms\':torch.are_deterministic_algorithms_enabled(),\n            \'cpu_memory_note\':\'process lifetime peak RSS; not isolated per-inference memory\'},completed_observations=0)\n    save_snapshot(result,output_dir,\'running\')\n    quality=[];compute=[];curves=[];trajectories=[];examples=[]\n    for sid,truth in sources.items():\n        for meta,y in B01.observations(truth,sid,config):\n            inside=lambda x:B01.interior(x,config)\n            print(f\'Starting {sid} | {meta["scenario"]}\',flush=True)\n            estimates={\'observed\':np.clip(y,0,1)}\n            tic=time.perf_counter()\n            estimates[\'gradient_nominal\']=B01.inverse_spectrum(np.fft.fft2(y,axes=(0,1)),\n                config[\'nominal_sigma\'],\'gradient\',config[\'gradient_lambda\'])\n            compute.append(dict(**meta,variant=\'gradient_nominal\',elapsed_seconds=time.perf_counter()-tic,denoiser_calls=0))\n            learned={}\n            variants=[(\'nominal\',config[\'nominal_sigma\'],0,False),(\'oracle_blur\',meta[\'true_sigma\'],0,False),\n                (\'operator_0.8\',config[\'operator_sigmas\'][0],0,False),(\'operator_1.2\',config[\'operator_sigmas\'][2],0,False),\n                (\'rotation_90\',config[\'nominal_sigma\'],1,False),(\'rotation_180\',config[\'nominal_sigma\'],2,False),\n                (\'denoise_only\',config[\'nominal_sigma\'],0,True)]\n            for name,sigma,rotation,only in variants:\n                observed=np.rot90(y,rotation).copy()\n                x,cost,trace=reconstruct(observed,sigma,model,device,config,denoise_only=only)\n                learned[name]=np.rot90(x,-rotation).copy()\n                compute.append(dict(**meta,variant=name,**cost))\n                trajectories.extend(dict(**meta,variant=name,**t) for t in trace)\n                pd.DataFrame(compute).to_csv(output_dir/\'compute.csv\',index=False)\n                print(f\'  {name}: {cost["elapsed_seconds"]:.1f}s, {cost["denoiser_calls"]} denoiser calls\',flush=True)\n            estimates.update(dpir_nominal=learned[\'nominal\'],dpir_oracle_blur=learned[\'oracle_blur\'],drunet_denoise_only=learned[\'denoise_only\'])\n            for name,x in estimates.items():\n                mse=float(np.mean((inside(x)-inside(truth))**2))\n                dmse=float(np.mean(inside(detail(x,config)-detail(truth,config))**2))\n                quality.append(dict(**meta,model=name,mse=mse,psnr_db=B01.psnr(mse),detail_mse=dmse))\n            curve,patches=evaluate_scores(truth,learned[\'nominal\'],y,\n                [learned[\'operator_0.8\'],learned[\'nominal\'],learned[\'operator_1.2\']],\n                [learned[\'nominal\'],learned[\'rotation_90\'],learned[\'rotation_180\']],config)\n            for k,v in meta.items():curve[k]=v\n            curves.append(curve)\n            np.savez_compressed(output_dir/\'predictions\'/f\'{sid}_{meta["scenario"]}.npz\',\n                nominal_reconstruction=learned[\'nominal\'].astype(np.float32),**patches)\n            if meta[\'scenario\']==config[\'scenarios\'][0]:\n                examples.append(dict(source_id=sid,reference=inside(truth),**{k:inside(v) for k,v in estimates.items()}))\n            result.update(quality=pd.DataFrame(quality),compute=pd.DataFrame(compute),curves=pd.concat(curves,ignore_index=True),trajectories=pd.DataFrame(trajectories))\n            result[\'summary\']=result[\'quality\'].groupby([\'scenario\',\'model\'],as_index=False).agg(mean_mse=(\'mse\',\'mean\'),mean_detail_mse=(\'detail_mse\',\'mean\'),sources=(\'source_id\',\'nunique\'),observations=(\'mse\',\'size\'))\n            result[\'summary\'][\'pooled_psnr_db\']=result[\'summary\'].mean_mse.map(B01.psnr)\n            result[\'completed_observations\']+=1\n            save_snapshot(result,output_dir,\'running\')\n    n=len(sources)*len(config[\'scenarios\'])\n    assert len(result[\'quality\'])==5*n and len(result[\'curves\'])==56*n\n    assert result[\'compute\'].denoiser_calls.sum()==49*n\n    assert all(digest(Path(data_dir)/r.filename)==r.sha256 for r in manifest.itertuples())\n    checks.update(source_count=len(sources),observation_count=n,quality_rows=len(result[\'quality\']),\n        risk_rows=len(result[\'curves\']),denoiser_calls=int(result[\'compute\'].denoiser_calls.sum()),input_files_unchanged=True)\n    result[\'figures\']=plot_results(result[\'quality\'],result[\'curves\'],result[\'trajectories\'],examples,output_dir)\n    for figure in result[\'figures\']:plt.close(figure)\n    save_snapshot(result,output_dir,\'complete_for_configured_subset\')\n    return result\n', 'diagnostic': '"""Paired acquisition-stage diagnosis with the unchanged Learned 02 adapter."""\nimport hashlib\nimport io\nimport json\nimport os\nimport platform\nimport time\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport PIL\nfrom PIL import Image, features\nimport torch\n\nB01 = None\nL02 = None\nSTAGES = [\'linear_float\', \'clipped_float\', \'quantized_8bit\', \'jpeg_q75\']\nSTAGE_LABELS = [\'Linear float\', \'+ Clipping\', \'+ 8-bit rounding\', \'+ JPEG Q75\']\nMODELS = [\'observed\', \'gradient_nominal\', \'drunet_denoise_only\', \'dpir_nominal\', \'dpir_oracle_blur\']\nMODEL_LABELS = {\'observed\': \'Input (display-clipped)\', \'gradient_nominal\': \'Classical gradient\',\n                \'drunet_denoise_only\': \'DRUNet denoise only\', \'dpir_nominal\': \'Nominal DPIR\',\n                \'dpir_oracle_blur\': \'True-blur DPIR diagnostic\'}\nCOLORS = dict(zip(MODELS, [\'#667085\', \'#B78103\', \'#768A46\', \'#2463A6\', \'#AE547A\']))\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef array_hash(array):\n    # Shape and dtype are recorded separately in acquisition.csv.\n    return hashlib.sha256(np.ascontiguousarray(array).tobytes()).hexdigest()\n\n\ndef acquisition_stages(truth, sid, config):\n    identity = int(hashlib.sha256(sid.encode()).hexdigest()[:8], 16)\n    rng = np.random.default_rng(np.random.SeedSequence([config[\'seed\'], identity]))\n    linear = B01.apply(truth, config[\'true_sigmas\'][0]) + config[\'noise_stds\'][0] * rng.standard_normal(truth.shape)\n    clipped = np.clip(linear, 0, 1)\n    integers = np.rint(clipped * 255).astype(np.uint8)\n    quantized = integers.astype(np.float64) / 255\n    stream = io.BytesIO()\n    Image.fromarray(integers).save(stream, format=\'JPEG\', quality=config[\'jpeg_quality\'],\n                                   subsampling=config[\'jpeg_subsampling\'], optimize=False)\n    encoded = stream.getvalue()\n    with Image.open(io.BytesIO(encoded)) as image:\n        jpeg = np.asarray(image.convert(\'RGB\'), dtype=np.float64) / 255\n    stages = dict(zip(STAGES, [linear, clipped, quantized, jpeg]))\n    inherited = {meta[\'scenario\']: y for meta, y in B01.observations(truth, sid, config)}\n    assert np.array_equal(linear, inherited[\'blur_noise\']), \'Linear simulator endpoint changed\'\n    assert np.array_equal(jpeg, inherited[\'blur_noise_jpeg\']), \'JPEG simulator endpoint changed\'\n    assert np.array_equal(clipped, np.clip(clipped, 0, 1))\n    assert np.max(np.abs(quantized - clipped)) <= .5 / 255 + 1e-15\n    rows = []\n    previous = linear\n    for index, (stage, observation) in enumerate(stages.items()):\n        assert observation.shape == truth.shape and observation.dtype == np.float64\n        assert np.isfinite(observation).all()\n        delta = observation - previous\n        rows.append(dict(source_id=sid, stage=stage, stage_index=index,\n            observation_sha256=array_hash(observation), dtype=str(observation.dtype),\n            height=observation.shape[0], width=observation.shape[1], channels=3,\n            minimum=float(observation.min()), maximum=float(observation.max()),\n            changed_component_fraction_from_previous=float(np.mean(delta != 0)),\n            measurement_delta_mse=float(np.mean(B01.interior(delta, config)**2)),\n            linear_out_of_range_fraction=float(np.mean((linear < 0) | (linear > 1))),\n            jpeg_bytes=len(encoded) if stage == \'jpeg_q75\' else 0,\n            jpeg_sha256=hashlib.sha256(encoded).hexdigest() if stage == \'jpeg_q75\' else \'\'))\n        previous = observation\n    return stages, rows, encoded\n\n\ndef errors(truth, estimate, config):\n    inside = lambda x: B01.interior(x, config)\n    rgb = np.mean((inside(estimate) - inside(truth))**2, axis=2)\n    # Identical metric definition; independently checked on archive readback.\n    detail_difference = inside(L02.detail(estimate, config) - L02.detail(truth, config))\n    detail = np.mean(detail_difference**2, axis=2)\n    rgb_patch = B01.patch_mean(rgb, config[\'patch_size\']).ravel()\n    detail_patch = B01.patch_mean(detail, config[\'patch_size\']).ravel()\n    row = dict(mse=float(rgb.mean()), psnr_db=B01.psnr(rgb.mean()), detail_mse=float(detail.mean()))\n    for threshold in config[\'detail_rmse_tolerances\']:\n        row[f\'bad_detail_rate_{threshold:g}\'] = float(np.mean(np.sqrt(detail_patch) > threshold))\n    return row, rgb_patch, detail_patch\n\n\ndef summaries(quality):\n    summary = quality.groupby([\'stage_index\', \'stage\', \'model\'], as_index=False).agg(\n        mean_mse=(\'mse\', \'mean\'), mean_detail_mse=(\'detail_mse\', \'mean\'), sources=(\'source_id\', \'nunique\'))\n    summary[\'pooled_psnr_db\'] = summary.mean_mse.map(B01.psnr)\n    deltas = []\n    for (sid, model), frame in quality.groupby([\'source_id\', \'model\']):\n        frame = frame.sort_values(\'stage_index\')\n        for before, after in zip(frame.to_dict(\'records\'), frame.to_dict(\'records\')[1:]):\n            deltas.append(dict(source_id=sid, model=model, from_stage=before[\'stage\'], to_stage=after[\'stage\'],\n                to_stage_index=after[\'stage_index\'], delta_mse=after[\'mse\']-before[\'mse\'],\n                delta_psnr_db=after[\'psnr_db\']-before[\'psnr_db\'],\n                delta_detail_mse=after[\'detail_mse\']-before[\'detail_mse\']))\n    gaps = []\n    for (sid, index, stage), frame in quality.groupby([\'source_id\', \'stage_index\', \'stage\']):\n        frame = frame.set_index(\'model\')\n        gaps.append(dict(source_id=sid, stage_index=index, stage=stage,\n            nominal_minus_classical_detail_mse=float(frame.loc[\'dpir_nominal\',\'detail_mse\']-frame.loc[\'gradient_nominal\',\'detail_mse\']),\n            nominal_minus_classical_psnr_db=float(frame.loc[\'dpir_nominal\',\'psnr_db\']-frame.loc[\'gradient_nominal\',\'psnr_db\'])))\n    gaps = pd.DataFrame(gaps).sort_values([\'source_id\', \'stage_index\'])\n    gaps[\'change_in_detail_gap\'] = gaps.groupby(\'source_id\').nominal_minus_classical_detail_mse.diff()\n    return summary, pd.DataFrame(deltas), gaps\n\n\ndef snapshot(result, output_dir, state, error=None):\n    output_dir = Path(output_dir)\n    for name in [\'source_manifest\', \'quality\', \'summary\', \'stage_deltas\', \'model_gaps\', \'acquisition\', \'compute\', \'trajectories\', \'endpoint_comparison\']:\n        if name in result:\n            result[name].to_csv(output_dir / (name + \'.csv\'), index=False)\n    for name in [\'config\', \'checks\', \'provenance\', \'environment\']:\n        (output_dir / (name + \'.json\')).write_text(json.dumps(result[name], indent=2) + \'\\n\')\n    status = dict(status=state, completed_observations=result[\'completed_observations\'],\n        planned_observations=4*len(result[\'config\'][\'source_ids\']), full_design_observations=16,\n        full_design_complete=result[\'completed_observations\']==16,\n        updated_utc=datetime.now(timezone.utc).isoformat())\n    if error is not None:\n        status[\'error\'] = str(error)\n    (output_dir/\'status.json\').write_text(json.dumps(status, indent=2)+\'\\n\')\n    files = [dict(name=str(p.relative_to(output_dir)), bytes=p.stat().st_size, sha256=sha256(p))\n             for p in sorted(output_dir.rglob(\'*\')) if p.is_file() and p.name != \'export_manifest.json\']\n    manifest = dict(experiment=\'acquisition_03\', role=\'development_only\', snapshot_status=state, files=files)\n    (output_dir/\'export_manifest.json\').write_text(json.dumps(manifest, indent=2)+\'\\n\')\n\n\ndef verify_saved_predictions(result, output_dir, sources):\n    max_error = 0.\n    expected_rows = len(result[\'quality\'])\n    count = 0\n    for sid, truth in sources.items():\n        for stage in STAGES:\n            with np.load(Path(output_dir)/\'predictions\'/f\'{sid}_{stage}.npz\', allow_pickle=False) as archive:\n                assert set(archive.files) == {\'supplied_observation\'} | set(MODELS) | {m+s for m in MODELS for s in [\'__rgb_patch_error\',\'__detail_patch_error\']}\n                assert array_hash(archive[\'supplied_observation\']) == result[\'acquisition\'].query(\'source_id == @sid and stage == @stage\').iloc[0].observation_sha256\n                for model in MODELS:\n                    estimate = archive[model].astype(np.float64)\n                    assert estimate.shape==truth.shape and np.isfinite(estimate).all()\n                    assert estimate.min()>=0 and estimate.max()<=1\n                    measured, rp, dp = errors(truth, estimate, result[\'config\'])\n                    row = result[\'quality\'].query(\'source_id == @sid and stage == @stage and model == @model\').iloc[0]\n                    for key,value in measured.items():\n                        difference=abs(value-row[key]);max_error=max(max_error,float(difference))\n                        assert difference<1e-12, (sid,stage,model,key,difference)\n                    assert np.allclose(rp, archive[model+\'__rgb_patch_error\'], atol=1e-15, rtol=0)\n                    assert np.allclose(dp, archive[model+\'__detail_patch_error\'], atol=1e-15, rtol=0)\n                    count+=1\n    assert count==expected_rows\n    return dict(saved_prediction_quality_rows_checked=count, saved_prediction_max_abs_metric_difference=max_error)\n\n\ndef plot_results(result, output_dir, sources):\n    plt.rcParams.update({\'font.size\':10, \'axes.spines.top\':False, \'axes.spines.right\':False,\n        \'axes.grid\':True, \'grid.alpha\':.2, \'figure.facecolor\':\'white\', \'savefig.facecolor\':\'white\'})\n    figures=[];scope=f"{len(sources)} development source(s); same noise across stages"\n    fig, axes=plt.subplots(1,2,figsize=(12,4.8))\n    for model in MODELS:\n        rows=result[\'summary\'].query(\'model == @model\').sort_values(\'stage_index\')\n        for ax,column in zip(axes,[\'pooled_psnr_db\',\'mean_detail_mse\']):\n            ax.plot(rows.stage_index, rows[column], marker=\'o\', color=COLORS[model], label=MODEL_LABELS[model],\n                    linestyle=\'--\' if model==\'dpir_oracle_blur\' else \'-\')\n    axes[0].set_ylabel(\'Pooled RGB PSNR (dB; higher is better)\')\n    axes[1].set_ylabel(\'Mean detail MSE (lower is better)\');axes[1].ticklabel_format(axis=\'y\',style=\'sci\',scilimits=(0,0))\n    for ax in axes:ax.set_xticks(range(4),STAGE_LABELS,rotation=12);ax.set_xlabel(\'Ordered acquisition stage\')\n    fig.suptitle(\'Reconstruction quality across acquisition stages\\n\'+scope)\n    handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,loc=\'lower center\',ncol=3,frameon=False)\n    fig.tight_layout(rect=(0,.13,1,.91));fig.savefig(Path(output_dir)/\'stage_quality.png\',dpi=130);figures.append(fig)\n    fig, axes=plt.subplots(1,2,figsize=(12,4.5))\n    source_colors=[\'#2463A6\',\'#B78103\',\'#768A46\',\'#AE547A\']\n    for i,(sid,frame) in enumerate(result[\'model_gaps\'].groupby(\'source_id\')):\n        for ax,column in zip(axes,[\'nominal_minus_classical_detail_mse\',\'nominal_minus_classical_psnr_db\']):\n            ax.plot(frame.stage_index,frame[column],marker=[\'o\',\'s\',\'^\',\'D\'][i],color=source_colors[i],label=sid)\n    axes[0].set_ylabel(\'Nominal − classical detail MSE\\nPositive: learned is worse\');axes[0].ticklabel_format(axis=\'y\',style=\'sci\',scilimits=(0,0))\n    axes[1].set_ylabel(\'Nominal − classical RGB PSNR (dB)\\nPositive: learned is better\')\n    for ax in axes:ax.axhline(0,color=\'#333333\',linewidth=1);ax.set_xticks(range(4),STAGE_LABELS,rotation=12);ax.legend(title=\'Source\',frameon=False)\n    fig.suptitle(\'Paired source-level reconstruction gaps\\n\'+scope);fig.tight_layout(rect=(0,0,1,.9));fig.savefig(Path(output_dir)/\'source_gaps.png\',dpi=130);figures.append(fig)\n    fig,ax=plt.subplots(figsize=(10,4.8))\n    delta=result[\'stage_deltas\'].groupby([\'to_stage_index\',\'model\']).delta_detail_mse.mean()\n    for i,model in enumerate(MODELS):\n        ax.bar(np.arange(3)+(i-2)*.15,[delta.loc[(j,model)] for j in [1,2,3]],width=.145,color=COLORS[model],label=MODEL_LABELS[model])\n    ax.axhline(0,color=\'#333333\',linewidth=1);ax.set_xticks(range(3),[\'Add clipping\',\'Add 8-bit rounding\',\'Add JPEG codec\'])\n    ax.set_ylabel(\'Change in mean detail MSE\\nPositive: added stage worsens error\');ax.ticklabel_format(axis=\'y\',style=\'sci\',scilimits=(0,0))\n    ax.legend(loc=\'upper left\',bbox_to_anchor=(1.01,1),frameon=False);fig.suptitle(\'Conditional adjacent-stage changes\\n\'+scope)\n    fig.tight_layout(rect=(0,0,1,.9));fig.savefig(Path(output_dir)/\'stage_changes.png\',dpi=130);figures.append(fig)\n    # Display the first configured source with a fixed centre inset; never select by outcome.\n    sid=next(iter(sources));truth=sources[sid];inside=lambda x:B01.interior(x,result[\'config\'])[128:384,128:384]\n    fig,axes=plt.subplots(4,5,figsize=(12,10))\n    shown=[\'reference\',\'observed\',\'gradient_nominal\',\'dpir_nominal\',\'dpir_oracle_blur\']\n    titles=[\'Reference\',\'Input\',\'Classical\',\'Nominal DPIR\',\'True-blur diagnostic\']\n    for i,stage in enumerate(STAGES):\n        with np.load(Path(output_dir)/\'predictions\'/f\'{sid}_{stage}.npz\',allow_pickle=False) as archive:\n            for j,key in enumerate(shown):\n                axes[i,j].imshow(inside(truth if key==\'reference\' else archive[key]));axes[i,j].set_xticks([]);axes[i,j].set_yticks([]);axes[i,j].grid(False)\n                if i==0:axes[i,j].set_title(titles[j])\n                if j==0:axes[i,j].set_ylabel(STAGE_LABELS[i])\n    fig.suptitle(f\'Source {sid}: fixed central 256 × 256 inset\\nAll methods are evaluated on the full 512 × 512 region\')\n    fig.tight_layout(rect=(0,0,1,.93));fig.savefig(Path(output_dir)/\'stage_examples.png\',dpi=125);figures.append(fig)\n    for fig in figures:plt.close(fig)\n    return figures\n\n\ndef run(data_dir, output_dir, vendor_dir, weights, provenance, expected_hashes, anchor_csv, config):\n    config=json.loads(json.dumps(config));output_dir=Path(output_dir)\n    output_dir.mkdir(parents=True,exist_ok=False);(output_dir/\'predictions\').mkdir();(output_dir/\'codec_inputs\').mkdir()\n    anchors=pd.read_csv(io.StringIO(anchor_csv),dtype={\'source_id\':str})\n    result=dict(config=config,provenance=provenance,checks={},completed_observations=0,\n        environment=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,\n            torch=torch.__version__,pillow=PIL.__version__,matplotlib=matplotlib.__version__,\n            jpeg_codec=features.version_codec(\'jpg\'),libjpeg_turbo=features.version_feature(\'libjpeg_turbo\'),\n            device=\'cuda\' if torch.cuda.is_available() else \'cpu\',\n            cuda_name=torch.cuda.get_device_name() if torch.cuda.is_available() else None))\n    snapshot(result,output_dir,\'initializing\')\n    try:\n        full_config=json.loads(json.dumps(config));full_config[\'source_ids\']=[\'0801\',\'0802\',\'0803\',\'0804\']\n        all_sources,full_manifest=B01.load_sources(data_dir,expected_hashes,full_config)\n        sources={sid:all_sources[sid] for sid in config[\'source_ids\']}\n        result[\'source_manifest\']=full_manifest\n        # Validate endpoints and cheap controls on all four sources before any new inference.\n        preflight=[]\n        for sid,truth in all_sources.items():\n            stages,_,_=acquisition_stages(truth,sid,config)\n            for stage,scenario in [(\'linear_float\',\'blur_noise\'),(\'jpeg_q75\',\'blur_noise_jpeg\')]:\n                y=stages[stage]\n                controls={\'observed\':np.clip(y,0,1),\'gradient_nominal\':B01.inverse_spectrum(np.fft.fft2(y,axes=(0,1)),config[\'nominal_sigma\'],\'gradient\',config[\'gradient_lambda\'])}\n                for name,x in controls.items():\n                    measured,_,_=errors(truth,x,config)\n                    old=anchors.query(\'source_id == @sid and scenario == @scenario and model == @name\').iloc[0]\n                    difference=max(abs(measured[\'mse\']-old.mse),abs(measured[\'detail_mse\']-old.detail_mse))\n                    assert difference<1e-12, \'Learned 02 input/classical endpoint mismatch\'\n                    preflight.append(difference)\n        result[\'checks\'].update(source_identities_verified=4,simulator_endpoints_verified_sources=4,\n            input_classical_anchor_rows_checked=len(preflight),input_classical_anchor_max_abs_mse_difference=max(preflight),\n            clipping_idempotence=True,quantization_half_step_bound=True)\n        L02.download_weights(weights,provenance)\n        model,device=L02.load_model(vendor_dir,weights,provenance,config)\n        result[\'checks\'].update(L02.validate_adapter(model,device,vendor_dir,config))\n        result[\'environment\'].update(cpu_threads=torch.get_num_threads(),deterministic_algorithms=torch.are_deterministic_algorithms_enabled())\n        quality=[];costs=[];traces=[];acquisition=[];comparisons=[]\n        snapshot(result,output_dir,\'running\')\n        for sid,truth in sources.items():\n            stages,measurements,encoded=acquisition_stages(truth,sid,config)\n            acquisition.extend(measurements);result[\'acquisition\']=pd.DataFrame(acquisition)\n            (output_dir/\'codec_inputs\'/f\'{sid}_q75.jpg\').write_bytes(encoded)\n            for index,(stage,y) in enumerate(stages.items()):\n                print(f\'{sid} | {stage} | observation {result["completed_observations"]+1}/{4*len(sources)}\',flush=True)\n                meta=dict(source_id=sid,stage=stage,stage_index=index)\n                estimates={\'observed\':np.clip(y,0,1)};tic=time.perf_counter()\n                estimates[\'gradient_nominal\']=B01.inverse_spectrum(np.fft.fft2(y,axes=(0,1)),config[\'nominal_sigma\'],\'gradient\',config[\'gradient_lambda\'])\n                costs.append(dict(**meta,model=\'gradient_nominal\',elapsed_seconds=time.perf_counter()-tic,denoiser_calls=0))\n                for name,sigma,only in [(\'dpir_nominal\',config[\'nominal_sigma\'],False),(\'dpir_oracle_blur\',config[\'true_sigmas\'][0],False),(\'drunet_denoise_only\',config[\'nominal_sigma\'],True)]:\n                    x,cost,trace=L02.reconstruct(y,sigma,model,device,config,denoise_only=only)\n                    estimates[name]=x;costs.append(dict(**meta,model=name,**cost));traces.extend(dict(**meta,model=name,**t) for t in trace)\n                    print(f\'  {MODEL_LABELS[name]}: {cost["elapsed_seconds"]:.2f}s\',flush=True)\n                arrays={\'supplied_observation\':y}\n                for name in MODELS:\n                    estimate=estimates[name];measured,rp,dp=errors(truth,estimate,config)\n                    quality.append(dict(**meta,model=name,**measured))\n                    arrays[name]=estimate.astype(np.float32) if name.startswith((\'dpir\',\'drunet\')) else estimate\n                    arrays[name+\'__rgb_patch_error\']=rp;arrays[name+\'__detail_patch_error\']=dp\n                    if stage in [\'linear_float\',\'jpeg_q75\']:\n                        scenario=\'blur_noise\' if stage==\'linear_float\' else \'blur_noise_jpeg\'\n                        old=anchors.query(\'source_id == @sid and scenario == @scenario and model == @name\').iloc[0]\n                        comparisons.append(dict(**meta,model=name,anchor_scenario=scenario,\n                            delta_mse_from_colab=measured[\'mse\']-old.mse,delta_detail_mse_from_colab=measured[\'detail_mse\']-old.detail_mse,\n                            delta_psnr_db_from_colab=measured[\'psnr_db\']-old.psnr_db))\n                np.savez_compressed(output_dir/\'predictions\'/f\'{sid}_{stage}.npz\',**arrays)\n                result.update(quality=pd.DataFrame(quality),compute=pd.DataFrame(costs),trajectories=pd.DataFrame(traces),endpoint_comparison=pd.DataFrame(comparisons))\n                result[\'completed_observations\']+=1\n                snapshot(result,output_dir,\'running\')\n        result[\'summary\'],result[\'stage_deltas\'],result[\'model_gaps\']=summaries(result[\'quality\'])\n        n=4*len(sources)\n        assert len(result[\'quality\'])==5*n and len(result[\'compute\'])==4*n and len(result[\'trajectories\'])==16*n\n        assert not result[\'quality\'].duplicated([\'source_id\',\'stage\',\'model\']).any()\n        assert result[\'compute\'].denoiser_calls.sum()==17*n\n        for (sid,name),frame in result[\'stage_deltas\'].groupby([\'source_id\',\'model\']):\n            endpoint=result[\'quality\'].query(\'source_id == @sid and model == @name\').sort_values(\'stage_index\')\n            assert abs(frame.delta_detail_mse.sum()-(endpoint.iloc[-1].detail_mse-endpoint.iloc[0].detail_mse))<1e-15\n        result[\'checks\'].update(quality_rows=len(result[\'quality\']),stage_delta_rows=len(result[\'stage_deltas\']),\n            observation_count=n,denoiser_calls=int(result[\'compute\'].denoiser_calls.sum()),paired_deltas_telescope=True)\n        result[\'checks\'].update(verify_saved_predictions(result,output_dir,sources))\n        assert all(sha256(Path(data_dir)/r.filename)==r.sha256 for r in full_manifest.itertuples())\n        result[\'checks\'][\'source_files_unchanged\']=True\n        result[\'figures\']=plot_results(result,output_dir,sources)\n        snapshot(result,output_dir,\'complete_for_configured_subset\')\n        return result\n    except Exception as error:\n        snapshot(result,output_dir,\'failed\',error)\n        raise\n\n\ndef verify_and_zip(output_dir):\n    output_dir=Path(output_dir)\n    manifest=json.loads((output_dir/\'export_manifest.json\').read_text())\n    status=json.loads((output_dir/\'status.json\').read_text())\n    assert status[\'status\']==\'complete_for_configured_subset\', \'Only completed configured runs are packaged\'\n    expected={r[\'name\'] for r in manifest[\'files\']}\n    actual={str(p.relative_to(output_dir)) for p in output_dir.rglob(\'*\') if p.is_file() and p.name!=\'export_manifest.json\'}\n    assert expected==actual and len(expected)==len(manifest[\'files\'])\n    for row in manifest[\'files\']:\n        path=output_dir/row[\'name\'];assert path.stat().st_size==row[\'bytes\'] and sha256(path)==row[\'sha256\']\n    destination=output_dir.with_suffix(\'.zip\')\n    assert not destination.exists(), f\'Existing ZIP preserved: {destination}\'\n    with zipfile.ZipFile(destination,\'x\',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n        for path in sorted(output_dir.rglob(\'*\')):\n            if path.is_file():z.write(path,arcname=str(Path(output_dir.name)/path.relative_to(output_dir)))\n    with zipfile.ZipFile(destination) as z:\n        assert z.testzip() is None\n        for row in manifest[\'files\']:\n            data=z.read(output_dir.name+\'/\'+row[\'name\'])\n            assert len(data)==row[\'bytes\'] and hashlib.sha256(data).hexdigest()==row[\'sha256\']\n    receipt=dict(zip_name=destination.name,zip_bytes=destination.stat().st_size,zip_sha256=sha256(destination),\n        verified_result_files=len(expected),configured_observations=status[\'completed_observations\'],\n        full_design_complete=status[\'full_design_complete\'])\n    destination.with_suffix(\'.receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\')\n    return destination,receipt\n', 'jpeg_aware': '"""Frozen FBCNN preprocessing comparison; no new-method or test-set claims."""\nimport ast\nimport hashlib\nimport io\nimport json\nimport platform\nimport time\nimport types\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport PIL\nfrom PIL import features\nimport torch\n\nB01 = L02 = D03 = None\nSTAGES = [\'quantized_8bit\', \'jpeg_q75\']\nMAIN_MODELS = [\'observed\', \'gradient_nominal\', \'dpir_nominal\', \'fbcnn\', \'fbcnn_gradient\', \'fbcnn_dpir\']\nPIPELINES = [\'dpir_nominal\', \'fbcnn_dpir\']\nVARIANTS = [(\'nominal\', 1.0, 0), (\'sigma08\', 0.8, 0), (\'sigma12\', 1.2, 0),\n            (\'rot90\', 1.0, 1), (\'rot180\', 1.0, 2)]\nLABELS = {\'observed\': \'Input\', \'gradient_nominal\': \'Classical\', \'dpir_nominal\': \'DPIR\',\n          \'fbcnn\': \'FBCNN only\', \'fbcnn_gradient\': \'FBCNN + classical\', \'fbcnn_dpir\': \'FBCNN + DPIR\'}\nCOLORS = dict(zip(MAIN_MODELS, [\'#667085\', \'#B78103\', \'#2463A6\', \'#AE547A\', \'#768A46\', \'#123F68\']))\nCONTROL_SCORES = [\'image_transform_spread_detail\', \'measurement_residual\', \'image_gradient\', \'random_expected\']\nFRAME_NAMES = [\'source_manifest\', \'acquisition\', \'preflight\', \'quality\', \'summary\', \'risk\',\n               \'risk_summary\', \'quality_changes\', \'selection_comparisons\', \'compute\', \'score_costs\',\n               \'trajectories\', \'fbcnn_diagnostics\', \'endpoint_comparison\']\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef dump(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n\n\ndef variant_name(pipeline, variant):\n    return pipeline if variant == \'nominal\' else pipeline + \'_\' + variant\n\n\ndef load_fbcnn(vendor_dir, checkpoint, provenance, device):\n    vendor_dir = Path(vendor_dir)\n    for rel, expected in provenance[\'files\'].items():\n        assert sha256(vendor_dir / rel) == expected, f\'FBCNN source mismatch: {rel}\'\n    L02.download_weights(checkpoint, provenance)\n    source = (vendor_dir / \'models/network_fbcnn.py\').read_text()\n    tree = ast.parse(source)\n    imports = [n for n in tree.body if isinstance(n, ast.Import) and\n               any(a.name == \'torchvision.models\' for a in n.names)]\n    assert len(imports) == 1 and len(imports[0].names) == 1\n    assert not any(isinstance(n, ast.Name) and n.id == \'models\' for n in ast.walk(tree)), \'Import is no longer unused\'\n    # Preserve original vendor bytes. Remove only an unused optional dependency.\n    tree.body.remove(imports[0])\n    module = types.ModuleType(\'fbcnn04_network\')\n    exec(compile(tree, str(vendor_dir / \'models/network_fbcnn.py\'), \'exec\'), module.__dict__)\n    model = module.FBCNN(in_nc=3, out_nc=3, nc=[64,128,256,512], nb=4, act_mode=\'R\')\n    state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n    assert isinstance(state, dict) and all(isinstance(v, torch.Tensor) for v in state.values())\n    model.load_state_dict(state, strict=True)\n    model.eval().requires_grad_(False)\n    model.to(device)\n    count = sum(p.numel() for p in model.parameters())\n    print(f\'Loaded verified FBCNN: {count:,} parameters; device={device}\', flush=True)\n    return model, count\n\n\ndef deblock(image, model, device):\n    assert image.ndim == 3 and image.shape[2] == 3 and np.isfinite(image).all()\n    assert image.min() >= 0 and image.max() <= 1\n    L02.sync(device)\n    if device.type == \'cuda\':\n        torch.cuda.reset_peak_memory_stats(device)\n    start = time.perf_counter()\n    with torch.inference_mode():\n        tensor = L02.to_tensor(image, device)\n        restored, q = model(tensor)  # Automatic quality; never supply Q75.\n        assert restored.shape == tensor.shape and bool(restored.isfinite().all())\n        assert q.shape == (1,1) and bool(q.isfinite().all()) and 0 <= float(q.item()) <= 1\n        clipped = float(((restored < 0) | (restored > 1)).float().mean().item())\n        output = restored.clamp(0,1).squeeze(0).permute(1,2,0).cpu().numpy().astype(np.float64)\n        degradation = float(q.item())\n    L02.sync(device)\n    return (output, dict(elapsed_seconds=time.perf_counter()-start, denoiser_calls=0, fbcnn_calls=1,\n        peak_cuda_memory_mb=float(torch.cuda.max_memory_allocated(device)/1024**2) if device.type==\'cuda\' else None),\n        dict(predicted_degradation=degradation, predicted_quality=100*(1-degradation), output_clipped_fraction=clipped))\n\n\ndef validate_fbcnn(model, device):\n    x = torch.linspace(0,1,3*31*37,device=device).reshape(1,3,31,37)\n    with torch.inference_mode():\n        a, qa = model(x)\n        b, qb = model(x)\n        forced, q_forced = model(x, qa)\n    assert a.shape == x.shape and torch.equal(a,b) and torch.equal(qa,qb)\n    assert bool(a.isfinite().all()) and 0 <= float(qa.item()) <= 1\n    assert torch.equal(qa,q_forced)\n    error = float((a-forced).abs().max().item())\n    assert error < 1e-7\n    return dict(fbcnn_strict_load=True, fbcnn_repeat_identical=True, fbcnn_odd_size_padding=True,\n                automatic_vs_own_predicted_quality_max_abs_error=error, fbcnn_small_validation_calls=3)\n\n\ndef snapshot(result, output_dir, state, error=None):\n    output_dir = Path(output_dir)\n    for name in FRAME_NAMES:\n        if name in result:\n            result[name].to_csv(output_dir / (name+\'.csv\'), index=False)\n    for name in [\'config\',\'provenance\',\'checks\',\'environment\',\'decisions\']:\n        if name in result:\n            dump(output_dir / (name+\'.json\'), result[name])\n    status = dict(status=state, configured_sources=result[\'config\'][\'source_ids\'],\n        completed_observations=result[\'completed_observations\'], planned_observations=2*len(result[\'config\'][\'source_ids\']),\n        full_design_observations=8, full_design_complete=result[\'completed_observations\']==8,\n        updated_utc=datetime.now(timezone.utc).isoformat())\n    if error is not None:\n        status[\'error\'] = str(error)\n    dump(output_dir/\'status.json\', status)\n    files = [dict(name=str(p.relative_to(output_dir)), bytes=p.stat().st_size, sha256=sha256(p))\n             for p in sorted(output_dir.rglob(\'*\')) if p.is_file() and p.name!=\'export_manifest.json\']\n    dump(output_dir/\'export_manifest.json\', dict(experiment=\'jpeg_aware_04\', role=\'development_only\',\n                                                snapshot_status=state, files=files))\n\n\ndef analyse(result):\n    q, r = result[\'quality\'], result[\'risk\']\n    summary = q[q.model.isin(MAIN_MODELS)].groupby([\'stage\',\'model\'],as_index=False).agg(\n        mean_mse=(\'mse\',\'mean\'), mean_detail_mse=(\'detail_mse\',\'mean\'), sources=(\'source_id\',\'nunique\'))\n    summary[\'pooled_psnr_db\'] = -10*np.log10(summary.mean_mse)\n    metrics = [\'rgb_mse\',\'detail_mse\']+[f\'bad_detail_rate_{t:g}\' for t in result[\'config\'][\'detail_rmse_tolerances\']]\n    risk_summary = r.groupby([\'stage\',\'pipeline\',\'region\',\'score\',\'coverage\'],as_index=False)[metrics].mean()\n    changes = []\n    pairs = [(\'fbcnn_dpir\',\'dpir_nominal\'),(\'fbcnn_dpir\',\'gradient_nominal\'),\n             (\'fbcnn_dpir\',\'fbcnn_gradient\'),(\'fbcnn_gradient\',\'gradient_nominal\'),(\'fbcnn\',\'observed\')]\n    for (sid,stage), frame in q.groupby([\'source_id\',\'stage\']):\n        f = frame.set_index(\'model\')\n        for a,b in pairs:\n            changes.append(dict(source_id=sid,stage=stage,method=a,comparator=b,\n                delta_detail_mse=float(f.loc[a,\'detail_mse\']-f.loc[b,\'detail_mse\']),\n                delta_mse=float(f.loc[a,\'mse\']-f.loc[b,\'mse\']),\n                delta_psnr_db=float(f.loc[a,\'psnr_db\']-f.loc[b,\'psnr_db\'])))\n    selections = []\n    for (sid,stage,pipeline,region,coverage), frame in r.groupby([\'source_id\',\'stage\',\'pipeline\',\'region\',\'coverage\']):\n        f = frame.set_index(\'score\'); op = f.loc[\'operator_spread_detail\']\n        for control in CONTROL_SCORES:\n            row = dict(source_id=sid,stage=stage,pipeline=pipeline,region=region,coverage=float(coverage),control=control)\n            row.update({\'delta_\'+m:float(op[m]-f.loc[control,m]) for m in metrics})\n            selections.append(row)\n    result.update(summary=summary, risk_summary=risk_summary, quality_changes=pd.DataFrame(changes),\n                  selection_comparisons=pd.DataFrame(selections))\n    full = sorted(result[\'config\'][\'source_ids\']) == [\'0801\',\'0802\',\'0803\',\'0804\'] and result[\'completed_observations\']==8\n    qc = result[\'quality_changes\'].query("stage==\'jpeg_q75\' and method==\'fbcnn_dpir\'")\n    cmp_means = qc.groupby(\'comparator\').delta_detail_mse.mean()\n    reconstruction_wins = int((qc.query("comparator==\'dpir_nominal\'").delta_detail_mse < 0).sum())\n    rs = r.query("stage==\'jpeg_q75\' and pipeline==\'fbcnn_dpir\' and region==\'all\' and coverage==0.5")\n    means = rs.groupby(\'score\')[metrics].mean()\n    best_control = means.loc[CONTROL_SCORES,\'detail_mse\'].idxmin()\n    per_source = rs.pivot(index=\'source_id\',columns=\'score\',values=\'detail_mse\')\n    selection_wins = int((per_source.operator_spread_detail < per_source[best_control]).sum())\n    op = means.loc[\'operator_spread_detail\']\n    decisions = dict(scope=\'descriptive_development_screen_only\', full_design_assessed=full,\n        status=\'assessed_development_only\' if full else \'not_assessed_full_design\',\n        jpeg_detail_mse_deltas_vs_inverse_controls={str(k):float(v) for k,v in cmp_means.items()},\n        reconstruction_source_wins_vs_raw_dpir=reconstruction_wins,\n        selection_best_pooled_operational_control=str(best_control), selection_source_wins=selection_wins,\n        reconstruction_screen_pass=None, selection_screen_pass=None,\n        independent_test=False, calibrated_reliability=False, novelty_established=False,\n        stronger_trained_image_only_uncertainty_comparator_run=False)\n    if full:\n        decisions[\'reconstruction_screen_pass\'] = bool((cmp_means < 0).all() and reconstruction_wins>=3)\n        tails = [m for m in metrics if m.startswith(\'bad_detail\')]\n        decisions[\'selection_screen_pass\'] = bool(\n            (op.detail_mse < means.loc[CONTROL_SCORES,\'detail_mse\']).all() and selection_wins>=3 and\n            all((op[m] <= means.loc[CONTROL_SCORES,m]+1e-12).all() for m in tails))\n    result[\'decisions\'] = decisions\n\n\ndef plot_results(result, output_dir, sources):\n    output_dir = Path(output_dir)\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':10,\'axes.spines.top\':False,\n        \'axes.spines.right\':False,\'axes.grid\':True,\'grid.alpha\':.18,\'figure.facecolor\':\'white\'})\n    stage_labels = {\'quantized_8bit\':\'8-bit, no JPEG\',\'jpeg_q75\':\'JPEG Q75\'}\n    scope = f"{len(sources)} development source(s); no independent test"\n    fig, axes = plt.subplots(2,2,figsize=(12,7.3),sharey=True)\n    for j,stage in enumerate(STAGES):\n        frame=result[\'summary\'].query(\'stage==@stage\').set_index(\'model\').loc[MAIN_MODELS]\n        for i,col in enumerate([\'pooled_psnr_db\',\'mean_detail_mse\']):\n            ax=axes[i,j]\n            for k,model in enumerate(MAIN_MODELS):\n                ax.scatter(frame.loc[model,col],k,s=55,c=COLORS[model],marker=\'D\' if model.startswith(\'fbcnn\') else \'o\')\n            ax.set_yticks(range(6),[LABELS[m] for m in MAIN_MODELS]);ax.set_ylim(5.6,-.6)\n            ax.set_title(stage_labels[stage]);ax.set_xlabel(\'Pooled RGB PSNR (dB; higher better)\' if i==0 else \'Mean detail MSE (lower better)\')\n            if i: ax.ticklabel_format(axis=\'x\',style=\'sci\',scilimits=(0,0))\n    fig.suptitle(\'Frozen reconstruction comparisons | \'+scope)\n    fig.tight_layout(rect=(0,0,1,.95));fig.savefig(output_dir/\'quality.png\',dpi=135);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11.5,4.4),sharey=True)\n    for ax,stage in zip(axes,STAGES):\n        frame=result[\'quality_changes\'].query(\'stage==@stage and method=="fbcnn_dpir"\')\n        for k,(cmp,color,marker) in enumerate([(\'dpir_nominal\',\'#2463A6\',\'o\'),(\'gradient_nominal\',\'#B78103\',\'s\'),(\'fbcnn_gradient\',\'#768A46\',\'D\')]):\n            f=frame.query(\'comparator==@cmp\').set_index(\'source_id\').reindex(list(sources))\n            ax.plot(np.arange(len(sources))+(k-1)*.10,f.delta_detail_mse,marker=marker,linestyle=\'none\',color=color,label=LABELS[cmp])\n        ax.axhline(0,color=\'#333333\',linewidth=1);ax.set_xticks(range(len(sources)),list(sources));ax.set_xlabel(\'Source\')\n        ax.set_title(stage_labels[stage]);ax.ticklabel_format(axis=\'y\',style=\'sci\',scilimits=(0,0))\n    axes[0].set_ylabel(\'FBCNN + DPIR minus comparator detail MSE\\nNegative favours FBCNN + DPIR\')\n    axes[1].legend(title=\'Comparator\',frameon=False)\n    fig.suptitle(\'Source-specific reconstruction changes | \'+scope)\n    fig.tight_layout(rect=(0,0,1,.94));fig.savefig(output_dir/\'source_changes.png\',dpi=135);plt.close(fig)\n    style={\'operator_spread_detail\':(\'#2463A6\',\'-\',\'o\',\'Operator detail\'),\n           \'image_transform_spread_detail\':(\'#AE547A\',\'--\',\'s\',\'Whole-pipeline rotations\'),\n           \'measurement_residual\':(\'#B78103\',\':\',\'^\',\'Original-measurement residual\'),\n           \'image_gradient\':(\'#768A46\',\'-.\',\'v\',\'Image gradient\'),\n           \'operator_spread_rgb\':(\'#123F68\',\'--\',\'D\',\'Operator RGB (secondary)\'),\n           \'random_expected\':(\'#667085\',\':\',\'x\',\'Random expectation\'),\n           \'oracle_detail_error\':(\'#222222\',\'-.\',\'+\',\'Reference-error ranking\')}\n    fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True)\n    for i,pipeline in enumerate(PIPELINES):\n        for j,stage in enumerate(STAGES):\n            ax=axes[i,j]\n            f=result[\'risk_summary\'].query(\'pipeline==@pipeline and stage==@stage and region=="all"\')\n            for score,(color,ls,marker,label) in style.items():\n                rows=f.query(\'score==@score\').sort_values(\'coverage\')\n                ax.plot(100*rows.coverage,rows.detail_mse,color=color,linestyle=ls,marker=marker,label=label)\n            ax.set_title(LABELS[pipeline]+\' | \'+stage_labels[stage]);ax.set_xlabel(\'Requested patch retention (%)\')\n            ax.set_ylabel(\'Retained detail MSE\');ax.set_xticks([50,75,90,100]);ax.ticklabel_format(axis=\'y\',style=\'sci\',scilimits=(0,0))\n    handles,labels=axes[0,0].get_legend_handles_labels()\n    fig.legend(handles,labels,loc=\'lower center\',ncol=3,frameon=False,fontsize=9)\n    fig.suptitle(\'Heuristic selection | All patches | \'+scope)\n    fig.tight_layout(rect=(0,.12,1,.95));fig.savefig(output_dir/\'risk_coverage.png\',dpi=135);plt.close(fig)\n    sid=next(iter(sources));truth=sources[sid]\n    shown=[\'reference\',\'observed\',\'gradient_nominal\',\'dpir_nominal\',\'fbcnn_gradient\',\'fbcnn_dpir\']\n    fig,axes=plt.subplots(2,6,figsize=(14,5.4))\n    for i,stage in enumerate(STAGES):\n        with np.load(output_dir/\'predictions\'/f\'{sid}_{stage}.npz\',allow_pickle=False) as saved:\n            for j,key in enumerate(shown):\n                value=truth if key==\'reference\' else saved[key]\n                axes[i,j].imshow(value[160:416,160:416]);axes[i,j].set_xticks([]);axes[i,j].set_yticks([]);axes[i,j].grid(False)\n                if i==0:axes[i,j].set_title(\'Reference\' if key==\'reference\' else LABELS[key],fontsize=10)\n                if j==0:axes[i,j].set_ylabel(stage_labels[stage])\n    fig.suptitle(f\'Fixed central 256 x 256 inset | Source {sid} only | Metrics use the full 512 x 512 interior\')\n    fig.tight_layout(rect=(0,0,1,.92));fig.savefig(output_dir/\'examples.png\',dpi=135);plt.close(fig)\n\n\ndef run(data_dir, output_dir, dpir_vendor, dpir_weights, fbcnn_vendor, fbcnn_weights,\n        provenance, expected_hashes, anchor_csv, anchor_acquisition_csv, config):\n    config=json.loads(json.dumps(config));output_dir=Path(output_dir)\n    assert config[\'stages\']==STAGES and config[\'operator_sigmas\']==[.8,1.,1.2] and config[\'outer_rotations\']==[0,1,2]\n    output_dir.mkdir(parents=True,exist_ok=False)\n    for d in [\'predictions\',\'codec_inputs\']:(output_dir/d).mkdir()\n    result=dict(config=config,provenance=provenance,checks={},completed_observations=0,\n        environment=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,\n            torch=str(torch.__version__),pillow=PIL.__version__,matplotlib=matplotlib.__version__,\n            jpeg_codec=features.version_codec(\'jpg\'),libjpeg_turbo=features.version_feature(\'libjpeg_turbo\'),\n            device=\'cuda\' if torch.cuda.is_available() else \'cpu\',\n            cuda_name=torch.cuda.get_device_name() if torch.cuda.is_available() else None))\n    snapshot(result,output_dir,\'initializing\')\n    try:\n        anchors=pd.read_csv(io.StringIO(anchor_csv),dtype={\'source_id\':str})\n        anchor_acq=pd.read_csv(io.StringIO(anchor_acquisition_csv),dtype={\'source_id\':str})\n        full_config={**config,\'source_ids\':[\'0801\',\'0802\',\'0803\',\'0804\']}\n        all_sources,manifest=B01.load_sources(data_dir,expected_hashes,full_config)\n        result[\'source_manifest\']=manifest\n        sources={sid:all_sources[sid] for sid in config[\'source_ids\']}\n        preflight=[]\n        for sid,truth in all_sources.items():\n            stages,measurements,_=D03.acquisition_stages(truth,sid,config)\n            for stage in STAGES:\n                ar=next(m for m in measurements if m[\'stage\']==stage)\n                old=anchor_acq.query(\'source_id==@sid and stage==@stage\').iloc[0]\n                assert ar[\'observation_sha256\']==old.observation_sha256,\'Acquisition 03 observation bytes changed\'\n                y=stages[stage]\n                controls={\'observed\':y,\'gradient_nominal\':B01.inverse_spectrum(np.fft.fft2(y,axes=(0,1)),1.,\'gradient\',config[\'gradient_lambda\'])}\n                for model,x in controls.items():\n                    row,_,_=D03.errors(truth,x,config)\n                    old=anchors.query(\'source_id==@sid and stage==@stage and model==@model\').iloc[0]\n                    error=max(abs(row[\'mse\']-old.mse),abs(row[\'detail_mse\']-old.detail_mse))\n                    assert error<1e-12,\'Input/classical anchor changed\'\n                    preflight.append(dict(source_id=sid,stage=stage,model=model,max_abs_mse_difference=error))\n        result[\'preflight\']=pd.DataFrame(preflight)\n        result[\'checks\'].update(source_identities_verified=4,observation_hashes_verified=8,\n            input_classical_anchor_rows=len(preflight),input_classical_anchor_max_abs_mse_difference=max(r[\'max_abs_mse_difference\'] for r in preflight))\n        L02.download_weights(dpir_weights,provenance[\'dpir\'])\n        dpir,device=L02.load_model(dpir_vendor,dpir_weights,provenance[\'dpir\'],config)\n        result[\'checks\'].update(L02.validate_adapter(dpir,device,dpir_vendor,config))\n        result[\'checks\'][\'drunet_small_validation_calls\']=2\n        fbcnn,parameters=load_fbcnn(fbcnn_vendor,fbcnn_weights,provenance[\'fbcnn\'],device)\n        result[\'checks\'].update(validate_fbcnn(fbcnn,device),fbcnn_parameters=parameters)\n        result[\'environment\'].update(cpu_threads=torch.get_num_threads(),deterministic_algorithms=torch.are_deterministic_algorithms_enabled())\n        quality=[];risk=[];costs=[];score_costs=[];traces=[];acquisitions=[];fdiag=[];endpoint=[]\n        snapshot(result,output_dir,\'running\')\n        for sid,truth in sources.items():\n            stages,measurements,encoded=D03.acquisition_stages(truth,sid,config)\n            acquisitions.extend(m for m in measurements if m[\'stage\'] in STAGES)\n            result[\'acquisition\']=pd.DataFrame(acquisitions)\n            (output_dir/\'codec_inputs\'/f\'{sid}_q75.jpg\').write_bytes(encoded)\n            for stage in STAGES:\n                meta=dict(source_id=sid,stage=stage);y=stages[stage]\n                print(f\'{sid} | {stage} | {result["completed_observations"]+1}/{2*len(sources)}\',flush=True)\n                estimates={\'observed\':y};aux={};local_cost={}\n                start=time.perf_counter()\n                estimates[\'gradient_nominal\']=B01.inverse_spectrum(np.fft.fft2(y,axes=(0,1)),1.,\'gradient\',config[\'gradient_lambda\'])\n                costs.append(dict(**meta,component=\'gradient_nominal\',elapsed_seconds=time.perf_counter()-start,denoiser_calls=0,fbcnn_calls=0))\n                for rotation in [0,1,2]:\n                    restored,cost,diagnostic=deblock(np.rot90(y,rotation).copy(),fbcnn,device)\n                    key=\'fbcnn\' if rotation==0 else f\'fbcnn_rot{rotation*90}_intermediate\'\n                    if rotation==0:estimates[\'fbcnn\']=restored\n                    else:aux[key]=restored.astype(np.float32)\n                    local_cost[key]=cost;costs.append(dict(**meta,component=key,**cost))\n                    fdiag.append(dict(**meta,rotation_degrees=rotation*90,**diagnostic))\n                    print(f\'  FBCNN rotation {rotation*90}: {cost["elapsed_seconds"]:.2f}s\',flush=True)\n                start=time.perf_counter()\n                estimates[\'fbcnn_gradient\']=B01.inverse_spectrum(np.fft.fft2(estimates[\'fbcnn\'],axes=(0,1)),1.,\'gradient\',config[\'gradient_lambda\'])\n                costs.append(dict(**meta,component=\'fbcnn_gradient_inverse_only\',elapsed_seconds=time.perf_counter()-start,denoiser_calls=0,fbcnn_calls=0))\n                for pipeline in PIPELINES:\n                    for variant,sigma,rotation in VARIANTS:\n                        key=variant_name(pipeline,variant)\n                        if pipeline==\'dpir_nominal\':inp=np.rot90(y,rotation).copy()\n                        else:inp=estimates[\'fbcnn\'] if rotation==0 else aux[f\'fbcnn_rot{rotation*90}_intermediate\'].astype(np.float64)\n                        restored,cost,trace=L02.reconstruct(inp,sigma,dpir,device,config)\n                        estimates[key]=np.rot90(restored,-rotation).copy()\n                        cost={**cost,\'fbcnn_calls\':0};local_cost[key]=cost\n                        costs.append(dict(**meta,component=key,**cost))\n                        traces.extend(dict(**meta,component=key,**t) for t in trace)\n                        print(f\'  {key}: {cost["elapsed_seconds"]:.2f}s\',flush=True)\n                arrays={\'supplied_observation\':y,**aux}\n                for name,x in estimates.items():\n                    measured,rp,de=D03.errors(truth,x,config)\n                    quality.append(dict(**meta,model=name,role=\'main\' if name in MAIN_MODELS else \'ensemble_component\',**measured))\n                    arrays[name]=x if name in [\'observed\',\'gradient_nominal\',\'fbcnn_gradient\'] else x.astype(np.float32)\n                    arrays[name+\'__rgb_patch_error\']=rp;arrays[name+\'__detail_patch_error\']=de\n                    if name in [\'observed\',\'gradient_nominal\',\'dpir_nominal\']:\n                        old=anchors.query(\'source_id==@sid and stage==@stage and model==@name\').iloc[0]\n                        endpoint.append(dict(**meta,model=name,delta_mse_from_acquisition03=measured[\'mse\']-old.mse,\n                            delta_detail_mse_from_acquisition03=measured[\'detail_mse\']-old.detail_mse,\n                            delta_psnr_from_acquisition03=measured[\'psnr_db\']-old.psnr_db))\n                for pipeline in PIPELINES:\n                    operators=[estimates[variant_name(pipeline,k)] for k in [\'sigma08\',\'nominal\',\'sigma12\']]\n                    transforms=[estimates[variant_name(pipeline,k)] for k in [\'nominal\',\'rot90\',\'rot180\']]\n                    curve,score_arrays=L02.evaluate_scores(truth,estimates[pipeline],y,operators,transforms,config)\n                    risk.extend(dict(**meta,pipeline=pipeline,**row) for row in curve.to_dict(\'records\'))\n                    arrays.update({pipeline+\'__score__\'+k:v for k,v in score_arrays.items()})\n                    for family,variants in [(\'operator\',[\'nominal\',\'sigma08\',\'sigma12\']),(\'transformation\',[\'nominal\',\'rot90\',\'rot180\'])]:\n                        components=[variant_name(pipeline,k) for k in variants]\n                        if pipeline==\'fbcnn_dpir\':\n                            components+=[\'fbcnn\'] if family==\'operator\' else [\'fbcnn\',\'fbcnn_rot90_intermediate\',\'fbcnn_rot180_intermediate\']\n                        score_costs.append(dict(**meta,pipeline=pipeline,score_family=family,\n                            elapsed_seconds=sum(local_cost[k][\'elapsed_seconds\'] for k in components),\n                            denoiser_calls=sum(local_cost[k][\'denoiser_calls\'] for k in components),\n                            fbcnn_calls=sum(local_cost[k][\'fbcnn_calls\'] for k in components),\n                            accounting=\'standalone score cost incl. shared nominal; do not sum overlapping score costs\'))\n                np.savez_compressed(output_dir/\'predictions\'/f\'{sid}_{stage}.npz\',**arrays)\n                result.update(quality=pd.DataFrame(quality),risk=pd.DataFrame(risk),compute=pd.DataFrame(costs),\n                    score_costs=pd.DataFrame(score_costs),trajectories=pd.DataFrame(traces),\n                    fbcnn_diagnostics=pd.DataFrame(fdiag),endpoint_comparison=pd.DataFrame(endpoint))\n                result[\'completed_observations\']+=1\n                snapshot(result,output_dir,\'running\')\n        n=2*len(sources)\n        assert len(result[\'quality\'])==14*n and len(result[\'risk\'])==112*n\n        assert result[\'compute\'].denoiser_calls.sum()==80*n and result[\'compute\'].fbcnn_calls.sum()==3*n\n        assert len(result[\'trajectories\'])==80*n and len(result[\'score_costs\'])==4*n\n        assert not result[\'quality\'].duplicated([\'source_id\',\'stage\',\'model\']).any()\n        assert not result[\'risk\'].duplicated([\'source_id\',\'stage\',\'pipeline\',\'region\',\'score\',\'coverage\']).any()\n        analyse(result)\n        result[\'checks\'].update(quality_rows=len(result[\'quality\']),main_quality_rows=6*n,risk_rows=len(result[\'risk\']),\n            drunet_experiment_calls=80*n,fbcnn_experiment_calls=3*n,\n            source_files_unchanged=all(sha256(Path(data_dir)/row.filename)==row.sha256 for row in manifest.itertuples()))\n        assert result[\'checks\'][\'source_files_unchanged\']\n        plot_results(result,output_dir,sources)\n        snapshot(result,output_dir,\'inference_complete_pending_independent_readback\')\n        return result\n    except Exception as error:\n        snapshot(result,output_dir,\'failed\',error)\n        raise\n', 'validator': '"""Independent saved-array readback for Notebook 04; no neural inference."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef validate(run_dir, data_dir):\n    run=Path(run_dir);data_dir=Path(data_dir)\n    cfg=json.loads((run/\'config.json\').read_text())\n    status=json.loads((run/\'status.json\').read_text())\n    assert status[\'status\'] in [\'inference_complete_pending_independent_readback\',\'complete_for_configured_subset\']\n    assert cfg[\'crop_size\']==512 and cfg[\'context_border\']==32 and cfg[\'patch_size\']==16 and cfg[\'detail_blur_sigma\']==1\n    manifest=json.loads((run/\'export_manifest.json\').read_text())\n    files={r[\'name\']:r for r in manifest[\'files\']}\n    assert len(files)==len(manifest[\'files\'])\n    assert set(files)=={str(p.relative_to(run)) for p in run.rglob(\'*\') if p.is_file() and p.name!=\'export_manifest.json\'}\n    for name,r in files.items():assert (run/name).stat().st_size==r[\'bytes\'] and sha(run/name)==r[\'sha256\']\n    read=lambda name:pd.read_csv(run/(name+\'.csv\'),dtype={\'source_id\':str})\n    source,quality,risk,acq=map(read,[\'source_manifest\',\'quality\',\'risk\',\'acquisition\'])\n    n=2*len(cfg[\'source_ids\']);assert len(quality)==14*n and len(risk)==112*n and len(acq)==n\n    assert set(acq.stage)=={\'quantized_8bit\',\'jpeg_q75\'} and set(acq.source_id)==set(cfg[\'source_ids\'])\n    assert not quality.duplicated([\'source_id\',\'stage\',\'model\']).any()\n    assert not risk.duplicated([\'source_id\',\'stage\',\'pipeline\',\'region\',\'score\',\'coverage\']).any()\n    assert status[\'completed_observations\']==n and status[\'full_design_complete\']==(n==8)\n    cut=lambda x:x[32:544,32:544]\n    pool=lambda x:x.reshape(32,16,32,16).mean(axis=(1,3)).ravel()\n    def smooth(x,sigma=1):\n        h,w=x.shape[:2]\n        htf=np.exp(-2*np.pi**2*sigma**2*(np.fft.fftfreq(h)[:,None]**2+np.fft.fftfreq(w)[None,:]**2))\n        return np.fft.ifft2(np.fft.fft2(x,axes=(0,1))*htf[:,:,None],axes=(0,1)).real\n    detail=lambda x:x-smooth(x)\n    maximum=patch_max=score_max=risk_max=0.;checked=patches=scored=risks=0\n    main=[\'observed\',\'gradient_nominal\',\'dpir_nominal\',\'fbcnn\',\'fbcnn_gradient\',\'fbcnn_dpir\']\n    pipeline_names=[\'dpir_nominal\',\'fbcnn_dpir\'];models=main.copy()\n    for pipe in pipeline_names:models.extend(pipe+\'_\'+v for v in [\'sigma08\',\'sigma12\',\'rot90\',\'rot180\'])\n    score_names=[\'operator_spread_detail\',\'image_transform_spread_detail\',\'operator_spread_rgb\',\'measurement_residual\',\'image_gradient\']\n    arrays_expected={\'supplied_observation\',\'fbcnn_rot90_intermediate\',\'fbcnn_rot180_intermediate\'} | set(models)\n    arrays_expected|={m+s for m in models for s in [\'__rgb_patch_error\',\'__detail_patch_error\']}\n    arrays_expected|={p+\'__score__\'+s for p in pipeline_names for s in score_names+[\'rgb_patch_error\',\'detail_patch_error\',\'reference_texture\']}\n    truths={}\n    for row in source.itertuples(index=False):\n        p=data_dir/row.filename\n        with Image.open(p) as img:rgb=np.asarray(img.convert(\'RGB\'))\n        assert sha(p)==row.sha256 and hashlib.sha256(rgb.tobytes()).hexdigest()==row.rgb_sha256\n        truth=rgb[row.crop_top:row.crop_top+row.extent,row.crop_left:row.crop_left+row.extent].astype(np.float64)/255\n        assert truth.shape==(576,576,3);truths[row.source_id]=truth\n    assert set(truths)=={\'0801\',\'0802\',\'0803\',\'0804\'}\n    for sid in cfg[\'source_ids\']:\n        truth=truths[sid];dt=detail(truth)\n        identity=int(hashlib.sha256(sid.encode()).hexdigest()[:8],16)\n        rng=np.random.default_rng(np.random.SeedSequence([cfg[\'seed\'],identity]))\n        linear=smooth(truth,1.6)+(2/255)*rng.standard_normal(truth.shape)\n        quant=np.rint(np.clip(linear,0,1)*255).astype(\'uint8\').astype(float)/255\n        for stage in [\'quantized_8bit\',\'jpeg_q75\']:\n            with np.load(run/\'predictions\'/f\'{sid}_{stage}.npz\',allow_pickle=False) as saved:\n                assert set(saved.files)==arrays_expected\n                y=saved[\'supplied_observation\'];ar=acq.query(\'source_id==@sid and stage==@stage\').iloc[0]\n                assert hashlib.sha256(np.ascontiguousarray(y).tobytes()).hexdigest()==ar.observation_sha256\n                if stage==\'quantized_8bit\':assert np.array_equal(y,quant)\n                else:\n                    jpeg=run/\'codec_inputs\'/f\'{sid}_q75.jpg\'\n                    assert jpeg.stat().st_size==ar.jpeg_bytes and sha(jpeg)==ar.jpeg_sha256\n                    with Image.open(jpeg) as img:assert np.array_equal(np.asarray(img.convert(\'RGB\')).astype(float)/255,y)\n                assert np.array_equal(saved[\'observed\'],y)\n                for aux in [\'fbcnn_rot90_intermediate\',\'fbcnn_rot180_intermediate\']:\n                    assert saved[aux].shape==(576,576,3) and np.isfinite(saved[aux]).all() and saved[aux].min()>=0 and saved[aux].max()<=1\n                for model in models:\n                    x=saved[model].astype(np.float64)\n                    assert x.shape==(576,576,3) and np.isfinite(x).all() and x.min()>=0 and x.max()<=1\n                    re=((cut(x)-cut(truth))**2).mean(axis=2)\n                    de=(cut(detail(x)-dt)**2).mean(axis=2)\n                    rp,dp=pool(re),pool(de)\n                    qr=quality.query(\'source_id==@sid and stage==@stage and model==@model\').iloc[0]\n                    actual={\'mse\':float(re.mean()),\'psnr_db\':float(-10*np.log10(re.mean())),\'detail_mse\':float(de.mean())}\n                    for t in [.025,.05,.1]:actual[f\'bad_detail_rate_{t:g}\']=float((np.sqrt(dp)>t).mean())\n                    for col,val in actual.items():\n                        e=abs(val-qr[col]);maximum=max(maximum,e);assert e<1e-12,(sid,stage,model,col,e)\n                    for domain,p in [(\'rgb\',rp),(\'detail\',dp)]:\n                        e=float(np.abs(p-saved[model+\'__\'+domain+\'_patch_error\']).max())\n                        patch_max=max(patch_max,e);assert e<1e-14;patches+=1\n                    checked+=1\n                for pipeline in pipeline_names:\n                    x=saved[pipeline].astype(np.float64)\n                    operator=[saved[pipeline+\'_sigma08\'].astype(float),x,saved[pipeline+\'_sigma12\'].astype(float)]\n                    transforms=[x,saved[pipeline+\'_rot90\'].astype(float),saved[pipeline+\'_rot180\'].astype(float)]\n                    def spread(xs):\n                        stack=np.stack(xs);return np.sqrt(((stack-stack.mean(axis=0))**2).mean(axis=0).mean(axis=-1))\n                    gy,gx=np.gradient(x.mean(axis=-1))\n                    raw={\'operator_spread_detail\':spread([detail(a) for a in operator]),\n                         \'image_transform_spread_detail\':spread([detail(a) for a in transforms]),\n                         \'operator_spread_rgb\':spread(operator),\n                         \'measurement_residual\':np.sqrt(((smooth(x)-y)**2).mean(axis=-1)),\n                         \'image_gradient\':np.hypot(gx,gy)}\n                    score_vectors={k:pool(cut(v)) for k,v in raw.items()}\n                    gy,gx=np.gradient(cut(truth).mean(axis=-1));texture=pool(np.hypot(gx,gy))\n                    score_vectors[\'reference_texture\']=texture\n                    rp=saved[pipeline+\'__rgb_patch_error\'];dp=saved[pipeline+\'__detail_patch_error\']\n                    score_vectors.update(rgb_patch_error=rp,detail_patch_error=dp)\n                    for name,value in score_vectors.items():\n                        vec=saved[pipeline+\'__score__\'+name];assert vec.shape==(1024,) and np.isfinite(vec).all()\n                        e=float(np.abs(value-vec).max());score_max=max(score_max,e);assert e<1e-12;scored+=1\n                    groups={\'all\':np.arange(1024),\'textured_quartile\':np.argsort(texture,kind=\'stable\')[-256:]}\n                    for region,ids in groups.items():\n                        for score in score_names+[\'random_expected\',\'oracle_detail_error\']:\n                            scores=None if score==\'random_expected\' else dp if score==\'oracle_detail_error\' else saved[pipeline+\'__score__\'+score]\n                            order=None if scores is None else np.lexsort((np.random.default_rng(82).random(len(ids)),scores[ids]))\n                            for coverage in [.5,.75,.9,1.]:\n                                k=int(np.ceil(coverage*len(ids)))\n                                chosen=ids if order is None else ids[order[:k]]\n                                rows=risk.query(\'source_id==@sid and stage==@stage and pipeline==@pipeline and region==@region and score==@score and coverage==@coverage\')\n                                assert len(rows)==1;rr=rows.iloc[0]\n                                assert rr.available_patches==len(ids) and rr.retained_patches==k\n                                vals={\'rgb_mse\':rp[chosen].mean(),\'detail_mse\':dp[chosen].mean()}\n                                for t in [.025,.05,.1]:vals[f\'bad_detail_rate_{t:g}\']=(np.sqrt(dp[chosen])>t).mean()\n                                for col,val in vals.items():\n                                    e=abs(float(val)-rr[col]);risk_max=max(risk_max,e);assert e<1e-12\n                                risks+=1\n    summary=read(\'summary\')\n    for row in summary.itertuples(index=False):\n        q=quality[(quality.stage==row.stage)&(quality.model==row.model)]\n        assert abs(q.mse.mean()-row.mean_mse)<1e-12 and abs(q.detail_mse.mean()-row.mean_detail_mse)<1e-12\n        assert abs(-10*np.log10(q.mse.mean())-row.pooled_psnr_db)<1e-12\n    metrics=[\'rgb_mse\',\'detail_mse\',\'bad_detail_rate_0.025\',\'bad_detail_rate_0.05\',\'bad_detail_rate_0.1\']\n    agg=risk.groupby([\'stage\',\'pipeline\',\'region\',\'score\',\'coverage\'])[metrics].mean().sort_index()\n    recorded=read(\'risk_summary\').set_index([\'stage\',\'pipeline\',\'region\',\'score\',\'coverage\'])[metrics].sort_index()\n    assert agg.index.equals(recorded.index) and np.allclose(agg,recorded,rtol=0,atol=1e-12)\n    for row in read(\'quality_changes\').to_dict(\'records\'):\n        frame=quality[(quality.source_id==row[\'source_id\'])&(quality.stage==row[\'stage\'])].set_index(\'model\')\n        for metric in [\'mse\',\'detail_mse\',\'psnr_db\']:\n            assert abs(frame.loc[row[\'method\'],metric]-frame.loc[row[\'comparator\'],metric]-row[\'delta_\'+metric])<1e-12\n    for row in read(\'selection_comparisons\').to_dict(\'records\'):\n        f=risk[(risk.source_id==row[\'source_id\'])&(risk.stage==row[\'stage\'])&(risk.pipeline==row[\'pipeline\'])&\n               (risk.region==row[\'region\'])&(risk.coverage==row[\'coverage\'])].set_index(\'score\')\n        for metric in metrics:assert abs(f.loc[\'operator_spread_detail\',metric]-f.loc[row[\'control\'],metric]-row[\'delta_\'+metric])<1e-12\n    compute=read(\'compute\');costs=read(\'score_costs\');diag=read(\'fbcnn_diagnostics\')\n    assert len(compute)==15*n and compute.denoiser_calls.sum()==80*n and compute.fbcnn_calls.sum()==3*n\n    assert len(read(\'trajectories\'))==80*n and len(diag)==3*n and len(costs)==4*n\n    assert diag.predicted_degradation.between(0,1).all() and np.allclose(diag.predicted_quality,100*(1-diag.predicted_degradation))\n    for row in costs.itertuples(index=False):\n        f=compute[(compute.source_id==row.source_id)&(compute.stage==row.stage)].set_index(\'component\')\n        names=[row.pipeline]+[row.pipeline+\'_\'+s for s in ([\'sigma08\',\'sigma12\'] if row.score_family==\'operator\' else [\'rot90\',\'rot180\'])]\n        if row.pipeline==\'fbcnn_dpir\':names+=[\'fbcnn\'] if row.score_family==\'operator\' else [\'fbcnn\',\'fbcnn_rot90_intermediate\',\'fbcnn_rot180_intermediate\']\n        assert abs(f.loc[names].elapsed_seconds.sum()-row.elapsed_seconds)<1e-7\n        assert f.loc[names].denoiser_calls.sum()==row.denoiser_calls and f.loc[names].fbcnn_calls.sum()==row.fbcnn_calls\n    decision=json.loads((run/\'decisions.json\').read_text())\n    full=set(cfg[\'source_ids\'])=={\'0801\',\'0802\',\'0803\',\'0804\'} and n==8\n    assert decision[\'full_design_assessed\']==full\n    if not full:assert decision[\'reconstruction_screen_pass\'] is None and decision[\'selection_screen_pass\'] is None\n    else:\n        means=quality[quality.stage==\'jpeg_q75\'].groupby(\'model\').detail_mse.mean()\n        q=quality[quality.stage==\'jpeg_q75\'].pivot(index=\'source_id\',columns=\'model\',values=\'detail_mse\')\n        recon=bool(all(means.fbcnn_dpir<means[c] for c in [\'dpir_nominal\',\'gradient_nominal\',\'fbcnn_gradient\']) and (q.fbcnn_dpir<q.dpir_nominal).sum()>=3)\n        controls=[\'image_transform_spread_detail\',\'measurement_residual\',\'image_gradient\',\'random_expected\']\n        r=risk.query("stage==\'jpeg_q75\' and pipeline==\'fbcnn_dpir\' and region==\'all\' and coverage==0.5")\n        m=r.groupby(\'score\')[metrics].mean();best=m.loc[controls,\'detail_mse\'].idxmin()\n        p=r.pivot(index=\'source_id\',columns=\'score\',values=\'detail_mse\');op=m.loc[\'operator_spread_detail\']\n        selection=bool((op.detail_mse<m.loc[controls,\'detail_mse\']).all() and (p.operator_spread_detail<p[best]).sum()>=3 and\n                       all((op[col]<=m.loc[controls,col]+1e-12).all() for col in metrics[2:]))\n        assert decision[\'reconstruction_screen_pass\']==recon and decision[\'selection_screen_pass\']==selection\n    assert checked==14*n and risks==112*n\n    return dict(status=\'passed\',configured_sources=cfg[\'source_ids\'],observations=n,full_design_complete=full,\n        manifest_files_verified_at_readback=len(files),quality_rows_recomputed=checked,patch_error_vectors=patches,\n        score_vectors_recomputed=scored,risk_rows_recomputed=risks,max_abs_quality_difference=maximum,\n        max_abs_patch_difference=patch_max,max_abs_score_difference=score_max,max_abs_risk_difference=risk_max,\n        source_hashes_verified=4,summary_and_contrasts_verified=True,compute_accounting_verified=True,\n        decision_scope_verified=True,neural_inference_repeated=False)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run-dir\',required=True,type=Path);parser.add_argument(\'--data-dir\',required=True,type=Path)\n    parser.add_argument(\'--audit-output\',required=True,type=Path)\n    args=parser.parse_args();audit=validate(args.run_dir,args.data_dir)\n    args.audit_output.write_text(json.dumps(audit,indent=2)+\'\\n\');print(json.dumps(audit,indent=2))\n', 'archive': '"""Verified summary/full ZIP export and bounded transfer parts for experiment 04."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef verify_manifest(run):\n    run=Path(run);manifest=json.loads((run/\'export_manifest.json\').read_text())\n    assert json.loads((run/\'status.json\').read_text())[\'status\']==\'complete_for_configured_subset\'\n    names={r[\'name\'] for r in manifest[\'files\']}\n    assert len(names)==len(manifest[\'files\'])\n    assert names=={str(p.relative_to(run)) for p in run.rglob(\'*\') if p.is_file() and p.name!=\'export_manifest.json\'}\n    for r in manifest[\'files\']:\n        p=run/r[\'name\'];assert p.stat().st_size==r[\'bytes\'] and sha(p)==r[\'sha256\']\n    return manifest\n\n\ndef make_zip(target,run,names,readme=None):\n    target=Path(target);run=Path(run)\n    expected={run.name+\'/\'+n:run/n for n in names}\n    if not target.exists():\n        temporary=target.with_suffix(\'.zip.partial\')\n        with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n            for name,p in sorted(expected.items()):z.write(p,name)\n            if readme:z.writestr(\'READ_ME_FIRST.txt\',readme)\n        temporary.replace(target)\n    with zipfile.ZipFile(target) as z:\n        assert z.testzip() is None\n        assert set(z.namelist())==set(expected)|({\'READ_ME_FIRST.txt\'} if readme else set())\n        for name,p in expected.items():\n            h=hashlib.sha256()\n            with z.open(name) as f:\n                for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n            assert h.hexdigest()==sha(p),name\n        if readme:assert z.read(\'READ_ME_FIRST.txt\').decode()==readme\n    return dict(path=str(target),bytes=target.stat().st_size,sha256=sha(target),entries=len(expected)+(1 if readme else 0))\n\n\ndef split_archive(archive,chunk_bytes=96*1024**2):\n    archive=Path(archive);assert 0<chunk_bytes<=96*1024**2\n    whole_hash=sha(archive);size=archive.stat().st_size;count=(size+chunk_bytes-1)//chunk_bytes\n    folder=archive.parent/(archive.stem+\'_upload_parts\');folder.mkdir(exist_ok=True)\n    parts=[]\n    with archive.open(\'rb\') as source:\n        for index in range(1,count+1):\n            block=source.read(chunk_bytes)\n            info=dict(format=\'jpeg_aware04_raw_zip_chunks_v1\',archive_name=archive.name,\n                archive_sha256=whole_hash,archive_bytes=size,index=index,total_parts=count,\n                chunk_bytes=len(block),chunk_sha256=hashlib.sha256(block).hexdigest())\n            part=folder/f\'{archive.stem}_part_{index:03d}_of_{count:03d}.zip\'\n            if not part.exists():\n                temporary=part.with_suffix(\'.zip.partial\')\n                with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_STORED) as z:\n                    z.writestr(\'archive_chunk.bin\',block)\n                    z.writestr(\'transfer_part.json\',json.dumps(info,indent=2)+\'\\n\')\n                temporary.replace(part)\n            with zipfile.ZipFile(part) as z:\n                assert z.testzip() is None and set(z.namelist())=={\'archive_chunk.bin\',\'transfer_part.json\'}\n                assert json.loads(z.read(\'transfer_part.json\'))==info\n                assert z.read(\'archive_chunk.bin\')==block\n            assert part.stat().st_size<100*1024**2\n            parts.append(part)\n    verified=verify_parts(parts)\n    assert verified[\'archive_sha256\']==whole_hash\n    return parts,verified\n\n\ndef verify_parts(parts,destination=None):\n    """Verify complete unordered parts; optionally reconstruct a previously absent ZIP."""\n    records=[]\n    for p in map(Path,parts):\n        with zipfile.ZipFile(p) as z:\n            assert z.testzip() is None and set(z.namelist())=={\'archive_chunk.bin\',\'transfer_part.json\'}\n            m=json.loads(z.read(\'transfer_part.json\'))\n            assert m[\'format\']==\'jpeg_aware04_raw_zip_chunks_v1\'\n            records.append((m,p))\n    assert records\n    records.sort(key=lambda a:a[0][\'index\']);first=records[0][0]\n    assert [m[\'index\'] for m,p in records]==list(range(1,first[\'total_parts\']+1)), \'Missing/duplicate transfer part\'\n    if destination is not None:\n        destination=Path(destination);assert not destination.exists()\n        output=destination.with_suffix(destination.suffix+\'.partial\');assert not output.exists()\n        stream=output.open(\'xb\')\n    else:stream=None\n    digest=hashlib.sha256();size=0\n    try:\n        for m,p in records:\n            assert all(m[k]==first[k] for k in [\'archive_name\',\'archive_sha256\',\'archive_bytes\',\'total_parts\'])\n            with zipfile.ZipFile(p) as z:block=z.read(\'archive_chunk.bin\')\n            assert len(block)==m[\'chunk_bytes\'] and hashlib.sha256(block).hexdigest()==m[\'chunk_sha256\']\n            digest.update(block);size+=len(block)\n            if stream:stream.write(block)\n        assert size==first[\'archive_bytes\'] and digest.hexdigest()==first[\'archive_sha256\']\n    finally:\n        if stream:stream.close()\n    if destination is not None:output.replace(destination)\n    return dict(archive_name=first[\'archive_name\'],archive_bytes=size,archive_sha256=digest.hexdigest(),\n        verified_parts=len(records),part_paths=[str(p) for m,p in records])\n\n\ndef export(run):\n    run=Path(run);manifest=verify_manifest(run)\n    names=[r[\'name\'] for r in manifest[\'files\']]+[\'export_manifest.json\']\n    summary_names=[n for n in names if not n.startswith((\'predictions/\',\'codec_inputs/\'))]\n    summary=make_zip(run.parent/(run.name+\'_SUMMARY.zip\'),run,summary_names,\n        \'SUMMARY ONLY. Raw predictions and codec inputs are omitted. The included full-run manifest lists omitted files too.\\n\'\n        \'Use ALL RAW transfer parts for independent saved-array verification.\\n\')\n    raw_path=run.parent/(run.name+\'_RAW.zip\');raw=make_zip(raw_path,run,names)\n    parts,verification=split_archive(raw_path)\n    receipt=dict(experiment=\'jpeg_aware_04\',summary=summary,raw=raw,transfer=verification,\n        manifest_files_verified=len(manifest[\'files\']),full_design_complete=json.loads((run/\'status.json\').read_text())[\'full_design_complete\'])\n    (run.parent/(run.name+\'_EXPORT_RECEIPT.json\')).write_text(json.dumps(receipt,indent=2)+\'\\n\')\n    return receipt\n'}
PROVENANCE = {'experiment': 'jpeg_aware_04', 'parent_checkpoint': '0021f09ddc2da06cf54a8835ba5eaffd7352cc79', 'dpir': {'upstream_repository': 'cszn/DPIR', 'upstream_commit': '15bca3fcc1f3cc51a1f99ccf027691e278c19354', 'weight_url': 'https://github.com/cszn/KAIR/releases/download/v1.0/drunet_color.pth', 'weight_bytes': 130579305, 'weight_sha256': '479abe3c5327dfd10ff54a80ec7d4098ca80752a5c9492cdff31cee430bec4b4', 'digest_authority': 'observed download; not a publisher-signed checksum', 'files': {'LICENSE': 'afd973a2b98d60d178342477feed2c5673640288b0663e83e33e0e63ece9634c', 'models/basicblock.py': 'aab51f43fee655fda5c499a0481fffd45bf390bdb8229944262c41e0a900cec1', 'models/network_unet.py': '41c3cc38b2eac3b239aec6a723aefef6c138e5dd8ceb6cd9e18f825248261e91', 'utils/utils_pnp.py': '1ac53932f7c47c55b622b88453e2440281d338be988a289240573764dd5421a4'}, 'training_overlap': 'Paper v2 reports 900 DIV2K training images; these four DIV2K sources are development only; exact checkpoint training image manifest unavailable.'}, 'fbcnn': {'upstream_repository': 'jiaxi-jiang/FBCNN', 'upstream_commit': '54d1831927506b3247e2d4d245abb4f4dab1a1cd', 'weight_url': 'https://github.com/jiaxi-jiang/FBCNN/releases/download/v1.0/fbcnn_color.pth', 'weight_bytes': 287755111, 'weight_sha256': '8b0e4ef23d59cf7ac934a342cb31a17619e4fa4a0b3374a9d78c5174312387e8', 'digest_authority': 'observed official-release download; not publisher-signed', 'files': {'LICENSE': 'c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4', 'data/dataset_jpeg.py': 'cb0f908f0f243552fcbdfe8eb81332f67c7caf217701c8f713959ac936665de1', 'main_test_fbcnn_color.py': '0367416fb68c9ab2c6f2f130d2a6620048e2440a800c803a07bbda8a9ecc1a64', 'models/network_fbcnn.py': 'e1afc1f6483d57b774365467393216508f29a708b576dc973bad41bae67e8d03', 'options/train_fbcnn_color.json': '94f69fd16a4acd3183a8a8ced2ee2d2cfa142319c66dde723fc70c7e9a89d585'}, 'license': 'Apache-2.0', 'architecture': 'official colour test; in/out 3; nc 64,128,256,512; nb 4; act_mode R', 'adaptation': 'AST removes verified-unused torchvision.models import only; full tensor, automatic quality, clip [0,1], no output uint8 rounding', 'training_overlap': 'Paper reports DIV2K and Flickr2K; exact released-checkpoint image manifest unavailable. These four sources are exposed development data.', 'quality_normalization': 'q=(100-quality)/100, verified in official dataset_jpeg.py; report 100*(1-q).', 'codec_limit': 'Paper reports MATLAB JPEG training; repository dataset/demo use OpenCV. Our frozen acquisition uses Pillow JPEG quality75/subsampling0; not a paper-benchmark reproduction.'}, 'embedded_source_sha256': {'baseline': 'ecc6317921f1b3ed43fee97944306bfb64a089ea43626b835fce76b65925ded3', 'learned': '52694138eeb414d1b80ea9761a1ff4edd56594449d73d56448a0d4e1229a73f8', 'diagnostic': '15b60df4d8f90dae43d4259e2ed16290f01366f0e2d629971c7545e11bcd9b41', 'jpeg_aware': '3e0cecf1d9d59cf68150f22dfbf1fb351f36f0feb6859ec286b9bdf4f0177d63', 'validator': '157fff0575c393070bbef5164524d7b41e0a0ae5a895300db872181da5a89fa4', 'archive': '3ee21e786a69e48518ae57d78115402748fb02f3cd3027aaed5f802bf0dc1c20'}, 'source_paths': {'baseline': 'experiments/baseline_01/baseline.py', 'learned': 'experiments/learned_02/learned.py', 'diagnostic': 'experiments/acquisition_03/diagnostic.py', 'jpeg_aware': 'experiments/jpeg_aware_04/jpeg_aware.py', 'validator': 'scripts/validate_jpeg_aware_04.py', 'archive': 'experiments/jpeg_aware_04/archive_io.py'}, 'design_freeze_sha256': 'bbeda736bb0893aa7e9ddfa83678e0d00d3004ae02e50f7cf28f3444a90618f4', 'anchor_sha256': {'quality.csv': 'd841df2fb30d7a5b34d177f9e48efe795676f267c0ab782dd0978c7398d7ab00', 'acquisition.csv': '8597ab9115b0c93fa15e3435ff9d6960642f68f42d1d4274ac8af8bfe0fa7be4'}}
VENDORS = {'dpir': {'LICENSE': 'MIT License\n\nCopyright (c) 2020 Kai Zhang (cskaizhang@gmail.com)\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n', 'models/basicblock.py': 'from collections import OrderedDict\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n\'\'\'\n# --------------------------------------------\n# Advanced nn.Sequential\n# https://github.com/xinntao/BasicSR\n# --------------------------------------------\n\'\'\'\n\n\ndef sequential(*args):\n    """Advanced nn.Sequential.\n\n    Args:\n        nn.Sequential, nn.Module\n\n    Returns:\n        nn.Sequential\n    """\n    if len(args) == 1:\n        if isinstance(args[0], OrderedDict):\n            raise NotImplementedError(\'sequential does not support OrderedDict input.\')\n        return args[0]  # No sequential is needed.\n    modules = []\n    for module in args:\n        if isinstance(module, nn.Sequential):\n            for submodule in module.children():\n                modules.append(submodule)\n        elif isinstance(module, nn.Module):\n            modules.append(module)\n    return nn.Sequential(*modules)\n\n\n\'\'\'\n# --------------------------------------------\n# Useful blocks\n# https://github.com/xinntao/BasicSR\n# --------------------------------\n# conv + normaliation + relu (conv)\n# (PixelUnShuffle)\n# (ConditionalBatchNorm2d)\n# concat (ConcatBlock)\n# sum (ShortcutBlock)\n# resblock (ResBlock)\n# Channel Attention (CA) Layer (CALayer)\n# Residual Channel Attention Block (RCABlock)\n# Residual Channel Attention Group (RCAGroup)\n# Residual Dense Block (ResidualDenseBlock_5C)\n# Residual in Residual Dense Block (RRDB)\n# --------------------------------------------\n\'\'\'\n\n\n# --------------------------------------------\n# return nn.Sequantial of (Conv + BN + ReLU)\n# --------------------------------------------\ndef conv(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CBR\', negative_slope=0.2):\n    L = []\n    for t in mode:\n        if t == \'C\':\n            L.append(nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias))\n        elif t == \'T\':\n            L.append(nn.ConvTranspose2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias))\n        elif t == \'B\':\n            L.append(nn.BatchNorm2d(out_channels, momentum=0.9, eps=1e-04, affine=True))\n        elif t == \'I\':\n            L.append(nn.InstanceNorm2d(out_channels, affine=True))\n        elif t == \'R\':\n            L.append(nn.ReLU(inplace=True))\n        elif t == \'r\':\n            L.append(nn.ReLU(inplace=False))\n        elif t == \'L\':\n            L.append(nn.LeakyReLU(negative_slope=negative_slope, inplace=True))\n        elif t == \'l\':\n            L.append(nn.LeakyReLU(negative_slope=negative_slope, inplace=False))\n        elif t == \'2\':\n            L.append(nn.PixelShuffle(upscale_factor=2))\n        elif t == \'3\':\n            L.append(nn.PixelShuffle(upscale_factor=3))\n        elif t == \'4\':\n            L.append(nn.PixelShuffle(upscale_factor=4))\n        elif t == \'U\':\n            L.append(nn.Upsample(scale_factor=2, mode=\'nearest\'))\n        elif t == \'u\':\n            L.append(nn.Upsample(scale_factor=3, mode=\'nearest\'))\n        elif t == \'v\':\n            L.append(nn.Upsample(scale_factor=4, mode=\'nearest\'))\n        elif t == \'M\':\n            L.append(nn.MaxPool2d(kernel_size=kernel_size, stride=stride, padding=0))\n        elif t == \'A\':\n            L.append(nn.AvgPool2d(kernel_size=kernel_size, stride=stride, padding=0))\n        else:\n            raise NotImplementedError(\'Undefined type: \'.format(t))\n    return sequential(*L)\n\n\n# --------------------------------------------\n# inverse of pixel_shuffle\n# --------------------------------------------\ndef pixel_unshuffle(input, upscale_factor):\n    r"""Rearranges elements in a Tensor of shape :math:`(C, rH, rW)` to a\n    tensor of shape :math:`(*, r^2C, H, W)`.\n\n    Authors:\n        Zhaoyi Yan, https://github.com/Zhaoyi-Yan\n        Kai Zhang, https://github.com/cszn/FFDNet\n\n    Date:\n        01/Jan/2019\n    """\n    batch_size, channels, in_height, in_width = input.size()\n\n    out_height = in_height // upscale_factor\n    out_width = in_width // upscale_factor\n\n    input_view = input.contiguous().view(\n        batch_size, channels, out_height, upscale_factor,\n        out_width, upscale_factor)\n\n    channels *= upscale_factor ** 2\n    unshuffle_out = input_view.permute(0, 1, 3, 5, 2, 4).contiguous()\n    return unshuffle_out.view(batch_size, channels, out_height, out_width)\n\n\nclass PixelUnShuffle(nn.Module):\n    r"""Rearranges elements in a Tensor of shape :math:`(C, rH, rW)` to a\n    tensor of shape :math:`(*, r^2C, H, W)`.\n\n    Authors:\n        Zhaoyi Yan, https://github.com/Zhaoyi-Yan\n        Kai Zhang, https://github.com/cszn/FFDNet\n\n    Date:\n        01/Jan/2019\n    """\n\n    def __init__(self, upscale_factor):\n        super(PixelUnShuffle, self).__init__()\n        self.upscale_factor = upscale_factor\n\n    def forward(self, input):\n        return pixel_unshuffle(input, self.upscale_factor)\n\n    def extra_repr(self):\n        return \'upscale_factor={}\'.format(self.upscale_factor)\n\n\n# --------------------------------------------\n# conditional batch norm\n# https://github.com/pytorch/pytorch/issues/8985#issuecomment-405080775\n# --------------------------------------------\nclass ConditionalBatchNorm2d(nn.Module):\n    def __init__(self, num_features, num_classes):\n        super().__init__()\n        self.num_features = num_features\n        self.bn = nn.BatchNorm2d(num_features, affine=False)\n        self.embed = nn.Embedding(num_classes, num_features * 2)\n        self.embed.weight.data[:, :num_features].normal_(1, 0.02)  # Initialise scale at N(1, 0.02)\n        self.embed.weight.data[:, num_features:].zero_()  # Initialise bias at 0\n\n    def forward(self, x, y):\n        out = self.bn(x)\n        gamma, beta = self.embed(y).chunk(2, 1)\n        out = gamma.view(-1, self.num_features, 1, 1) * out + beta.view(-1, self.num_features, 1, 1)\n        return out\n\n\n# --------------------------------------------\n# Concat the output of a submodule to its input\n# --------------------------------------------\nclass ConcatBlock(nn.Module):\n    def __init__(self, submodule):\n        super(ConcatBlock, self).__init__()\n        self.sub = submodule\n\n    def forward(self, x):\n        output = torch.cat((x, self.sub(x)), dim=1)\n        return output\n\n    def __repr__(self):\n        return self.sub.__repr__() + \'concat\'\n\n\n# --------------------------------------------\n# sum the output of a submodule to its input\n# --------------------------------------------\nclass ShortcutBlock(nn.Module):\n    def __init__(self, submodule):\n        super(ShortcutBlock, self).__init__()\n\n        self.sub = submodule\n\n    def forward(self, x):\n        output = x + self.sub(x)\n        return output\n\n    def __repr__(self):\n        tmpstr = \'Identity + \\n|\'\n        modstr = self.sub.__repr__().replace(\'\\n\', \'\\n|\')\n        tmpstr = tmpstr + modstr\n        return tmpstr\n\n\n# --------------------------------------------\n# Res Block: x + conv(relu(conv(x)))\n# --------------------------------------------\nclass ResBlock(nn.Module):\n    def __init__(self, in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CRC\', negative_slope=0.2):\n        super(ResBlock, self).__init__()\n\n        assert in_channels == out_channels, \'Only support in_channels==out_channels.\'\n        if mode[0] in [\'R\', \'L\']:\n            mode = mode[0].lower() + mode[1:]\n\n        self.res = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n\n    def forward(self, x):\n        #res = self.res(x)\n        return x + self.res(x)\n\n\n# --------------------------------------------\n# simplified information multi-distillation block (IMDB)\n# x + conv1(concat(split(relu(conv(x)))x3))\n# --------------------------------------------\nclass IMDBlock(nn.Module):\n    """\n    @inproceedings{hui2019lightweight,\n      title={Lightweight Image Super-Resolution with Information Multi-distillation Network},\n      author={Hui, Zheng and Gao, Xinbo and Yang, Yunchu and Wang, Xiumei},\n      booktitle={Proceedings of the 27th ACM International Conference on Multimedia (ACM MM)},\n      pages={2024--2032},\n      year={2019}\n    }\n    @inproceedings{zhang2019aim,\n      title={AIM 2019 Challenge on Constrained Super-Resolution: Methods and Results},\n      author={Kai Zhang and Shuhang Gu and Radu Timofte and others},\n      booktitle={IEEE International Conference on Computer Vision Workshops},\n      year={2019}\n    }\n    """\n    def __init__(self, in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CL\', d_rate=0.25, negative_slope=0.05):\n        super(IMDBlock, self).__init__()\n        self.d_nc = int(in_channels * d_rate)\n        self.r_nc = int(in_channels - self.d_nc)\n\n        assert mode[0] == \'C\', \'convolutional layer first\'\n\n        self.conv1 = conv(in_channels, in_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv2 = conv(self.r_nc, in_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv3 = conv(self.r_nc, in_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv4 = conv(self.r_nc, self.d_nc, kernel_size, stride, padding, bias, mode[0], negative_slope)\n        self.conv1x1 = conv(self.d_nc*4, out_channels, kernel_size=1, stride=1, padding=0, bias=bias, mode=mode[0], negative_slope=negative_slope)\n\n    def forward(self, x):\n        d1, r = torch.split(self.conv1(x), (self.d_nc, self.r_nc), dim=1)\n        d2, r = torch.split(self.conv2(r), (self.d_nc, self.r_nc), dim=1)\n        d3, r = torch.split(self.conv3(r), (self.d_nc, self.r_nc), dim=1)\n        r = self.conv4(r)\n        res = self.conv1x1(torch.cat((d1, d2, d3, r), dim=1))\n        return x + res\n\n\n#        d1, r1 = torch.split(self.conv1(x), (self.d_nc, self.r_nc), dim=1)\n#        d2, r2 = torch.split(self.conv2(r1), (self.d_nc, self.r_nc), dim=1)\n#        d3, r3 = torch.split(self.conv3(r2), (self.d_nc, self.r_nc), dim=1)\n#        d4 = self.conv4(r3)\n# --------------------------------------------\n# Channel Attention (CA) Layer\n# --------------------------------------------\nclass CALayer(nn.Module):\n    def __init__(self, channel=64, reduction=16):\n        super(CALayer, self).__init__()\n\n        self.avg_pool = nn.AdaptiveAvgPool2d(1)\n        self.conv_fc = nn.Sequential(\n                nn.Conv2d(channel, channel // reduction, 1, padding=0, bias=True),\n                nn.ReLU(inplace=True),\n                nn.Conv2d(channel // reduction, channel, 1, padding=0, bias=True),\n                nn.Sigmoid()\n        )\n\n    def forward(self, x):\n        y = self.avg_pool(x)\n        y = self.conv_fc(y)\n        return x * y\n\n\n# --------------------------------------------\n# Residual Channel Attention Block (RCAB)\n# --------------------------------------------\nclass RCABlock(nn.Module):\n    def __init__(self, in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CRC\', reduction=16, negative_slope=0.2):\n        super(RCABlock, self).__init__()\n        assert in_channels == out_channels, \'Only support in_channels==out_channels.\'\n        if mode[0] in [\'R\',\'L\']:\n            mode = mode[0].lower() + mode[1:]\n\n        self.res = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.ca = CALayer(out_channels, reduction)\n\n    def forward(self, x):\n        res = self.res(x)\n        res = self.ca(res)\n        return res + x\n\n\n# --------------------------------------------\n# Residual Channel Attention Group (RG)\n# --------------------------------------------\nclass RCAGroup(nn.Module):\n    def __init__(self, in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CRC\', reduction=16, nb=12, negative_slope=0.2):\n        super(RCAGroup, self).__init__()\n        assert in_channels == out_channels, \'Only support in_channels==out_channels.\'\n        if mode[0] in [\'R\',\'L\']:\n            mode = mode[0].lower() + mode[1:]\n\n        RG = [RCABlock(in_channels, out_channels, kernel_size, stride, padding, bias, mode, reduction, negative_slope)  for _ in range(nb)]\n        RG.append(conv(out_channels, out_channels, mode=\'C\'))\n        self.rg = nn.Sequential(*RG)  # self.rg = ShortcutBlock(nn.Sequential(*RG))\n\n    def forward(self, x):\n        res = self.rg(x)\n        return res + x\n\n\n# --------------------------------------------\n# Residual Dense Block\n# style: 5 convs\n# --------------------------------------------\nclass ResidualDenseBlock_5C(nn.Module):\n    def __init__(self, nc=64, gc=32, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CR\', negative_slope=0.2):\n        super(ResidualDenseBlock_5C, self).__init__()\n        # gc: growth channel\n        self.conv1 = conv(nc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv2 = conv(nc+gc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv3 = conv(nc+2*gc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv4 = conv(nc+3*gc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.conv5 = conv(nc+4*gc, nc, kernel_size, stride, padding, bias, mode[:-1], negative_slope)\n\n    def forward(self, x):\n        x1 = self.conv1(x)\n        x2 = self.conv2(torch.cat((x, x1), 1))\n        x3 = self.conv3(torch.cat((x, x1, x2), 1))\n        x4 = self.conv4(torch.cat((x, x1, x2, x3), 1))\n        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))\n        return x5.mul_(0.2) + x\n\n\n# --------------------------------------------\n# Residual in Residual Dense Block\n# 3x5c\n# --------------------------------------------\nclass RRDB(nn.Module):\n    def __init__(self, nc=64, gc=32, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CR\', negative_slope=0.2):\n        super(RRDB, self).__init__()\n\n        self.RDB1 = ResidualDenseBlock_5C(nc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.RDB2 = ResidualDenseBlock_5C(nc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n        self.RDB3 = ResidualDenseBlock_5C(nc, gc, kernel_size, stride, padding, bias, mode, negative_slope)\n\n    def forward(self, x):\n        out = self.RDB1(x)\n        out = self.RDB2(out)\n        out = self.RDB3(out)\n        return out.mul_(0.2) + x\n\n\n"""\n# --------------------------------------------\n# Upsampler\n# Kai Zhang, https://github.com/cszn/KAIR\n# --------------------------------------------\n# upsample_pixelshuffle\n# upsample_upconv\n# upsample_convtranspose\n# --------------------------------------------\n"""\n\n\n# --------------------------------------------\n# conv + subp (+ relu)\n# --------------------------------------------\ndef upsample_pixelshuffle(in_channels=64, out_channels=3, kernel_size=3, stride=1, padding=1, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR.\'\n    up1 = conv(in_channels, out_channels * (int(mode[0]) ** 2), kernel_size, stride, padding, bias, mode=\'C\'+mode, negative_slope=negative_slope)\n    return up1\n\n\n# --------------------------------------------\n# nearest_upsample + conv (+ R)\n# --------------------------------------------\ndef upsample_upconv(in_channels=64, out_channels=3, kernel_size=3, stride=1, padding=1, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR\'\n    if mode[0] == \'2\':\n        uc = \'UC\'\n    elif mode[0] == \'3\':\n        uc = \'uC\'\n    elif mode[0] == \'4\':\n        uc = \'vC\'\n    mode = mode.replace(mode[0], uc)\n    up1 = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode=mode, negative_slope=negative_slope)\n    return up1\n\n\n# --------------------------------------------\n# convTranspose (+ relu)\n# --------------------------------------------\ndef upsample_convtranspose(in_channels=64, out_channels=3, kernel_size=2, stride=2, padding=0, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR.\'\n    kernel_size = int(mode[0])\n    stride = int(mode[0])\n    mode = mode.replace(mode[0], \'T\')\n    up1 = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n    return up1\n\n\n\'\'\'\n# --------------------------------------------\n# Downsampler\n# Kai Zhang, https://github.com/cszn/KAIR\n# --------------------------------------------\n# downsample_strideconv\n# downsample_maxpool\n# downsample_avgpool\n# --------------------------------------------\n\'\'\'\n\n\n# --------------------------------------------\n# strideconv (+ relu)\n# --------------------------------------------\ndef downsample_strideconv(in_channels=64, out_channels=64, kernel_size=2, stride=2, padding=0, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR.\'\n    kernel_size = int(mode[0])\n    stride = int(mode[0])\n    mode = mode.replace(mode[0], \'C\')\n    down1 = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n    return down1\n\n\n# --------------------------------------------\n# maxpooling + conv (+ relu)\n# --------------------------------------------\ndef downsample_maxpool(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=0, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\'], \'mode examples: 2, 2R, 2BR, 3, ..., 3BR.\'\n    kernel_size_pool = int(mode[0])\n    stride_pool = int(mode[0])\n    mode = mode.replace(mode[0], \'MC\')\n    pool = conv(kernel_size=kernel_size_pool, stride=stride_pool, mode=mode[0], negative_slope=negative_slope)\n    pool_tail = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode=mode[1:], negative_slope=negative_slope)\n    return sequential(pool, pool_tail)\n\n\n# --------------------------------------------\n# averagepooling + conv (+ relu)\n# --------------------------------------------\ndef downsample_avgpool(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\'], \'mode examples: 2, 2R, 2BR, 3, ..., 3BR.\'\n    kernel_size_pool = int(mode[0])\n    stride_pool = int(mode[0])\n    mode = mode.replace(mode[0], \'AC\')\n    pool = conv(kernel_size=kernel_size_pool, stride=stride_pool, mode=mode[0], negative_slope=negative_slope)\n    pool_tail = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode=mode[1:], negative_slope=negative_slope)\n    return sequential(pool, pool_tail)\n\n\n\'\'\'\n# --------------------------------------------\n# NonLocalBlock2D:\n# embedded_gaussian\n# +W(softmax(thetaXphi)Xg)\n# --------------------------------------------\n\'\'\'\n\n\n# --------------------------------------------\n# non-local block with embedded_gaussian\n# https://github.com/AlexHex7/Non-local_pytorch\n# --------------------------------------------\nclass NonLocalBlock2D(nn.Module):\n    def __init__(self, nc=64, kernel_size=1, stride=1, padding=0, bias=True, act_mode=\'B\', downsample=False, downsample_mode=\'maxpool\', negative_slope=0.2):\n\n        super(NonLocalBlock2D, self).__init__()\n\n        inter_nc = nc // 2\n        self.inter_nc = inter_nc\n        self.W = conv(inter_nc, nc, kernel_size, stride, padding, bias, mode=\'C\'+act_mode)\n        self.theta = conv(nc, inter_nc, kernel_size, stride, padding, bias, mode=\'C\')\n\n        if downsample:\n            if downsample_mode == \'avgpool\':\n                downsample_block = downsample_avgpool\n            elif downsample_mode == \'maxpool\':\n                downsample_block = downsample_maxpool\n            elif downsample_mode == \'strideconv\':\n                downsample_block = downsample_strideconv\n            else:\n                raise NotImplementedError(\'downsample mode [{:s}] is not found\'.format(downsample_mode))\n            self.phi = downsample_block(nc, inter_nc, kernel_size, stride, padding, bias, mode=\'2\')\n            self.g = downsample_block(nc, inter_nc, kernel_size, stride, padding, bias, mode=\'2\')\n        else:\n            self.phi = conv(nc, inter_nc, kernel_size, stride, padding, bias, mode=\'C\')\n            self.g = conv(nc, inter_nc, kernel_size, stride, padding, bias, mode=\'C\')\n\n    def forward(self, x):\n        \'\'\'\n        :param x: (b, c, t, h, w)\n        :return:\n        \'\'\'\n\n        batch_size = x.size(0)\n\n        g_x = self.g(x).view(batch_size, self.inter_nc, -1)\n        g_x = g_x.permute(0, 2, 1)\n\n        theta_x = self.theta(x).view(batch_size, self.inter_nc, -1)\n        theta_x = theta_x.permute(0, 2, 1)\n        phi_x = self.phi(x).view(batch_size, self.inter_nc, -1)\n        f = torch.matmul(theta_x, phi_x)\n        f_div_C = F.softmax(f, dim=-1)\n\n        y = torch.matmul(f_div_C, g_x)\n        y = y.permute(0, 2, 1).contiguous()\n        y = y.view(batch_size, self.inter_nc, *x.size()[2:])\n        W_y = self.W(y)\n        z = W_y + x\n\n        return z\n', 'models/network_unet.py': "import torch\nimport torch.nn as nn\nimport models.basicblock as B\nimport numpy as np\n\n'''\n# ====================\n# unet\n# ====================\n'''\n\n\nclass UNet(nn.Module):\n    def __init__(self, in_nc=1, out_nc=1, nc=[64, 128, 256, 512], nb=2, act_mode='R', downsample_mode='strideconv', upsample_mode='convtranspose'):\n        super(UNet, self).__init__()\n\n        self.m_head = B.conv(in_nc, nc[0], mode='C'+act_mode[-1])\n\n        # downsample\n        if downsample_mode == 'avgpool':\n            downsample_block = B.downsample_avgpool\n        elif downsample_mode == 'maxpool':\n            downsample_block = B.downsample_maxpool\n        elif downsample_mode == 'strideconv':\n            downsample_block = B.downsample_strideconv\n        else:\n            raise NotImplementedError('downsample mode [{:s}] is not found'.format(downsample_mode))\n\n        self.m_down1 = B.sequential(*[B.conv(nc[0], nc[0], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[0], nc[1], mode='2'+act_mode))\n        self.m_down2 = B.sequential(*[B.conv(nc[1], nc[1], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[1], nc[2], mode='2'+act_mode))\n        self.m_down3 = B.sequential(*[B.conv(nc[2], nc[2], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[2], nc[3], mode='2'+act_mode))\n\n        self.m_body  = B.sequential(*[B.conv(nc[3], nc[3], mode='C'+act_mode) for _ in range(nb+1)])\n\n        # upsample\n        if upsample_mode == 'upconv':\n            upsample_block = B.upsample_upconv\n        elif upsample_mode == 'pixelshuffle':\n            upsample_block = B.upsample_pixelshuffle\n        elif upsample_mode == 'convtranspose':\n            upsample_block = B.upsample_convtranspose\n        else:\n            raise NotImplementedError('upsample mode [{:s}] is not found'.format(upsample_mode))\n\n        self.m_up3 = B.sequential(upsample_block(nc[3], nc[2], mode='2'+act_mode), *[B.conv(nc[2], nc[2], mode='C'+act_mode) for _ in range(nb)])\n        self.m_up2 = B.sequential(upsample_block(nc[2], nc[1], mode='2'+act_mode), *[B.conv(nc[1], nc[1], mode='C'+act_mode) for _ in range(nb)])\n        self.m_up1 = B.sequential(upsample_block(nc[1], nc[0], mode='2'+act_mode), *[B.conv(nc[0], nc[0], mode='C'+act_mode) for _ in range(nb)])\n\n        self.m_tail = B.conv(nc[0], out_nc, bias=True, mode='C')\n\n    def forward(self, x0):\n\n        x1 = self.m_head(x0)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body(x4)\n        x = self.m_up3(x+x4)\n        x = self.m_up2(x+x3)\n        x = self.m_up1(x+x2)\n        x = self.m_tail(x+x1) + x0\n\n        \n        return x\n\n\nclass UNetRes(nn.Module):\n    def __init__(self, in_nc=1, out_nc=1, nc=[64, 128, 256, 512], nb=4, act_mode='R', downsample_mode='strideconv', upsample_mode='convtranspose'):\n        super(UNetRes, self).__init__()\n\n        self.m_head = B.conv(in_nc, nc[0], bias=False, mode='C')\n\n        # downsample\n        if downsample_mode == 'avgpool':\n            downsample_block = B.downsample_avgpool\n        elif downsample_mode == 'maxpool':\n            downsample_block = B.downsample_maxpool\n        elif downsample_mode == 'strideconv':\n            downsample_block = B.downsample_strideconv\n        else:\n            raise NotImplementedError('downsample mode [{:s}] is not found'.format(downsample_mode))\n\n        self.m_down1 = B.sequential(*[B.ResBlock(nc[0], nc[0], bias=False, mode='C'+act_mode+'C') for _ in range(nb)], downsample_block(nc[0], nc[1], bias=False, mode='2'))\n        self.m_down2 = B.sequential(*[B.ResBlock(nc[1], nc[1], bias=False, mode='C'+act_mode+'C') for _ in range(nb)], downsample_block(nc[1], nc[2], bias=False, mode='2'))\n        self.m_down3 = B.sequential(*[B.ResBlock(nc[2], nc[2], bias=False, mode='C'+act_mode+'C') for _ in range(nb)], downsample_block(nc[2], nc[3], bias=False, mode='2'))\n\n        self.m_body  = B.sequential(*[B.ResBlock(nc[3], nc[3], bias=False, mode='C'+act_mode+'C') for _ in range(nb)])\n\n        # upsample\n        if upsample_mode == 'upconv':\n            upsample_block = B.upsample_upconv\n        elif upsample_mode == 'pixelshuffle':\n            upsample_block = B.upsample_pixelshuffle\n        elif upsample_mode == 'convtranspose':\n            upsample_block = B.upsample_convtranspose\n        else:\n            raise NotImplementedError('upsample mode [{:s}] is not found'.format(upsample_mode))\n\n        self.m_up3 = B.sequential(upsample_block(nc[3], nc[2], bias=False, mode='2'), *[B.ResBlock(nc[2], nc[2], bias=False, mode='C'+act_mode+'C') for _ in range(nb)])\n        self.m_up2 = B.sequential(upsample_block(nc[2], nc[1], bias=False, mode='2'), *[B.ResBlock(nc[1], nc[1], bias=False, mode='C'+act_mode+'C') for _ in range(nb)])\n        self.m_up1 = B.sequential(upsample_block(nc[1], nc[0], bias=False, mode='2'), *[B.ResBlock(nc[0], nc[0], bias=False, mode='C'+act_mode+'C') for _ in range(nb)])\n\n        self.m_tail = B.conv(nc[0], out_nc, bias=False, mode='C')\n\n    def forward(self, x0):\n        x1 = self.m_head(x0)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body(x4)\n        x = self.m_up3(x+x4)\n        x = self.m_up2(x+x3)\n        x = self.m_up1(x+x2)\n        x = self.m_tail(x+x1)\n\n        return x\n\n\nclass ResUNet(nn.Module):\n    def __init__(self, in_nc=1, out_nc=1, nc=[64, 128, 256, 512], nb=4, act_mode='L', downsample_mode='strideconv', upsample_mode='convtranspose'):\n        super(ResUNet, self).__init__()\n\n        self.m_head = B.conv(in_nc, nc[0], bias=False, mode='C')\n\n        # downsample\n        if downsample_mode == 'avgpool':\n            downsample_block = B.downsample_avgpool\n        elif downsample_mode == 'maxpool':\n            downsample_block = B.downsample_maxpool\n        elif downsample_mode == 'strideconv':\n            downsample_block = B.downsample_strideconv\n        else:\n            raise NotImplementedError('downsample mode [{:s}] is not found'.format(downsample_mode))\n\n        self.m_down1 = B.sequential(*[B.IMDBlock(nc[0], nc[0], bias=False, mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[0], nc[1], bias=False, mode='2'))\n        self.m_down2 = B.sequential(*[B.IMDBlock(nc[1], nc[1], bias=False, mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[1], nc[2], bias=False, mode='2'))\n        self.m_down3 = B.sequential(*[B.IMDBlock(nc[2], nc[2], bias=False, mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[2], nc[3], bias=False, mode='2'))\n\n        self.m_body  = B.sequential(*[B.IMDBlock(nc[3], nc[3], bias=False, mode='C'+act_mode) for _ in range(nb)])\n\n        # upsample\n        if upsample_mode == 'upconv':\n            upsample_block = B.upsample_upconv\n        elif upsample_mode == 'pixelshuffle':\n            upsample_block = B.upsample_pixelshuffle\n        elif upsample_mode == 'convtranspose':\n            upsample_block = B.upsample_convtranspose\n        else:\n            raise NotImplementedError('upsample mode [{:s}] is not found'.format(upsample_mode))\n\n        self.m_up3 = B.sequential(upsample_block(nc[3], nc[2], bias=False, mode='2'), *[B.IMDBlock(nc[2], nc[2], bias=False, mode='C'+act_mode) for _ in range(nb)])\n        self.m_up2 = B.sequential(upsample_block(nc[2], nc[1], bias=False, mode='2'), *[B.IMDBlock(nc[1], nc[1], bias=False, mode='C'+act_mode) for _ in range(nb)])\n        self.m_up1 = B.sequential(upsample_block(nc[1], nc[0], bias=False, mode='2'), *[B.IMDBlock(nc[0], nc[0], bias=False, mode='C'+act_mode) for _ in range(nb)])\n\n        self.m_tail = B.conv(nc[0], out_nc, bias=False, mode='C')\n\n    def forward(self, x):\n        \n        h, w = x.size()[-2:]\n        paddingBottom = int(np.ceil(h/8)*8-h)\n        paddingRight = int(np.ceil(w/8)*8-w)\n        x = nn.ReplicationPad2d((0, paddingRight, 0, paddingBottom))(x)\n\n        x1 = self.m_head(x)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body(x4)\n        x = self.m_up3(x+x4)\n        x = self.m_up2(x+x3)\n        x = self.m_up1(x+x2)\n        x = self.m_tail(x+x1)\n        x = x[..., :h, :w]\n\n        return x\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nclass UNetResSubP(nn.Module):\n    def __init__(self, in_nc=1, out_nc=1, nc=[64, 128, 256, 512], nb=2, act_mode='R', downsample_mode='strideconv', upsample_mode='convtranspose'):\n        super(UNetResSubP, self).__init__()\n        sf = 2\n        self.m_ps_down = B.PixelUnShuffle(sf)\n        self.m_ps_up = nn.PixelShuffle(sf)\n        self.m_head = B.conv(in_nc*sf*sf, nc[0], mode='C'+act_mode[-1])\n\n        # downsample\n        if downsample_mode == 'avgpool':\n            downsample_block = B.downsample_avgpool\n        elif downsample_mode == 'maxpool':\n            downsample_block = B.downsample_maxpool\n        elif downsample_mode == 'strideconv':\n            downsample_block = B.downsample_strideconv\n        else:\n            raise NotImplementedError('downsample mode [{:s}] is not found'.format(downsample_mode))\n\n        self.m_down1 = B.sequential(*[B.ResBlock(nc[0], nc[0], mode='C'+act_mode+'C') for _ in range(nb)], downsample_block(nc[0], nc[1], mode='2'+act_mode))\n        self.m_down2 = B.sequential(*[B.ResBlock(nc[1], nc[1], mode='C'+act_mode+'C') for _ in range(nb)], downsample_block(nc[1], nc[2], mode='2'+act_mode))\n        self.m_down3 = B.sequential(*[B.ResBlock(nc[2], nc[2], mode='C'+act_mode+'C') for _ in range(nb)], downsample_block(nc[2], nc[3], mode='2'+act_mode))\n\n        self.m_body  = B.sequential(*[B.ResBlock(nc[3], nc[3], mode='C'+act_mode+'C') for _ in range(nb+1)])\n\n        # upsample\n        if upsample_mode == 'upconv':\n            upsample_block = B.upsample_upconv\n        elif upsample_mode == 'pixelshuffle':\n            upsample_block = B.upsample_pixelshuffle\n        elif upsample_mode == 'convtranspose':\n            upsample_block = B.upsample_convtranspose\n        else:\n            raise NotImplementedError('upsample mode [{:s}] is not found'.format(upsample_mode))\n\n        self.m_up3 = B.sequential(upsample_block(nc[3], nc[2], mode='2'+act_mode), *[B.ResBlock(nc[2], nc[2], mode='C'+act_mode+'C') for _ in range(nb)])\n        self.m_up2 = B.sequential(upsample_block(nc[2], nc[1], mode='2'+act_mode), *[B.ResBlock(nc[1], nc[1], mode='C'+act_mode+'C') for _ in range(nb)])\n        self.m_up1 = B.sequential(upsample_block(nc[1], nc[0], mode='2'+act_mode), *[B.ResBlock(nc[0], nc[0], mode='C'+act_mode+'C') for _ in range(nb)])\n\n        self.m_tail = B.conv(nc[0], out_nc*sf*sf, bias=False, mode='C')\n\n    def forward(self, x0):\n        x0_d = self.m_ps_down(x0)\n        x1 = self.m_head(x0_d)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body(x4)\n        x = self.m_up3(x+x4)\n        x = self.m_up2(x+x3)\n        x = self.m_up1(x+x2)\n        x = self.m_tail(x+x1)\n        x = self.m_ps_up(x) + x0\n\n        return x\n\n\nclass UNetPlus(nn.Module):\n    def __init__(self, in_nc=3, out_nc=3, nc=[64, 128, 256, 512], nb=1, act_mode='R', downsample_mode='strideconv', upsample_mode='convtranspose'):\n        super(UNetPlus, self).__init__()\n\n        self.m_head = B.conv(in_nc, nc[0], mode='C')\n\n        # downsample\n        if downsample_mode == 'avgpool':\n            downsample_block = B.downsample_avgpool\n        elif downsample_mode == 'maxpool':\n            downsample_block = B.downsample_maxpool\n        elif downsample_mode == 'strideconv':\n            downsample_block = B.downsample_strideconv\n        else:\n            raise NotImplementedError('downsample mode [{:s}] is not found'.format(downsample_mode))\n\n        self.m_down1 = B.sequential(*[B.conv(nc[0], nc[0], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[0], nc[1], mode='2'+act_mode[1]))\n        self.m_down2 = B.sequential(*[B.conv(nc[1], nc[1], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[1], nc[2], mode='2'+act_mode[1]))\n        self.m_down3 = B.sequential(*[B.conv(nc[2], nc[2], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[2], nc[3], mode='2'+act_mode[1]))\n\n        self.m_body  = B.sequential(*[B.conv(nc[3], nc[3], mode='C'+act_mode) for _ in range(nb+1)])\n\n        # upsample\n        if upsample_mode == 'upconv':\n            upsample_block = B.upsample_upconv\n        elif upsample_mode == 'pixelshuffle':\n            upsample_block = B.upsample_pixelshuffle\n        elif upsample_mode == 'convtranspose':\n            upsample_block = B.upsample_convtranspose\n        else:\n            raise NotImplementedError('upsample mode [{:s}] is not found'.format(upsample_mode))\n\n        self.m_up3 = B.sequential(upsample_block(nc[3], nc[2], mode='2'+act_mode), *[B.conv(nc[2], nc[2], mode='C'+act_mode) for _ in range(nb-1)], B.conv(nc[2], nc[2], mode='C'+act_mode[1]))\n        self.m_up2 = B.sequential(upsample_block(nc[2], nc[1], mode='2'+act_mode), *[B.conv(nc[1], nc[1], mode='C'+act_mode) for _ in range(nb-1)], B.conv(nc[1], nc[1], mode='C'+act_mode[1]))\n        self.m_up1 = B.sequential(upsample_block(nc[1], nc[0], mode='2'+act_mode), *[B.conv(nc[0], nc[0], mode='C'+act_mode) for _ in range(nb-1)], B.conv(nc[0], nc[0], mode='C'+act_mode[1]))\n\n        self.m_tail = B.conv(nc[0], out_nc, mode='C')\n\n    def forward(self, x0):\n        x1 = self.m_head(x0)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body(x4)\n        x = self.m_up3(x+x4)\n        x = self.m_up2(x+x3)\n        x = self.m_up1(x+x2)\n        x = self.m_tail(x+x1) + x0\n        return x\n\n'''\n# ====================\n# nonlocalunet\n# ====================\n'''\n\nclass NonLocalUNet(nn.Module):\n    def __init__(self, in_nc=3, out_nc=3, nc=[64,128,256,512], nb=1, act_mode='R', downsample_mode='strideconv', upsample_mode='convtranspose'):\n        super(NonLocalUNet, self).__init__()\n\n        down_nonlocal = B.NonLocalBlock2D(nc[2], kernel_size=1, stride=1, padding=0, bias=True, act_mode='B', downsample=False, downsample_mode='strideconv')\n        up_nonlocal = B.NonLocalBlock2D(nc[2], kernel_size=1, stride=1, padding=0, bias=True, act_mode='B', downsample=False, downsample_mode='strideconv')\n\n        self.m_head = B.conv(in_nc, nc[0], mode='C'+act_mode[-1])\n\n        # downsample\n        if downsample_mode == 'avgpool':\n            downsample_block = B.downsample_avgpool\n        elif downsample_mode == 'maxpool':\n            downsample_block = B.downsample_maxpool\n        elif downsample_mode == 'strideconv':\n            downsample_block = B.downsample_strideconv\n        else:\n            raise NotImplementedError('downsample mode [{:s}] is not found'.format(downsample_mode))\n\n\n        self.m_down1 = B.sequential(*[B.conv(nc[0], nc[0], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[0], nc[1], mode='2'+act_mode))\n        self.m_down2 = B.sequential(*[B.conv(nc[1], nc[1], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[1], nc[2], mode='2'+act_mode))\n        self.m_down3 = B.sequential(down_nonlocal, *[B.conv(nc[2], nc[2], mode='C'+act_mode) for _ in range(nb)], downsample_block(nc[2], nc[3], mode='2'+act_mode))\n\n        self.m_body  = B.sequential(*[B.conv(nc[3], nc[3], mode='C'+act_mode) for _ in range(nb+1)])\n\n        # upsample\n        if upsample_mode == 'upconv':\n            upsample_block = B.upsample_upconv\n        elif upsample_mode == 'pixelshuffle':\n            upsample_block = B.upsample_pixelshuffle\n        elif upsample_mode == 'convtranspose':\n            upsample_block = B.upsample_convtranspose\n        else:\n            raise NotImplementedError('upsample mode [{:s}] is not found'.format(upsample_mode))\n\n\n        self.m_up3 = B.sequential(upsample_block(nc[3], nc[2], mode='2'+act_mode), *[B.conv(nc[2], nc[2], mode='C'+act_mode) for _ in range(nb)], up_nonlocal)\n        self.m_up2 = B.sequential(upsample_block(nc[2], nc[1], mode='2'+act_mode), *[B.conv(nc[1], nc[1], mode='C'+act_mode) for _ in range(nb)])\n        self.m_up1 = B.sequential(upsample_block(nc[1], nc[0], mode='2'+act_mode), *[B.conv(nc[0], nc[0], mode='C'+act_mode) for _ in range(nb)])\n\n        self.m_tail = B.conv(nc[0], out_nc, mode='C')\n\n    def forward(self, x0):\n        x1 = self.m_head(x0)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body(x4)\n        x = self.m_up3(x+x4)\n        x = self.m_up2(x+x3)\n        x = self.m_up1(x+x2)\n        x = self.m_tail(x+x1) + x0\n        return x\n\n\nif __name__ == '__main__':\n    x = torch.rand(1,3,256,256)\n#    net = UNet(act_mode='BR')\n    net = NonLocalUNet()\n    net.eval()\n    with torch.no_grad():\n        y = net(x)\n    y.size()\n\n", 'utils/utils_pnp.py': "# -*- coding: utf-8 -*-\r\nimport numpy as np\r\n\r\n\r\n'''\r\nmodified by Kai Zhang (github: https://github.com/cszn)\r\n03/03/2019\r\n'''\r\n\r\n\r\n# --------------------------------\r\n# get rho and sigma\r\n# --------------------------------\r\n#def get_rho_sigma(sigma=2.55/255, iter_num=15, modelSigma1=49.0, modelSigma2=2.55):\r\n#    '''\r\n#    One can change the sigma to implicitly change the trade-off parameter\r\n#    between fidelity term and prior term\r\n#    '''\r\n#    modelSigmaS = np.logspace(np.log10(modelSigma1), np.log10(modelSigma2), iter_num).astype(np.float32)\r\n#    sigmas = modelSigmaS/255.\r\n#    rhos = list(map(lambda x: 0.23*(sigma**2)/(x**2), sigmas))\r\n#    return rhos, sigmas\r\n\r\n# --------------------------------\r\n# get rho and sigma\r\n# --------------------------------\r\ndef get_rho_sigma(sigma=2.55/255, iter_num=15, modelSigma1=49.0, modelSigma2=2.55, w=1.0):\r\n    '''\r\n    One can change the sigma to implicitly change the trade-off parameter\r\n    between fidelity term and prior term\r\n    '''\r\n    modelSigmaS = np.logspace(np.log10(modelSigma1), np.log10(modelSigma2), iter_num).astype(np.float32)\r\n    modelSigmaS_lin = np.linspace(modelSigma1, modelSigma2, iter_num).astype(np.float32)\r\n    sigmas = (modelSigmaS*w+modelSigmaS_lin*(1-w))/255.\r\n    rhos = list(map(lambda x: 0.23*(sigma**2)/(x**2), sigmas))\r\n    return rhos, sigmas\r\n\r\n\r\ndef get_rho_sigma1(sigma=2.55/255, iter_num=15, modelSigma1=49.0, modelSigma2=2.55, lamda=3.0):\r\n    '''\r\n    One can change the sigma to implicitly change the trade-off parameter\r\n    between fidelity term and prior term\r\n    '''\r\n    modelSigmaS = np.logspace(np.log10(modelSigma1), np.log10(modelSigma2), iter_num).astype(np.float32)\r\n    sigmas = modelSigmaS/255.\r\n    rhos = list(map(lambda x: (sigma**2)/(x**2)/lamda, sigmas))\r\n    return rhos, sigmas\r\n\r\n\r\nif __name__ == '__main__':\r\n    rhos, sigmas = get_rho_sigma(sigma=2.55/255, iter_num=30, modelSigma2=2.55)\r\n    print(rhos)\r\n    print(sigmas*255)\r\n"}, 'fbcnn': {'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'data/dataset_jpeg.py': 'import random\r\nimport numpy as np\r\nimport torch\r\nimport torch.utils.data as data\r\nimport utils.utils_image as util\r\nimport cv2\r\n\r\nclass DatasetJPEG(data.Dataset):\r\n\r\n    def __init__(self, opt):\r\n        super(DatasetJPEG, self).__init__()\r\n        self.opt = opt\r\n        self.n_channels = opt[\'n_channels\'] if opt[\'n_channels\'] else 3\r\n        self.patch_size = self.opt[\'H_size\'] if opt[\'H_size\'] else 64\r\n\r\n        # -------------------------------------\r\n        # get the path of H, return None if input is None\r\n        # -------------------------------------\r\n        self.paths_H = util.get_image_paths(opt[\'dataroot_H\'])\r\n\r\n    def __getitem__(self, index):\r\n        # -------------------------------------\r\n        # get H image\r\n        # -------------------------------------\r\n        H_path = self.paths_H[index]\r\n        img_H = util.imread_uint(H_path, self.n_channels)\r\n\r\n        L_path = H_path\r\n\r\n        if self.opt[\'phase\'] == \'train\':\r\n            """\r\n            # --------------------------------\r\n            # get L/H/M patch pairs\r\n            # --------------------------------\r\n            """\r\n            H, W = img_H.shape[:2]\r\n\r\n            # ---------------------------------\r\n            # randomly crop the patch\r\n            # ---------------------------------\r\n            self.patch_size_plus8 = self.patch_size+8\r\n            # ---------------------------------\r\n            # randomly crop the patch\r\n            # ---------------------------------\r\n            rnd_h = random.randint(0, max(0, H - self.patch_size_plus8))\r\n            rnd_w = random.randint(0, max(0, W - self.patch_size_plus8))\r\n            patch_H = img_H[rnd_h:rnd_h + self.patch_size_plus8, rnd_w:rnd_w + self.patch_size_plus8, :]\r\n\r\n            # ---------------------------------\r\n            # augmentation - flip, rotate\r\n            # ---------------------------------\r\n            mode = random.randint(0, 7)\r\n            patch_H = util.augment_img(patch_H, mode=mode)\r\n#            if random.random() > 0.5:\r\n#                patch_H = self.jitter(util.uint2tensor4(patch_H))\r\n#                patch_H = util.tensor2uint(patch_H)\r\n\r\n            # ---------------------------------\r\n            # HWC to CHW, numpy(uint) to tensor\r\n            # ---------------------------------\r\n#            img_H = util.uint2tensor3(patch_H)\r\n#            img_L = img_H.clone()\r\n            img_L = patch_H.copy()\r\n            img_H = patch_H.copy()\r\n\r\n            # ---------------------------------\r\n            # get noise level\r\n            # ---------------------------------\r\n\r\n            if random.random() > 0.75:\r\n                quality_factor = random.randint(8, 96)\r\n            else:\r\n                quality_factor = random.choice([10,20,30,40,50,60])\r\n\r\n            noise_level = (100-quality_factor)/100.0\r\n            img_L = cv2.cvtColor(img_L, cv2.COLOR_RGB2BGR)\r\n            result, encimg = cv2.imencode(\'.jpg\', img_L, [int(cv2.IMWRITE_JPEG_QUALITY), quality_factor])\r\n            img_L = cv2.imdecode(encimg, 1)\r\n            img_L = cv2.cvtColor(img_L, cv2.COLOR_BGR2RGB)\r\n\r\n            noise_level = torch.FloatTensor([noise_level])\r\n\r\n            H, W = img_H.shape[:2]\r\n            if random.random() > 0.5:\r\n                rnd_h = random.randint(0, max(0, H - self.patch_size))\r\n                rnd_w = random.randint(0, max(0, W - self.patch_size))\r\n            else:\r\n                rnd_h = 0\r\n                rnd_w = 0\r\n            img_H = img_H[rnd_h:rnd_h + self.patch_size, rnd_w:rnd_w + self.patch_size]\r\n            img_L = img_L[rnd_h:rnd_h + self.patch_size, rnd_w:rnd_w + self.patch_size]\r\n\r\n\r\n            img_L, img_H = util.uint2tensor3(img_L), util.uint2tensor3(img_H)\r\n\r\n            # ---------------------------------\r\n            # add noise\r\n            # ---------------------------------\r\n#            noise = torch.randn(img_L.size()).mul_(noise_level).float()\r\n#            img_L.add_(noise)\r\n\r\n        else:\r\n            """\r\n            # --------------------------------\r\n            # get L/H/M image pairs\r\n            # --------------------------------\r\n            """\r\n            img_L = img_H.copy()\r\n            \r\n            \r\n            quality_factor = 10\r\n            noise_level = (100-quality_factor)/100.0\r\n            img_L = cv2.cvtColor(img_L, cv2.COLOR_RGB2BGR)\r\n            result, encimg = cv2.imencode(\'.jpg\', img_L, [int(cv2.IMWRITE_JPEG_QUALITY), quality_factor])\r\n            img_L = cv2.imdecode(encimg, 1)\r\n            img_L = cv2.cvtColor(img_L, cv2.COLOR_BGR2RGB)\r\n\r\n            noise_level = torch.FloatTensor([noise_level])\r\n\r\n            noise_level_map = torch.ones((1, img_L.shape[0], img_L.shape[0])).mul_(noise_level).float()\r\n            img_L, img_H = util.uint2tensor3(img_L), util.uint2tensor3(img_H)\r\n\r\n\r\n        return {\'L\': img_L, \'H\': img_H, \'qf\': noise_level, \'L_path\': L_path, \'H_path\': H_path}\r\n\r\n    def __len__(self):\r\n        return len(self.paths_H)\r\n', 'main_test_fbcnn_color.py': "import os.path\nimport logging\nimport numpy as np\nfrom datetime import datetime\nfrom collections import OrderedDict\nimport torch\nimport cv2\nfrom utils import utils_logger\nfrom utils import utils_image as util\nimport requests\n\ndef main():\n\n    quality_factor_list = [10, 20, 30, 40, 50, 60, 70, 80, 90]\n    testset_name = 'LIVE1_color'  # 'LIVE1_color' 'BSDS500_color' 'ICB'\n    n_channels = 3            # set 1 for grayscale image, set 3 for color image\n    model_name = 'fbcnn_color.pth'\n    nc = [64,128,256,512]\n    nb = 4\n    show_img = False                 # default: False\n    testsets = 'testsets'    \n    results = 'test_results'     \n\n    for quality_factor in quality_factor_list:\n\n        result_name = testset_name + '_' + model_name[:-4]\n        H_path = os.path.join(testsets, testset_name)\n        E_path = os.path.join(results, result_name, str(quality_factor))   # E_path, for Estimated images\n        util.mkdir(E_path)\n\n        model_pool = 'model_zoo'  # fixed\n        model_path = os.path.join(model_pool, model_name)\n        if os.path.exists(model_path):\n            print(f'loading model from {model_path}')\n        else:\n            os.makedirs(os.path.dirname(model_path), exist_ok=True)\n            url = 'https://github.com/jiaxi-jiang/FBCNN/releases/download/v1.0/{}'.format(os.path.basename(model_path))\n            r = requests.get(url, allow_redirects=True)\n            print(f'downloading model {model_path}')\n            open(model_path, 'wb').write(r.content)\n\n        logger_name = result_name + '_qf_' + str(quality_factor)\n        utils_logger.logger_info(logger_name, log_path=os.path.join(E_path, logger_name+'.log'))\n        logger = logging.getLogger(logger_name)\n        logger.info('--------------- quality factor: {:d} ---------------'.format(quality_factor))\n\n        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n        border = 0\n\n\n        # ----------------------------------------\n        # load model\n        # ----------------------------------------\n\n        from models.network_fbcnn import FBCNN as net\n        model = net(in_nc=n_channels, out_nc=n_channels, nc=nc, nb=nb, act_mode='R')\n        model.load_state_dict(torch.load(model_path), strict=True)\n        model.eval()\n        for k, v in model.named_parameters():\n         v.requires_grad = False\n        model = model.to(device)\n        logger.info('Model path: {:s}'.format(model_path))\n\n        test_results = OrderedDict()\n        test_results['psnr'] = []\n        test_results['ssim'] = []\n        test_results['psnrb'] = []\n\n        H_paths = util.get_image_paths(H_path)\n        for idx, img in enumerate(H_paths):\n\n            # ------------------------------------\n            # (1) img_L\n            # ------------------------------------\n            img_name, ext = os.path.splitext(os.path.basename(img))\n            logger.info('{:->4d}--> {:>10s}'.format(idx+1, img_name+ext))\n            img_L = util.imread_uint(img, n_channels=n_channels)\n\n            if n_channels == 3:\n             img_L = cv2.cvtColor(img_L, cv2.COLOR_RGB2BGR)                \n            _, encimg = cv2.imencode('.jpg', img_L, [int(cv2.IMWRITE_JPEG_QUALITY), quality_factor])\n            img_L = cv2.imdecode(encimg, 0) if n_channels == 1 else cv2.imdecode(encimg, 3)\n            if n_channels == 3:\n             img_L = cv2.cvtColor(img_L, cv2.COLOR_BGR2RGB)               \n            img_L = util.uint2tensor4(img_L)\n            img_L = img_L.to(device)\n\n            # ------------------------------------\n            # (2) img_E\n            # ------------------------------------\n\n            #img_E,QF = model(img_L, torch.tensor([[0.6]]))      \n            img_E,QF = model(img_L)\n            QF = 1 - QF\n            img_E = util.tensor2single(img_E)\n            img_E = util.single2uint(img_E)\n            img_H = util.imread_uint(H_paths[idx], n_channels=n_channels).squeeze()\n            # --------------------------------\n            # PSNR and SSIM, PSNRB\n            # --------------------------------\n\n            psnr = util.calculate_psnr(img_E, img_H, border=border)\n            ssim = util.calculate_ssim(img_E, img_H, border=border)\n            psnrb = util.calculate_psnrb(img_H, img_E, border=border)\n            test_results['psnr'].append(psnr)\n            test_results['ssim'].append(ssim)\n            test_results['psnrb'].append(psnrb)\n            logger.info('{:s} - PSNR: {:.2f} dB; SSIM: {:.3f}; PSNRB: {:.2f} dB.'.format(img_name+ext, psnr, ssim, psnrb))\n            logger.info('predicted quality factor: {:d}'.format(round(float(QF*100))))\n\n            util.imshow(np.concatenate([img_E, img_H], axis=1), title='Recovered / Ground-truth') if show_img else None\n            util.imsave(img_E, os.path.join(E_path, img_name+'.png'))\n\n        ave_psnr = sum(test_results['psnr']) / len(test_results['psnr'])\n        ave_ssim = sum(test_results['ssim']) / len(test_results['ssim'])\n        ave_psnrb = sum(test_results['psnrb']) / len(test_results['psnrb'])\n        logger.info(\n             'Average PSNR/SSIM/PSNRB - {} -: {:.2f}$\\\\vert${:.4f}$\\\\vert${:.2f}.'.format(result_name+'_'+str(quality_factor), ave_psnr, ave_ssim, ave_psnrb))\n\n\nif __name__ == '__main__':\n    main()\n", 'models/network_fbcnn.py': 'from collections import OrderedDict\nimport torch\nimport torch.nn as nn\nimport numpy as np\nimport torch.nn.functional as F\nimport torchvision.models as models\n\n\'\'\'\n# --------------------------------------------\n# Advanced nn.Sequential\n# https://github.com/xinntao/BasicSR\n# --------------------------------------------\n\'\'\'\n\n\ndef sequential(*args):\n    """Advanced nn.Sequential.\n\n    Args:\n        nn.Sequential, nn.Module\n\n    Returns:\n        nn.Sequential\n    """\n    if len(args) == 1:\n        if isinstance(args[0], OrderedDict):\n            raise NotImplementedError(\'sequential does not support OrderedDict input.\')\n        return args[0]  # No sequential is needed.\n    modules = []\n    for module in args:\n        if isinstance(module, nn.Sequential):\n            for submodule in module.children():\n                modules.append(submodule)\n        elif isinstance(module, nn.Module):\n            modules.append(module)\n    return nn.Sequential(*modules)\n\n# --------------------------------------------\n# return nn.Sequantial of (Conv + BN + ReLU)\n# --------------------------------------------\ndef conv(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CBR\', negative_slope=0.2):\n    L = []\n    for t in mode:\n        if t == \'C\':\n            L.append(nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias))\n        elif t == \'T\':\n            L.append(nn.ConvTranspose2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=bias))\n        elif t == \'B\':\n            L.append(nn.BatchNorm2d(out_channels, momentum=0.9, eps=1e-04, affine=True))\n        elif t == \'I\':\n            L.append(nn.InstanceNorm2d(out_channels, affine=True))\n        elif t == \'R\':\n            L.append(nn.ReLU(inplace=True))\n        elif t == \'r\':\n            L.append(nn.ReLU(inplace=False))\n        elif t == \'L\':\n            L.append(nn.LeakyReLU(negative_slope=negative_slope, inplace=True))\n        elif t == \'l\':\n            L.append(nn.LeakyReLU(negative_slope=negative_slope, inplace=False))\n        elif t == \'2\':\n            L.append(nn.PixelShuffle(upscale_factor=2))\n        elif t == \'3\':\n            L.append(nn.PixelShuffle(upscale_factor=3))\n        elif t == \'4\':\n            L.append(nn.PixelShuffle(upscale_factor=4))\n        elif t == \'U\':\n            L.append(nn.Upsample(scale_factor=2, mode=\'nearest\'))\n        elif t == \'u\':\n            L.append(nn.Upsample(scale_factor=3, mode=\'nearest\'))\n        elif t == \'v\':\n            L.append(nn.Upsample(scale_factor=4, mode=\'nearest\'))\n        elif t == \'M\':\n            L.append(nn.MaxPool2d(kernel_size=kernel_size, stride=stride, padding=0))\n        elif t == \'A\':\n            L.append(nn.AvgPool2d(kernel_size=kernel_size, stride=stride, padding=0))\n        else:\n            raise NotImplementedError(\'Undefined type: \'.format(t))\n    return sequential(*L)\n\n# --------------------------------------------\n# Res Block: x + conv(relu(conv(x)))\n# --------------------------------------------\nclass ResBlock(nn.Module):\n    def __init__(self, in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CRC\', negative_slope=0.2):\n        super(ResBlock, self).__init__()\n\n        assert in_channels == out_channels, \'Only support in_channels==out_channels.\'\n        if mode[0] in [\'R\', \'L\']:\n            mode = mode[0].lower() + mode[1:]\n\n        self.res = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n\n    def forward(self, x):\n        res = self.res(x)\n        return x + res\n\n# --------------------------------------------\n# conv + subp (+ relu)\n# --------------------------------------------\ndef upsample_pixelshuffle(in_channels=64, out_channels=3, kernel_size=3, stride=1, padding=1, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR.\'\n    up1 = conv(in_channels, out_channels * (int(mode[0]) ** 2), kernel_size, stride, padding, bias, mode=\'C\'+mode, negative_slope=negative_slope)\n    return up1\n\n\n# --------------------------------------------\n# nearest_upsample + conv (+ R)\n# --------------------------------------------\ndef upsample_upconv(in_channels=64, out_channels=3, kernel_size=3, stride=1, padding=1, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR\'\n    if mode[0] == \'2\':\n        uc = \'UC\'\n    elif mode[0] == \'3\':\n        uc = \'uC\'\n    elif mode[0] == \'4\':\n        uc = \'vC\'\n    mode = mode.replace(mode[0], uc)\n    up1 = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode=mode, negative_slope=negative_slope)\n    return up1\n\n\n# --------------------------------------------\n# convTranspose (+ relu)\n# --------------------------------------------\ndef upsample_convtranspose(in_channels=64, out_channels=3, kernel_size=2, stride=2, padding=0, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR.\'\n    kernel_size = int(mode[0])\n    stride = int(mode[0])\n    mode = mode.replace(mode[0], \'T\')\n    up1 = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n    return up1\n\n\n\'\'\'\n# --------------------------------------------\n# Downsampler\n# Kai Zhang, https://github.com/cszn/KAIR\n# --------------------------------------------\n# downsample_strideconv\n# downsample_maxpool\n# downsample_avgpool\n# --------------------------------------------\n\'\'\'\n\n\n# --------------------------------------------\n# strideconv (+ relu)\n# --------------------------------------------\ndef downsample_strideconv(in_channels=64, out_channels=64, kernel_size=2, stride=2, padding=0, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\', \'4\'], \'mode examples: 2, 2R, 2BR, 3, ..., 4BR.\'\n    kernel_size = int(mode[0])\n    stride = int(mode[0])\n    mode = mode.replace(mode[0], \'C\')\n    down1 = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n    return down1\n\n\n# --------------------------------------------\n# maxpooling + conv (+ relu)\n# --------------------------------------------\ndef downsample_maxpool(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=0, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\'], \'mode examples: 2, 2R, 2BR, 3, ..., 3BR.\'\n    kernel_size_pool = int(mode[0])\n    stride_pool = int(mode[0])\n    mode = mode.replace(mode[0], \'MC\')\n    pool = conv(kernel_size=kernel_size_pool, stride=stride_pool, mode=mode[0], negative_slope=negative_slope)\n    pool_tail = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode=mode[1:], negative_slope=negative_slope)\n    return sequential(pool, pool_tail)\n\n\n# --------------------------------------------\n# averagepooling + conv (+ relu)\n# --------------------------------------------\ndef downsample_avgpool(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'2R\', negative_slope=0.2):\n    assert len(mode)<4 and mode[0] in [\'2\', \'3\'], \'mode examples: 2, 2R, 2BR, 3, ..., 3BR.\'\n    kernel_size_pool = int(mode[0])\n    stride_pool = int(mode[0])\n    mode = mode.replace(mode[0], \'AC\')\n    pool = conv(kernel_size=kernel_size_pool, stride=stride_pool, mode=mode[0], negative_slope=negative_slope)\n    pool_tail = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode=mode[1:], negative_slope=negative_slope)\n    return sequential(pool, pool_tail)\n\n\n\nclass QFAttention(nn.Module):\n    def __init__(self, in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True, mode=\'CRC\', negative_slope=0.2):\n        super(QFAttention, self).__init__()\n\n        assert in_channels == out_channels, \'Only support in_channels==out_channels.\'\n        if mode[0] in [\'R\', \'L\']:\n            mode = mode[0].lower() + mode[1:]\n\n        self.res = conv(in_channels, out_channels, kernel_size, stride, padding, bias, mode, negative_slope)\n\n    def forward(self, x, gamma, beta):\n        gamma = gamma.unsqueeze(-1).unsqueeze(-1)\n        beta = beta.unsqueeze(-1).unsqueeze(-1)\n        res = (gamma)*self.res(x) + beta\n        return x + res\n\n\nclass FBCNN(nn.Module):\n    def __init__(self, in_nc=3, out_nc=3, nc=[64, 128, 256, 512], nb=4, act_mode=\'R\', downsample_mode=\'strideconv\',\n                 upsample_mode=\'convtranspose\'):\n        super(FBCNN, self).__init__()\n\n        self.m_head = conv(in_nc, nc[0], bias=True, mode=\'C\')\n        self.nb = nb\n        self.nc = nc\n        # downsample\n        if downsample_mode == \'avgpool\':\n            downsample_block = downsample_avgpool\n        elif downsample_mode == \'maxpool\':\n            downsample_block = downsample_maxpool\n        elif downsample_mode == \'strideconv\':\n            downsample_block = downsample_strideconv\n        else:\n            raise NotImplementedError(\'downsample mode [{:s}] is not found\'.format(downsample_mode))\n\n        self.m_down1 = sequential(\n            *[ResBlock(nc[0], nc[0], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)],\n            downsample_block(nc[0], nc[1], bias=True, mode=\'2\'))\n        self.m_down2 = sequential(\n            *[ResBlock(nc[1], nc[1], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)],\n            downsample_block(nc[1], nc[2], bias=True, mode=\'2\'))\n        self.m_down3 = sequential(\n            *[ResBlock(nc[2], nc[2], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)],\n            downsample_block(nc[2], nc[3], bias=True, mode=\'2\'))\n\n        self.m_body_encoder = sequential(\n            *[ResBlock(nc[3], nc[3], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)])\n\n        self.m_body_decoder = sequential(\n            *[ResBlock(nc[3], nc[3], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)])\n\n        # upsample\n        if upsample_mode == \'upconv\':\n            upsample_block = upsample_upconv\n        elif upsample_mode == \'pixelshuffle\':\n            upsample_block = upsample_pixelshuffle\n        elif upsample_mode == \'convtranspose\':\n            upsample_block = upsample_convtranspose\n        else:\n            raise NotImplementedError(\'upsample mode [{:s}] is not found\'.format(upsample_mode))\n\n        self.m_up3 = nn.ModuleList([upsample_block(nc[3], nc[2], bias=True, mode=\'2\'),\n                                  *[QFAttention(nc[2], nc[2], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)]])\n\n        self.m_up2 = nn.ModuleList([upsample_block(nc[2], nc[1], bias=True, mode=\'2\'),\n                                  *[QFAttention(nc[1], nc[1], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)]])\n\n        self.m_up1 = nn.ModuleList([upsample_block(nc[1], nc[0], bias=True, mode=\'2\'),\n                                  *[QFAttention(nc[0], nc[0], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)]])\n\n\n        self.m_tail = conv(nc[0], out_nc, bias=True, mode=\'C\')\n\n\n        self.qf_pred = sequential(*[ResBlock(nc[3], nc[3], bias=True, mode=\'C\' + act_mode + \'C\') for _ in range(nb)],\n                                  torch.nn.AdaptiveAvgPool2d((1,1)),\n                                  torch.nn.Flatten(),\n                                  torch.nn.Linear(512, 512), \n                                  nn.ReLU(),\n                                  torch.nn.Linear(512, 512),\n                                  nn.ReLU(),\n                                  torch.nn.Linear(512, 1),\n                                  nn.Sigmoid()\n                                )\n\n        self.qf_embed = sequential(torch.nn.Linear(1, 512),\n                                  nn.ReLU(),\n                                  torch.nn.Linear(512, 512),\n                                  nn.ReLU(),\n                                  torch.nn.Linear(512, 512),\n                                  nn.ReLU()\n                                )\n\n        self.to_gamma_3 = sequential(torch.nn.Linear(512, nc[2]),nn.Sigmoid())\n        self.to_beta_3 =  sequential(torch.nn.Linear(512, nc[2]),nn.Tanh())\n        self.to_gamma_2 = sequential(torch.nn.Linear(512, nc[1]),nn.Sigmoid())\n        self.to_beta_2 =  sequential(torch.nn.Linear(512, nc[1]),nn.Tanh())\n        self.to_gamma_1 = sequential(torch.nn.Linear(512, nc[0]),nn.Sigmoid())\n        self.to_beta_1 =  sequential(torch.nn.Linear(512, nc[0]),nn.Tanh())\n\n\n    def forward(self, x, qf_input=None):\n\n        h, w = x.size()[-2:]\n        paddingBottom = int(np.ceil(h / 8) * 8 - h)\n        paddingRight = int(np.ceil(w / 8) * 8 - w)\n        x = nn.ReplicationPad2d((0, paddingRight, 0, paddingBottom))(x)\n\n        x1 = self.m_head(x)\n        x2 = self.m_down1(x1)\n        x3 = self.m_down2(x2)\n        x4 = self.m_down3(x3)\n        x = self.m_body_encoder(x4)\n        qf = self.qf_pred(x)\n        x = self.m_body_decoder(x)\n        qf_embedding = self.qf_embed(qf_input) if qf_input is not None else self.qf_embed(qf)\n        gamma_3 = self.to_gamma_3(qf_embedding)\n        beta_3 = self.to_beta_3(qf_embedding)\n\n        gamma_2 = self.to_gamma_2(qf_embedding)\n        beta_2 = self.to_beta_2(qf_embedding)\n\n        gamma_1 = self.to_gamma_1(qf_embedding)\n        beta_1 = self.to_beta_1(qf_embedding)\n\n\n        x = x + x4\n        x = self.m_up3[0](x)\n        for i in range(self.nb):\n            x = self.m_up3[i+1](x, gamma_3,beta_3)\n\n        x = x + x3\n\n        x = self.m_up2[0](x)\n        for i in range(self.nb):\n            x = self.m_up2[i+1](x, gamma_2, beta_2)\n        x = x + x2\n\n        x = self.m_up1[0](x)\n        for i in range(self.nb):\n            x = self.m_up1[i+1](x, gamma_1, beta_1)\n\n        x = x + x1\n        x = self.m_tail(x)\n        x = x[..., :h, :w]\n\n        return x, qf\n\nif __name__ == "__main__":\n    x = torch.randn(1, 3, 96, 96)#.cuda()#.to(torch.device(\'cuda\'))\n    fbar=FBAR()\n    y,qf = fbar(x)\n    print(y.shape,qf.shape)\n', 'options/train_fbcnn_color.json': '{\n  "task": "FBCNN-Color"  //  root/task/images-models-options\n  , "model": "fbcnn" \n  , "gpu_ids": [0,1]\n  , "n_channels": 3  // broadcast to "datasets", 1 for grayscale, 3 for color\n  , "merge_bn": false               // BN for DnCNN\n  , "merge_bn_startpoint": 40000000  // merge BN after N iterations\n  , "path": {\n    "root": "deblocking"      \n    , "pretrained_netG": null      // path of pretrained model\n  }\n\n  , "datasets": {\n    "train": {\n      "name": "train_dataset"           // just name\n      , "dataset_type": "jpeg"         // "jpeg" | "jpeggray" | "jpeggraydouble"\n      , "dataroot_H": "../trainsets/Data_DIV2K_Flickr2K"// path of original training dataset\n      , "dataroot_L": null            // path of L training dataset\n      , "H_size": 96                  // patch size 40 | 64 | 96 | 128 | 192\n      , "dataloader_shuffle": true\n      , "dataloader_num_workers":16 \n      , "dataloader_batch_size": 128     // batch size 1 | 16 | 32 | 48 | 64 | 128\n    }, "test": {\n      "name": "test_dataset"            // just name\n      , "dataset_type": "jpeg"         // "jpeg" | "jpeggray" | "jpeggraydouble" \n      , "dataroot_H": "../testsets/Classic5/origin"  // path of H testing dataset\n      , "dataroot_L": "../testsets/Classic5/single/10"              // path of L testing dataset\n    }\n  }\n\n  , "netG": {\n    "net_type": "fbcnn"\n    , "in_nc": 3        // input channel number\n    , "out_nc": 3 // output channel number\n    , "nc": [64, 128, 256, 512] // number of channels\n    , "nb": 4          \n    , "act_mode": "BR"  // "BR" for BN+ReLU | "R" for ReLU\n    , "upsample_mode": "convtranspose"  // "pixelshuffle" | "convtranspose" | "upconv"\n    , "downsample_mode": "strideconv"   // "strideconv" | "avgpool" | "maxpool"\n\n    , "init_type": "orthogonal"         // "orthogonal" | "normal" | "uniform" | "xavier_normal" | "xavier_uniform" | "kaiming_normal" | "kaiming_uniform"\n    , "init_bn_type": "uniform"         // "uniform" | "constant"\n    , "init_gain": 0.2\n  }\n\n  , "train": {\n    "G_lossfn_type": "l1"               // "l1" preferred | "l2sum" | "l2" | "ssim"\n    , "G_lossfn_weight": 1.0            // default\n    , "QF_lossfn_type": "l1"\n    , "QF_lossfn_weight":0.001 //1.0\n\n    , "G_optimizer_type": "adam"        // fixed, adam is enough\n    , "G_optimizer_lr": 2e-5         // learning rate\n    , "G_optimizer_clipgrad": null      // unused\n\n    , "G_scheduler_type": "MultiStepLR" // "MultiStepLR" is enough\n    , "G_scheduler_milestones": [200000, 400000, 1200000, 1600000, 1800000]\n    , "G_scheduler_gamma": 0.5\n\n    , "G_regularizer_orthstep": null    // unused\n    , "G_regularizer_clipstep": null    // unused\n\n    , "checkpoint_test": 2500           // for testing 5000\n    , "checkpoint_save":  5000           // for saving model 5000\n    , "checkpoint_print": 500           // for print\n  }\n}\n\n'}}
DESIGN_FREEZE = "# JPEG-aware baseline 04: frozen development comparison\n\n17 September 2026. Written before new FBCNN reconstruction outcomes are inspected. This follows the observed Acquisition 03 JPEG failure and is not an independently preregistered study.\n\n## Question and scope\n\nDoes established blind JPEG preprocessing improve our fixed reconstruction baseline, and does heuristic operator sensitivity still add useful detail selection after that change?\n\n- Reuse exposed sources 0801–0804, native 576 x 576 centre crops and 512 x 512 interior. True periodic Gaussian blur 1.6 px, noise 2/255, seed 20260917 and source-specific noise are unchanged.\n- Derive two observations from the same noise draw: `quantized_8bit` and `jpeg_q75`, exactly as Acquisition 03, including Pillow quality 75, subsampling 0 and optimize=False. The uncompressed 8-bit condition measures any damage from unnecessary preprocessing. Full design: four sources x two stages = eight observations.\n- Local CPU validation is predeclared as source 0801 at both stages, with identical resolution, models and settings. Other sources are not silently substituted or resized. Colab defaults to the full four-source design.\n- Both pretrained families include DIV2K in their reported training data; exact checkpoint image manifests are unavailable. These images establish engineering behaviour only, never unseen-image performance. The four sources, not their patches or two stages, are the source replication units.\n\n## Fixed models and information\n\nUse official FBCNN colour weights from `jiaxi-jiang/FBCNN` release v1.0; pin source commit `54d1831927506b3247e2d4d245abb4f4dab1a1cd`, verify source and checkpoint digests, load all parameters strictly. Use the official test-time architecture (`act_mode='R'`) and automatic quality prediction; never supply the true Q75 value or manually choose quality after seeing references. The demo's unused torchvision import may be removed at load time, with source verification and an explicit adaptation record; network operations remain unchanged.\n\nSix main comparisons, all applied to both stages:\n\n1. Input observation.\n2. Nominal classical gradient inverse, lambda 0.05.\n3. Nominal DPIR-style reconstruction, unchanged eight iterations and prior settings.\n4. FBCNN alone.\n5. FBCNN followed by the same nominal classical inverse.\n6. FBCNN followed by the same nominal DPIR adapter.\n\nAll operational reconstructions receive decoded RGB pixels and fixed nominal blur/noise assumptions only. FBCNN carries a pretrained JPEG-specific prior; it receives no source identity, true blur, clean reference, codec bitstream or Q75 metadata. Its automatic inferred quality is logged, not calibrated as uncertainty. Apply one full-tensor pass and clip its output to [0,1] without uint8 rounding before downstream inversion. This composition is our declared development adapter, not an end-to-end deblurring model, a reproduction of the paper's benchmark, an exact JPEG likelihood, or a new method. The prior's MATLAB training codec and the repository demo's OpenCV encoding differ from our frozen Pillow pipeline; preserve and disclose the difference.\n\nKeep the inherited DRUNet checkpoint, analytic FFT update, eight iterations, 49-to-2 denoiser schedule, trade-off 0.23, internal geometric transforms, blur assumption 1.0 and noise assumption 2/255. No search over preprocessing strength, noise, blur grid or regularisation. The effective noise after FBCNN need not be Gaussian with the same variance: unchanged downstream settings isolate the preprocessing intervention but do not define an optimally tuned cascade.\n\n## Reliability comparison and compute\n\nFor both raw DPIR and FBCNN+DPIR, use three assumed blur widths (0.8, 1.0, 1.2) for the operator score and three whole-pipeline rotations (0, 90, 180 degrees) for the fixed-operator transformation score. Restore orientations before comparison. Reuse the nominal output. FBCNN preprocessing is shared across blur hypotheses, but rerun on each rotated observation for the whole-pipeline transformation comparator.\n\nScores retain the previous definitions: detail operator spread, RGB operator spread, detail transformation spread, original-measurement residual and reconstructed-image gradient. Add exact expected random selection and reference-error ranking as evaluation controls. The residual compares nominal reblurred output against the original supplied observation for both pipelines; it is not JPEG-consistent likelihood evidence. Low spread does not prove accuracy when candidate models share a missing acquisition stage.\n\nEach pipeline needs five eight-iteration DPIR reconstructions per observation. Both score ensembles use 24 DRUNet calls including their shared nominal prediction. FBCNN cost differs: one pass for the operator family and three for whole-pipeline transformations. Report actual elapsed time, model calls and memory; do not claim equal total compute. Per observation: 80 experiment DRUNet calls and three FBCNN calls. Full design: 640 + 24; one-source local validation: 160 + 6. Small loading/parity checks are recorded separately.\n\nUse high-pass `D(x)=x-G_1*x` before cropping, 16 x 16 patches, coverages 50%, 75%, 90%, 100%, and detail-RMSE exceedance tolerances 0.025, 0.05, 0.10. Report all patches and the reference-gradient top quartile separately. References define errors, texture strata and the oracle ranking only; operational scores never use them. Transformation spread is a limited heuristic image-only control, not a trained uncertainty estimator. This run cannot establish superiority over the latter or over unrun blind samplers.\n\n## Predeclared decisions\n\nPrimary reconstruction outcome: mean JPEG detail MSE across sources; retain pooled RGB PSNR and source-level changes as secondary outcomes. Compare FBCNN+DPIR separately against raw DPIR, raw classical and FBCNN+classical, retaining FBCNN-only and input results. A provisional reconstruction screen requires lower pooled JPEG detail MSE than all three inverse comparators and improvement over raw DPIR on at least three of the four sources. Report RGB reversals and uncompressed-condition losses without hiding them. Failure of this cascade is not failure of every acquisition-aware method.\n\nPrimary selection outcome: JPEG all-patch detail MSE at 50% retention on the FBCNN+DPIR nominal reconstruction. A provisional selection screen requires the detail operator score to beat transformation spread, original-measurement residual, gradient and expected random retention in pooled detail MSE, win on at least three of four sources against the best pooled operational control, and have no higher pooled bad-detail rate than any listed control at each of the three tolerances. RGB operator spread, 75/90% retention and textured strata are secondary, never substitutes chosen after seeing outcomes. Zero exceedance rates are uninformative, not proof of reliability.\n\nThese are strict descriptive development screens, not statistical significance, calibration, independent generalisation or novelty gates. With only one local source, return `not_assessed_full_design`; do not reinterpret it as a pass/fail of the four-source question. Even a full pass requires stronger uncertainty/baseline comparisons and source-separated evaluation after training-overlap checks. Do not build a residual network merely because this screen fails.\n\n## Evidence and export\n\nSave all main and ensemble reconstructions, supplied observations, intermediate FBCNN outputs, patch errors, score vectors, JPEG bytes, quality/risk/source comparisons, component and score costs, trajectories, environment, status and file hashes. Recompute numerical metrics and risks from saved arrays with an independent readback implementation. Source images and model weights remain external inputs.\n\nThe notebook exports a small summary ZIP and a complete raw archive. It automatically splits the complete archive into verified ZIP transfer parts of at most 96 MiB payload when needed; no separate packaging notebook or reconstruction rerun is required. The summary explicitly cannot substitute for raw-array verification.\n\nSources: [official FBCNN](https://github.com/jiaxi-jiang/FBCNN/tree/54d1831927506b3247e2d4d245abb4f4dab1a1cd), [FBCNN paper, Section 4.1](https://arxiv.org/pdf/2109.14573), [Learned 02 design](../learned_02/design_freeze.md), [Acquisition 03 design](../acquisition_03/design_freeze.md), [Lee/Jang assessment](../../docs/jang_paper_assessment_2026-09-17.md).\n"
ANCHORS = {'quality.csv': 'source_id,stage,stage_index,model,mse,psnr_db,detail_mse,bad_detail_rate_0.025,bad_detail_rate_0.05,bad_detail_rate_0.1\n0801,linear_float,0,observed,0.0005517382981011453,32.58266869084283,0.0001973661827290087,0.0703125,0.0009765625,0.0\n0801,linear_float,0,gradient_nominal,0.00040238448320535603,33.953587749407504,0.0001530425182053485,0.0537109375,0.0009765625,0.0\n0801,linear_float,0,drunet_denoise_only,0.0005081697899717747,32.939911567562,0.00015455096743861198,0.0615234375,0.0009765625,0.0\n0801,linear_float,0,dpir_nominal,0.00034826043643405256,34.580958600646426,0.00013528801732669296,0.0498046875,0.0,0.0\n0801,linear_float,0,dpir_oracle_blur,0.00016456677485542575,37.83657841988086,9.912671429833074e-05,0.015625,0.0,0.0\n0801,clipped_float,1,observed,0.0005517382981011453,32.58266869084283,0.0001973661827290087,0.0703125,0.0009765625,0.0\n0801,clipped_float,1,gradient_nominal,0.00040238085310318253,33.95362692935902,0.0001530410235936804,0.0537109375,0.0009765625,0.0\n0801,clipped_float,1,drunet_denoise_only,0.0005081702100005029,32.93990797789399,0.00015455102377037158,0.0615234375,0.0009765625,0.0\n0801,clipped_float,1,dpir_nominal,0.00034826015266307796,34.58096213938369,0.00013528781424007076,0.0498046875,0.0,0.0\n0801,clipped_float,1,dpir_oracle_blur,0.0001645649961998374,37.83662535915413,9.912582074506396e-05,0.015625,0.0,0.0\n0801,quantized_8bit,2,observed,0.0005529998110514886,32.572750170846504,0.00019831548499507745,0.0703125,0.0009765625,0.0\n0801,quantized_8bit,2,gradient_nominal,0.0004031678301951835,33.9451412866099,0.0001533444943355442,0.0537109375,0.0009765625,0.0\n0801,quantized_8bit,2,drunet_denoise_only,0.000508189901610744,32.93973969191797,0.00015453466216506633,0.0615234375,0.0009765625,0.0\n0801,quantized_8bit,2,dpir_nominal,0.00034820394771385737,34.581663094374534,0.00013526395687544273,0.0498046875,0.0,0.0\n0801,quantized_8bit,2,dpir_oracle_blur,0.00016498295552684698,37.825609206527176,9.923717494132928e-05,0.017578125,0.0,0.0\n0801,jpeg_q75,3,observed,0.0005401965469698654,32.6744819622544,0.0001663694330681154,0.064453125,0.0009765625,0.0\n0801,jpeg_q75,3,gradient_nominal,0.00043308017122318994,33.634317001895525,0.0001552780874919902,0.056640625,0.0009765625,0.0\n0801,jpeg_q75,3,drunet_denoise_only,0.0005274911576935355,32.77784816024755,0.00015781596834698407,0.0634765625,0.0009765625,0.0\n0801,jpeg_q75,3,dpir_nominal,0.0005587137730104786,32.52810622592363,0.0003010349699212224,0.0908203125,0.0263671875,0.0009765625\n0801,jpeg_q75,3,dpir_oracle_blur,0.0005141579765485012,32.88903422245683,0.0002427822184917154,0.1005859375,0.0078125,0.0\n0802,linear_float,0,observed,0.0006792701082287692,31.679574964308145,0.00027798071059088024,0.109375,0.0009765625,0.0\n0802,linear_float,0,gradient_nominal,0.000528596443465465,32.768757632408565,0.00022984576488111876,0.0771484375,0.0009765625,0.0\n0802,linear_float,0,drunet_denoise_only,0.000645937856728105,31.89809261847424,0.00023814319501344776,0.09765625,0.0009765625,0.0\n0802,linear_float,0,dpir_nominal,0.00048326207462365197,33.15817286027995,0.00021422436772709782,0.068359375,0.0009765625,0.0\n0802,linear_float,0,dpir_oracle_blur,0.0002746872455582133,35.611615055313315,0.00016721871230399913,0.0361328125,0.0,0.0\n0802,clipped_float,1,observed,0.0006792701082287692,31.679574964308145,0.00027798071059088024,0.109375,0.0009765625,0.0\n0802,clipped_float,1,gradient_nominal,0.0005284602923092098,32.769876393477716,0.00022978685328514642,0.0771484375,0.0009765625,0.0\n0802,clipped_float,1,drunet_denoise_only,0.0006459447194641692,31.898046477308498,0.0002381431948688755,0.09765625,0.0009765625,0.0\n0802,clipped_float,1,dpir_nominal,0.0004832764984393892,33.15804323930553,0.0002142252996287697,0.068359375,0.0009765625,0.0\n0802,clipped_float,1,dpir_oracle_blur,0.00027466792951942354,35.61192046243215,0.00016719829811282413,0.0361328125,0.0,0.0\n0802,quantized_8bit,2,observed,0.0006804863776852653,31.671805642965545,0.00027891518735896853,0.109375,0.0009765625,0.0\n0802,quantized_8bit,2,gradient_nominal,0.0005292257287262396,32.76359050402497,0.0002300695448765614,0.078125,0.0009765625,0.0\n0802,quantized_8bit,2,drunet_denoise_only,0.000645973191689437,31.897855051363916,0.00023813101566266096,0.09765625,0.0009765625,0.0\n0802,quantized_8bit,2,dpir_nominal,0.00048314851500213114,33.15919350963438,0.00021416214155040717,0.068359375,0.0009765625,0.0\n0802,quantized_8bit,2,dpir_oracle_blur,0.0002750420504691785,35.60600902931128,0.00016724214427839702,0.0361328125,0.0,0.0\n0802,jpeg_q75,3,observed,0.0007101691125158193,31.486382204540732,0.00025955167683207884,0.109375,0.0009765625,0.0\n0802,jpeg_q75,3,gradient_nominal,0.0006192831276310665,32.0811075218249,0.0002469058555058093,0.095703125,0.0009765625,0.0\n0802,jpeg_q75,3,drunet_denoise_only,0.0006896654817272859,31.613615102304525,0.00024513885388347456,0.1015625,0.0009765625,0.0\n0802,jpeg_q75,3,dpir_nominal,0.0006022606056376859,32.20215543460181,0.0002593254942014211,0.0986328125,0.00390625,0.0\n0802,jpeg_q75,3,dpir_oracle_blur,0.0006753662457313253,31.704606488515545,0.0003071950656726332,0.1279296875,0.00390625,0.0\n0803,linear_float,0,observed,0.0007830532590210386,31.062086985904106,0.0002346359761004098,0.078125,0.0009765625,0.0\n0803,linear_float,0,gradient_nominal,0.0005577622239255964,32.53550902935188,0.00018232816993788545,0.060546875,0.0009765625,0.0\n0803,linear_float,0,drunet_denoise_only,0.0007395999233156892,31.310031425420352,0.00019112973275771554,0.0693359375,0.0009765625,0.0\n0803,linear_float,0,dpir_nominal,0.0004726958967178419,33.254181676157096,0.00015504588878897756,0.0517578125,0.0009765625,0.0\n0803,linear_float,0,dpir_oracle_blur,0.00011343577270212483,39.452499662790295,6.331406268614967e-05,0.0009765625,0.0,0.0\n0803,clipped_float,1,observed,0.0007830532590210386,31.062086985904106,0.0002346359761004098,0.078125,0.0009765625,0.0\n0803,clipped_float,1,gradient_nominal,0.000557739220242052,32.535688148310086,0.0001823143574229616,0.060546875,0.0009765625,0.0\n0803,clipped_float,1,drunet_denoise_only,0.0007396085508320869,31.30998076477543,0.0001911285695924928,0.0693359375,0.0009765625,0.0\n0803,clipped_float,1,dpir_nominal,0.0004727091147894006,33.25406023538897,0.00015504410515588778,0.0517578125,0.0009765625,0.0\n0803,clipped_float,1,dpir_oracle_blur,0.00011342821808330654,39.45278890475025,6.330318875242616e-05,0.0009765625,0.0,0.0\n0803,quantized_8bit,2,observed,0.0007842943464565411,31.05520915566955,0.00023563371101404325,0.078125,0.0009765625,0.0\n0803,quantized_8bit,2,gradient_nominal,0.0005585449813902833,32.5294184603458,0.00018266476499481378,0.060546875,0.0009765625,0.0\n0803,quantized_8bit,2,drunet_denoise_only,0.0007396160455545269,31.30993675635903,0.00019117308403985962,0.0693359375,0.0009765625,0.0\n0803,quantized_8bit,2,dpir_nominal,0.0004727138061127298,33.25401713476617,0.0001550892054429066,0.0517578125,0.0009765625,0.0\n0803,quantized_8bit,2,dpir_oracle_blur,0.00011392478125897485,39.43381796610876,6.351214466337625e-05,0.0009765625,0.0,0.0\n0803,jpeg_q75,3,observed,0.000799284156096313,30.97298795797972,0.00021377364732371698,0.0771484375,0.0009765625,0.0\n0803,jpeg_q75,3,gradient_nominal,0.0006297656064780769,32.00821061267994,0.0001948002048951637,0.064453125,0.0009765625,0.0\n0803,jpeg_q75,3,drunet_denoise_only,0.0007724095940143557,31.121523405352928,0.00019663012506085338,0.07421875,0.0009765625,0.0\n0803,jpeg_q75,3,dpir_nominal,0.0007145917622020622,31.45941994641848,0.0003195225464268955,0.10546875,0.0185546875,0.0009765625\n0803,jpeg_q75,3,dpir_oracle_blur,0.0008599474614872736,30.655280811787094,0.00039919758172882576,0.1689453125,0.017578125,0.0\n0804,linear_float,0,observed,0.002229020549032272,26.518859278005213,0.000983692867992418,0.4921875,0.0927734375,0.0\n0804,linear_float,0,gradient_nominal,0.001822287606082521,27.393830785829234,0.0008997021467645005,0.458984375,0.080078125,0.0\n0804,linear_float,0,drunet_denoise_only,0.002197199242147662,26.58130559446394,0.0009451921349656786,0.47265625,0.091796875,0.0\n0804,linear_float,0,dpir_nominal,0.0016508863833506746,27.822828145517132,0.0008442803650984388,0.4384765625,0.06640625,0.0\n0804,linear_float,0,dpir_oracle_blur,0.0007772150927791376,31.09458774403891,0.0005714570851167753,0.3212890625,0.0146484375,0.0\n0804,clipped_float,1,observed,0.002229020549032272,26.518859278005213,0.000983692867992418,0.4921875,0.0927734375,0.0\n0804,clipped_float,1,gradient_nominal,0.0018222814670629459,27.39384541659777,0.0008996994407591863,0.458984375,0.080078125,0.0\n0804,clipped_float,1,drunet_denoise_only,0.0021972068650505058,26.581290527196323,0.0009451918162837774,0.47265625,0.091796875,0.0\n0804,clipped_float,1,dpir_nominal,0.001650879045532691,27.822847448971167,0.0008442758725757094,0.4384765625,0.06640625,0.0\n0804,clipped_float,1,dpir_oracle_blur,0.0007771948765436684,31.094700710369324,0.0005714438999797935,0.3212890625,0.0146484375,0.0\n0804,quantized_8bit,2,observed,0.0022301982100859648,26.51656537076859,0.0009846427570311318,0.4970703125,0.0927734375,0.0\n0804,quantized_8bit,2,gradient_nominal,0.001822909296284322,27.39234940288096,0.0008999501908805215,0.4580078125,0.0810546875,0.0\n0804,quantized_8bit,2,drunet_denoise_only,0.00219707853479143,26.581544189024456,0.0009451165492749314,0.47265625,0.091796875,0.0\n0804,quantized_8bit,2,dpir_nominal,0.0016503726400478314,27.82417984728763,0.0008440925239170039,0.4384765625,0.068359375,0.0\n0804,quantized_8bit,2,dpir_oracle_blur,0.0007774122244818421,31.093486345521065,0.0005714943917991711,0.3212890625,0.0146484375,0.0\n0804,jpeg_q75,3,observed,0.0022585963555120515,26.46161376975712,0.0009683008665079789,0.4873046875,0.09375,0.0\n0804,jpeg_q75,3,gradient_nominal,0.0019081243999171525,27.19393314937786,0.0009193140774677663,0.466796875,0.08203125,0.0\n0804,jpeg_q75,3,drunet_denoise_only,0.002241532563989358,26.494549475150485,0.0009550915602409772,0.4765625,0.0908203125,0.0\n0804,jpeg_q75,3,dpir_nominal,0.0020719218386911964,26.836266319699398,0.0011707144351741994,0.5087890625,0.138671875,0.001953125\n0804,jpeg_q75,3,dpir_oracle_blur,0.0018119411079390332,27.418559219532167,0.0010732326722935825,0.5224609375,0.1201171875,0.0\n', 'acquisition.csv': 'source_id,stage,stage_index,observation_sha256,dtype,height,width,channels,minimum,maximum,changed_component_fraction_from_previous,measurement_delta_mse,linear_out_of_range_fraction,jpeg_bytes,jpeg_sha256\n0801,linear_float,0,59b5708be31c414d2f9f6dc419fff3452584ad399cb449ff18094016f977558d,float64,576,576,3,-0.01049394553612078,1.0094212296877787,0.0,0.0,9.142714763374486e-05,0,\n0801,clipped_float,1,401ae43021750fe7d2b4c4b769ae16c4a7b4afbdd2cf1fa54cc01aa0a84f9cb0,float64,576,576,3,0.0,1.0,9.142714763374486e-05,9.011685683611753e-10,9.142714763374486e-05,0,\n0801,quantized_8bit,2,ee787d943a66d5cd982a213dc12fd2ac65b940fb9a3caec12c534f411de6e1d7,float64,576,576,3,0.0,1.0,0.9999085728523662,1.2812759746328564e-06,9.142714763374486e-05,0,\n0801,jpeg_q75,3,d91651a84e34fc062cadb283b39ef5ce4ec61d60cd3351404f3cc1f16fb21c3f,float64,576,576,3,0.00784313725490196,1.0,0.831298828125,8.761302180952034e-05,9.142714763374486e-05,34805,715ccc923874d6bcb637bc7317421a1f7dd59ed99fe9fb394bb7915700dd3c1f\n0802,linear_float,0,8d56dc865c996c093f0ba00ee0343dc5ea7193ce9e5786dd91ec47fb55d9eb0c,float64,576,576,3,-0.01963134989171287,0.926905228332161,0.0,0.0,0.0032059783307613167,0,\n0802,clipped_float,1,2412277a721236b69b00140be95de3edb5245b7cfcc8fcfafab70f0f3320d4c5,float64,576,576,3,0.0,0.926905228332161,0.0032059783307613167,6.014621896276507e-08,0.0032059783307613167,0,\n0802,quantized_8bit,2,4e2540975708accec0d0a3d4fddacea11eed1960a5d5b2c0c3466c23b0a727e3,float64,576,576,3,0.0,0.9254901960784314,0.9967940216692387,1.2784164573596277e-06,0.0032059783307613167,0,\n0802,jpeg_q75,3,1d99b26f16302ff2d3e4693676ded77e5108f86def9eb164fac651f0091f4c13,float64,576,576,3,0.0,0.9333333333333333,0.8437580375514403,0.00010556289573120911,0.0032059783307613167,47088,61cee091a64b27ebb4fb640ae3ec22d7b4d6ca065c55d211711f8829806398e3\n0803,linear_float,0,ddac9c8a7bb0a1dee8d6770eacf6c164886a58901acd953816e487b90190b44a,float64,576,576,3,-0.017290243004809913,1.0223213307854542,0.0,0.0,0.0019882892875514404,0,\n0803,clipped_float,1,49487f855c44fd33b550177e0bba9343889d6687f631f4d6d5ab7cabb2b4f358,float64,576,576,3,0.0,1.0,0.0019882892875514404,4.269143899569106e-08,0.0019882892875514404,0,\n0803,quantized_8bit,2,35ce589d82b71bab3646a4231745b4c7dda9c9c7e58f664b7d8c6c9d72989b59,float64,576,576,3,0.0,1.0,0.9980117107124485,1.281952836001138e-06,0.0019882892875514404,0,\n0803,jpeg_q75,3,75d3923799a0917c62b5c4b160a2a9a549d8b3c67574a6f5839e6997bb432fe1,float64,576,576,3,0.0,1.0,0.8457232188786008,0.00010952844048988575,0.0019882892875514404,49727,1972002d1a39e745bbea6311c1f1037e571fed6bdca4707c6af04a2594933751\n0804,linear_float,0,844bf24ed4017059f539365fa3113c6d0c690b104feae4d34dae9b0ca24af5ed,float64,576,576,3,-0.018373423295166637,0.993805033819376,0.0,0.0,0.00036972736625514403,0,\n0804,clipped_float,1,22a38b87205e584833552dc706796316d807059dfc8dce19783ec109098bf12f,float64,576,576,3,0.0,0.993805033819376,0.00036972736625514403,1.451025036651039e-08,0.00036972736625514403,0,\n0804,quantized_8bit,2,30fef7e319529fd808a48c0c35ef2d9ae199499953e95a94cd504e24376087fa,float64,576,576,3,0.0,0.9921568627450981,0.9996302726337448,1.2796157509886289e-06,0.00036972736625514403,0,\n0804,jpeg_q75,3,33f12a49b5f0ae553f3af1407ee0d56db9815cc5883c279d51e230602628ca8f,float64,576,576,3,0.0,0.9882352941176471,0.843238610789609,0.00010353099134172154,0.00036972736625514403,46195,d8a05a1f62635ce0985467900c59b80f7895e2e6c55c914c4b191b0e36110dd1\n'}
EXPECTED_RGB_HASHES = {'0801': 'c3b9ecc733b39ad48d0c260cbdc70e8cd08ed088ef9b82dc8d7dbc01b929d2e5', '0802': '320780296837730b43beebe107266e00779a0785ae3af52570e535d19165d682', '0803': '4ef27a27f8e05cd012a0389c28d55ef9377297b7c965024e5ad9b4bd4405c70d', '0804': 'c24a0821e1d8eaf46687e8803ec8dd75583d4f95b94dc79a50b42296e7c75296'}
for key, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==PROVENANCE['embedded_source_sha256'][key]
assert hashlib.sha256(DESIGN_FREEZE.encode()).hexdigest()==PROVENANCE['design_freeze_sha256']
for name, source in ANCHORS.items():
    assert hashlib.sha256(source.encode()).hexdigest()==PROVENANCE['anchor_sha256'][name]
modules={k:types.ModuleType(k+'04_embedded') for k in SOURCES}
for key,module in modules.items():
    exec(compile(SOURCES[key],'<'+module.__name__+'>','exec'),module.__dict__)
B01,L02,D03,J04,AUDIT,ARCHIVE=[modules[k] for k in ['baseline','learned','diagnostic','jpeg_aware','validator','archive']]
L02.B01=B01
D03.B01,D03.L02=B01,L02
J04.B01,J04.L02,J04.D03=B01,L02,D03
CONFIG=json.loads(json.dumps(L02.CONFIG))
CONFIG.update(experiment='jpeg_aware_04',stages=J04.STAGES,
    model_information='decoded RGB and fixed nominal blur/noise only; automatic FBCNN quality; no true-blur reconstruction')
print('Embedded sources, design and anchors verified.')


### 2. Locate Drive inputs and print the actual scope
The default includes all four sources. Environment overrides exist solely for reproducible local validation; they never change image resolution or model settings. An existing result folder is preserved rather than overwritten.

In [ ]:
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR=Path('/content/drive/MyDrive/reliable-reconstruction-under-mismatch')
else:
    PROJECT_DIR=Path.cwd()
    if PROJECT_DIR.name=='notebooks':PROJECT_DIR=PROJECT_DIR.parent
DATA_DIR=Path(os.environ.get('IMAGING04_DATA_DIR',str(PROJECT_DIR/'samples')))
DPIR_WEIGHTS=Path(os.environ.get('IMAGING04_WEIGHTS',str(PROJECT_DIR/'model_cache/drunet_color.pth')))
FBCNN_WEIGHTS=Path(os.environ.get('IMAGING04_FBCNN_WEIGHTS',str(PROJECT_DIR/'model_cache/fbcnn_color.pth')))
VENDOR_DIRS={k:PROJECT_DIR/'model_cache'/(k+'_'+PROVENANCE[k]['upstream_commit'][:7]) for k in VENDORS}
for key,files in VENDORS.items():
    for relative,source in files.items():
        path=VENDOR_DIRS[key]/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert J04.sha256(path)==PROVENANCE[key]['files'][relative], f'Changed vendor file: {path}'
        else:path.write_bytes(source.encode())
if 'IMAGING04_SOURCE_IDS' in os.environ:
    requested=[s.strip() for s in os.environ['IMAGING04_SOURCE_IDS'].split(',')]
    assert requested and len(requested)==len(set(requested)) and set(requested)<=set(CONFIG['source_ids'])
    CONFIG['source_ids']=requested
for sid in EXPECTED_RGB_HASHES:
    assert (DATA_DIR/(sid+'.png')).is_file(), f'Missing original PNG: {DATA_DIR/(sid+".png")}'
timestamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR=Path(os.environ.get('IMAGING04_OUTPUT_DIR',str(PROJECT_DIR/'results'/('jpeg_aware_04_'+timestamp))))
assert not OUTPUT_DIR.exists(), f'Existing run preserved; choose a new output directory: {OUTPUT_DIR}'
print('ACTUAL SOURCES:',CONFIG['source_ids'])
print('ACTUAL STAGES:',CONFIG['stages'])
print('Observations:',2*len(CONFIG['source_ids']),'/ full design: 8')
print('Experiment calls:',160*len(CONFIG['source_ids']),'DRUNet +',6*len(CONFIG['source_ids']),'FBCNN; plus 2 and 3 small checks')
print('Device:','CUDA GPU' if J04.torch.cuda.is_available() else 'CPU')
print('Destination:',OUTPUT_DIR)


### 3. Record the frozen decision rules
Primary reconstruction endpoint: JPEG mean detail MSE. The provisional screen requires FBCNN + DPIR to beat raw DPIR, raw classical and FBCNN + classical in the pooled comparison, and raw DPIR on at least three of four sources.

Primary selection endpoint: JPEG **all-patch detail MSE at 50% retention** on FBCNN + DPIR. Detail operator spread must beat all four operational controls, win against the best pooled control on at least three of four sources, and have no higher pooled bad-detail rate at any of the three fixed tolerances. All other coverages, RGB spread and textured strata are secondary.

These are descriptive development screens, not significance, calibration or novelty tests. A one-source run reports `not_assessed_full_design`. Retain negative outcomes and uncompressed-condition losses.

In [ ]:
display(pd.DataFrame([
    ['Input','Decoded RGB','No inference'],
    ['Classical','Decoded RGB + nominal blur 1.0','lambda 0.05'],
    ['DPIR','Decoded RGB + nominal blur/noise','8 DRUNet calls'],
    ['FBCNN','Decoded RGB; automatic quality','1 FBCNN call'],
    ['FBCNN + classical','FBCNN output + same inverse assumptions','1 FBCNN call + FFT inverse'],
    ['FBCNN + DPIR','FBCNN output + same solver assumptions','1 FBCNN call + 8 DRUNet calls'],
],columns=['Method','Information','Nominal cost']))
print('Design SHA-256:',PROVENANCE['design_freeze_sha256'])
print('FBCNN weight SHA-256:',PROVENANCE['fbcnn']['weight_sha256'])
print('A score ensemble uses 24 DRUNet calls. FBCNN operator spread adds 1 FBCNN pass; whole-pipeline rotations add 3.')


### 4. Run the frozen comparison
This long cell checks all four source hashes and all eight historical observation hashes before learned inference. It saves every reconstruction, score vector and patch error after each observation. CPU execution can be slow; GPU is recommended. A completed inference run still requires Section 5's independent readback.

In [ ]:
result=J04.run(DATA_DIR,OUTPUT_DIR,VENDOR_DIRS['dpir'],DPIR_WEIGHTS,
    VENDOR_DIRS['fbcnn'],FBCNN_WEIGHTS,PROVENANCE,EXPECTED_RGB_HASHES,
    ANCHORS['quality.csv'],ANCHORS['acquisition.csv'],CONFIG)
print(json.dumps(result['checks'],indent=2))


### 5. Independently recompute the saved evidence
A separate implementation reads the saved arrays and recomputes reconstruction errors, patch errors, score construction, every risk row, summaries, contrasts and compute accounting. Neural inference is not repeated. Only successful readback marks the configured run complete.

In [ ]:
independent_audit=AUDIT.validate(OUTPUT_DIR,DATA_DIR)
J04.dump(OUTPUT_DIR/'independent_readback.json',independent_audit)
result['checks']['independent_readback_passed']=independent_audit['status']=='passed'
J04.snapshot(result,OUTPUT_DIR,'complete_for_configured_subset')
print(json.dumps(independent_audit,indent=2))


### 6. Reconstruction quality
Pooled RGB PSNR is computed from pooled MSE, not by averaging dB. Detail MSE uses the inherited high-pass operator before removing the context border.

In [ ]:
display(result['summary'].round(8))
display(DisplayImage(filename=str(OUTPUT_DIR/'quality.png')))


### 7. Source-level changes and blind quality estimates
Negative detail-MSE differences favour FBCNN + DPIR. FBCNN's inferred quality is a model output, not a calibrated uncertainty measure or verified codec metadata.

In [ ]:
display(result['quality_changes'].round(8))
display(DisplayImage(filename=str(OUTPUT_DIR/'source_changes.png')))
display(result['fbcnn_diagnostics'].round(5))


### 8. Selective-detail errors and tails
Retain all coverages and all seven rankings. The oracle uses reference errors solely as an evaluation bound. Random curves show exact expected error under uniform retention, not one sampled mask. Nonzero error at low spread can reflect a shared missing acquisition stage.

In [ ]:
display(DisplayImage(filename=str(OUTPUT_DIR/'risk_coverage.png')))
primary=result['risk_summary'].query("stage=='jpeg_q75' and pipeline=='fbcnn_dpir' and region=='all' and coverage==0.5")
display(primary.round(8))
print('All regions, coverages, source comparisons and tail rates are saved in the CSVs.')


### 9. Inspect images, historical drift and compute
The montage uses a fixed central inset from the first configured source; metrics use the full interior. Historical DPIR differences may reflect CPU/GPU/software drift and are reported explicitly. Standalone score costs overlap at the shared nominal reconstruction: do not sum them as total experiment cost.

In [ ]:
display(DisplayImage(filename=str(OUTPUT_DIR/'examples.png')))
display(result['endpoint_comparison'].round(10))
display(result['compute'].groupby('component',as_index=False).agg(
    runs=('elapsed_seconds','size'),seconds=('elapsed_seconds','sum'),drunet_calls=('denoiser_calls','sum'),fbcnn_calls=('fbcnn_calls','sum')).round(3))
display(result['score_costs'].round(3))


### 10. Read the actual decision status
A successful numerical run is not a successful scientific hypothesis. The JSON records the predeclared screen outcomes only when all four sources have completed. Independent testing, calibration and novelty remain false regardless of these outcomes.

In [ ]:
print(json.dumps(result['decisions'],indent=2))
print('Executed sources:',CONFIG['source_ids'])
print('Review all results, including failures, before proposing another method or tuning settings.')


### 11. Save verified summary and RAW transfer ZIPs
Results already reside in the dated Drive folder in Colab. This cell checks all final hashes, verifies both archives byte for byte and automatically creates numbered RAW ZIP parts with payloads of at most 96 MiB. It verifies their complete reassembly hash. Rerunning this cell verifies existing archives instead of overwriting them.

**Upload the SUMMARY ZIP plus every numbered RAW part ZIP printed below.** The summary is sufficient for a first reading, but raw parts are needed to verify arrays independently. Do not upload the large unsplit RAW ZIP as well.

In [ ]:
export_receipt=ARCHIVE.export(OUTPUT_DIR)
print('UPLOAD SUMMARY:',export_receipt['summary']['path'])
print('UPLOAD ALL THESE RAW PARTS:')
for path in export_receipt['transfer']['part_paths']:print(path)
print('Verified parts:',export_receipt['transfer']['verified_parts'])
print('Full raw ZIP bytes:',export_receipt['raw']['bytes'])
print('Full raw ZIP SHA-256:',export_receipt['raw']['sha256'])
print('Full four-source design complete:',export_receipt['full_design_complete'])


## Interpretation and reproduction
The useful next decision is whether this established preprocessing baseline improves reconstruction and whether operator sensitivity adds selective-detail value under the fixed controls. Failure of this particular cascade does not reject the broad PhD topic or the separate scoping-review route.

The full design has eight observations, 48 main quality rows (112 including ensemble members), 896 risk rows, 640 experiment DRUNet calls and 24 FBCNN calls. Local source-0801 validation has two observations, 12 main quality rows (28 including ensemble members), 224 risk rows, 160 DRUNet calls and six FBCNN calls. Two DRUNet and three FBCNN loading/parity calls are separate. Source PNGs and model weights are external inputs, not included in result exports.

References: [official FBCNN source and release](https://github.com/jiaxi-jiang/FBCNN/tree/54d1831927506b3247e2d4d245abb4f4dab1a1cd), [FBCNN paper](https://arxiv.org/abs/2109.14573), [official pinned DPIR](https://github.com/cszn/DPIR/tree/15bca3fcc1f3cc51a1f99ccf027691e278c19354). The exact sources, licences, design and reviewed Acquisition 03 anchors are embedded in this notebook. Observed checkpoint hashes are not publisher-signed checksums.
